# MiBici - Entrega final: refactorización
## Objetivo
En el notebook 07 trabajamos la reproducibilidad, FastAPI, Docker, el monitoreo de drift, la comparación entre modelos y las pruebas. En esta parte organizamos ese código en los archivos que corresponden dentro del repositorio.

Cada sección escribe un archivo en `build/migracion/`, con la misma estructura de carpetas que el repositorio. La idea es que los módulos puedan ejecutarse sin depender de variables creadas en otras celdas. Al final probamos los cambios en una copia del repositorio y comparamos los resultados con el notebook 07 antes de mover los archivos.

| Archivo | Cambio | Viene de (notebook 07) |
|---|---|---|
| `mibici_mlops/config.py` | modificado: bloque al final | valores de las secciones 2 a 5 |
| `mibici_mlops/features.py` | modificado: `add_holidays` | 5.15 |
| `mibici_mlops/modeling/export_model.py` | nuevo | 2.1, 2.3 y 2.6 |
| `mibici_mlops/serving/__init__.py` y `app.py` | nuevo | 3 y 4.1 |
| `mibici_mlops/monitoring.py` | nuevo | 5.1 a 5.14 |
| `mibici_mlops/modeling/challenger.py` | nuevo | 5.15 |
| `mibici_mlops/modeling/champion_report.py` | nuevo | evidencia del champion en MLflow |
| `mibici_mlops/modeling/deploy.py` | nuevo | 4.0, 4.4 a 4.9 y 5.16 |
| `Dockerfile`, `.dockerignore`, `requirements-serving.txt` | nuevo | 4.2 y 4.3 |
| `requirements.txt` | modificado: versiones fijas | 2.1 |
| `pyproject.toml` | modificado: dependencias y pytest | 3 a 6 |
| `Makefile` | modificado: comandos nuevos | 4 y 5 |
| `tests/conftest.py` y `tests/helpers.py` | modificado y nuevo | 6.3 |
| `tests/test_*.py` (6 archivos) | nuevo | 6.4 a 6.8 |
| `README.md` | modificado: sección 10 y estructura | todo el notebook |

El reporte final se preparará después de integrar y revisar los archivos.

## Antes de ejecutar
- Abrirlo desde `notebooks/` con el entorno del proyecto activado, igual que el 07.
- El notebook 07 debe haberse ejecutado previamente, ya que aquí comparamos sus resultados con los nuevos: `reports/drift/`, `models/serving/`, las ejecuciones de MLflow y `reports/tests/pytest_entrega_final.txt`.
- Docker Desktop abierto solo para la verificación de la imagen (V.8). Si no está abierto, esa revisión se salta y el resto sigue.

## 1. Preparación
Identificamos la raíz del repositorio y limpiamos `build/migracion/` para que solo queden los archivos de esta corrida. `build/` está en `.gitignore`.

Hay tres opciones de ejecución: `RUN_CHALLENGER` vuelve a entrenar los challengers (unos minutos con los datos reales), `RUN_DOCKER` construye la imagen y `LOG_TO_MLFLOW` registra al final el monitoreo, la comparación y el champion en MLflow.

In [1]:
import json
from importlib import metadata
from pathlib import Path
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent

STAGE = ROOT / "build" / "migracion"
if STAGE.exists():
    shutil.rmtree(STAGE)
for folder in ["mibici_mlops/modeling", "mibici_mlops/serving", "tests"]:
    (STAGE / folder).mkdir(parents=True)

RUN_CHALLENGER = True   # reentrena A y B para comparar con 5.15
RUN_DOCKER = True       # construye la imagen desde la copia del repositorio
LOG_TO_MLFLOW = True    # al final, registra monitoreo, comparación y champion en los experimentos Fase3
print("Raíz del proyecto:", ROOT)
print("Archivos nuevos en:", STAGE)

Raíz del proyecto: /home/josel/mlops/equipo-8-mibici-mlops
Archivos nuevos en: /home/josel/mlops/equipo-8-mibici-mlops/build/migracion


## 2. `mibici_mlops/config.py`
Conservamos la configuración de la fase 2 y agregamos los valores que necesitamos para esta entrega:

- `PHASE2_MODEL_VERSION = 7`: la versión que reproducimos y con la que se monitorea octubre.
- Las carpetas de la entrega: `models/serving`, `reports/reproducibility`, `data/monitoring` y `reports/drift`.
- Los experimentos de MLflow de la entrega final: `MiBici_Fase3_Monitoreo_Drift` y `MiBici_Fase3_Champion_Challenger`. Siguen el nombre de `MiBici_Fase2_Modelado` y, como ese, guardan sus artefactos en `mlruns/`. Los experimentos creados en el notebook 07 (`MiBici_Monitoreo_Drift` y `MiBici_Champion_Challenger`) no se tocan; quedan con sus corridas de prueba.
- `reports/champion`, donde se guardan las tablas y gráficas del champion.
- El nombre de la imagen, el repositorio de DockerHub y la etiqueta vigente de cada champion (`RELEASES`). La v10 pasa de 1.1.0 a 1.2.0, que es la misma v10 con la página `/ui`.
- La regla de promoción (`MIN_IMPROVEMENT`) y los interruptores `AUTO_PROMOTE` y `AUTO_DEPLOY`, que empiezan en `False` igual que en el 07.

In [2]:
%%writefile ../build/migracion/mibici_mlops/config.py
"""Rutas, periodo de estudio, cortes de la separación, reglas de limpieza y esquemas."""

from pathlib import Path

import pandas as pd
import pyarrow as pa


# Rutas del proyecto
PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'


def locate_project_root(start: Path) -> Path:
    """Buscar la raíz del repositorio sin importar módulos propios."""
    for directory in (start, *start.parents):
        if (directory / 'pyproject.toml').is_file() and (directory / 'data' / 'raw').is_dir():
            return directory
    raise FileNotFoundError(
        'No se encontró pyproject.toml y data/raw en los directorios superiores.'
    )


# Periodo de estudio
STUDY_START = pd.Timestamp('2020-01-01')
STUDY_END = pd.Timestamp('2026-10-01')

# Cortes de la separación temporal; cada fecha es el inicio del conjunto siguiente
TRAIN_END = pd.Timestamp('2025-01-01')
VALIDATION_END = pd.Timestamp('2026-01-01')

SPLIT_PERIODS = {
    'train': (STUDY_START, TRAIN_END),
    'validation': (TRAIN_END, VALIDATION_END),
    'test': (VALIDATION_END, STUDY_END),
}

# Configuración del procesamiento y MLflow
MLFLOW_EXPERIMENT = 'MiBici'
EXPECTED_MONTHS = pd.period_range('2020-01', '2026-09', freq='M')
CHUNK_SIZE = 50_000
RUN_ALL_FILES = True  # True: 81 CSV; False: prueba con dos meses
SAVE_TRIPS = True      # Escritura Parquet por bloques

# Reglas de limpieza
EXCLUDED_STATIONS = frozenset({407, 124, 405})
VALID_GENDERS = frozenset({'M', 'F'})
MIN_AGE, MAX_AGE = 10, 100

# Formatos de fecha
DATETIME_FORMAT = '%Y-%m-%d %H:%M:%S'
ALTERNATIVE_DATETIME_FORMAT = '%d/%m/%Y %H:%M'

# Nombres y columnas del dataset
COLUMN_MAPPING = {'A}äe_nacimiento': 'Año_de_nacimiento'}

REQUIRED_COLUMNS = (
    'Viaje_Id',
    'Usuario_Id',
    'Genero',
    'Año_de_nacimiento',
    'Inicio_del_viaje',
    'Fin_del_viaje',
    'Origen_Id',
    'Destino_Id',
)

INTEGER_COLUMNS = (
    'Viaje_Id',
    'Usuario_Id',
    'Origen_Id',
    'Destino_Id',
)

EXCLUSION_REASONS = (
    'Viaje_Id inválido',
    'Viaje_Id duplicado',
    'Inicio inválido',
    'Fuera del periodo',
    'Origen inválido',
    'Estación excluida',
)

# Esquema de los viajes procesados
TRIP_PARQUET_SCHEMA = pa.schema([
    pa.field('Viaje_Id', pa.int64()),
    pa.field('Usuario_Id', pa.int64()),
    pa.field('Genero', pa.string()),
    pa.field('Año_de_nacimiento', pa.int64()),
    pa.field('Inicio_del_viaje', pa.timestamp('ns')),
    pa.field('Fin_del_viaje', pa.timestamp('ns')),
    pa.field('Origen_Id', pa.int64()),
    pa.field('Destino_Id', pa.int64()),
    pa.field('duracion_minutos', pa.float64()),
    pa.field('categoria_duracion', pa.string()),
    pa.field('tipo_recorrido', pa.string()),
])

# Esquema de la demanda horaria
COMPLETE_SCHEMA = pa.schema([
    pa.field('Origen_Id', pa.int32()),
    pa.field('Hora', pa.timestamp('ns')),
    pa.field('Cantidad_viajes', pa.int32()),
])

# Configuración de Fase 2, extraída del notebook 06 (versión 3).
# Fase 2: rutas
VALIDATION_REPORTS_DIR = PROJECT_ROOT / "reports" / "validation"
MODEL_REPORTS_DIR = PROJECT_ROOT / "reports" / "modeling"
MODEL_FIGURES_DIR = FIGURES_DIR / "modeling"
STATIONS_PATH = DATA_DIR / "raw" / "Manual_datos" / "nomenclatura_2026_09.csv"
WEATHER_DIR = DATA_DIR / "raw" / "weather"
WEATHER_PATH = WEATHER_DIR / "weather_daily_zones_2020_2026.csv"
STATION_ZONES_PATH = WEATHER_DIR / "mibici_station_weather_zones.csv"
POI_DIR = DATA_DIR / "raw" / "pois"
OSM_PATH = POI_DIR / "mexico-latest.osm.pbf"
POI_PATH = POI_DIR / "mibici_station_pois.csv"
DAILY_PATH = PROCESSED_DATA_DIR / "mibici_daily.parquet"
INTEGRATED_PATH = PROCESSED_DATA_DIR / "mibici_integrated.parquet"
FINAL_MODEL_PATH = MODELS_DIR / "mibici_final_model.joblib"

# Fase 2: fuentes externas
DATA_START = STUDY_START
DATA_END = STUDY_END - pd.Timedelta(days=1)  # STUDY_END es exclusivo: último día 2026-09-30
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_COLUMNS = ["temperature_2m_mean", "precipitation_sum"]
WEATHER_TIMEZONE = "America/Mexico_City"
WEATHER_GRID_DEGREES = 0.05
OSM_URL = "https://download.geofabrik.de/north-america/mexico-latest.osm.pbf"
POI_RADIUS_METERS = 500
POI_BBOX_MARGIN = 0.01
METRIC_CRS = "EPSG:32613"  # UTM 13N, distancias en metros
POI_CATEGORIES = {
    "education": {"amenity": ["school", "university", "college"]},
    "transport": {"railway": ["station", "tram_stop"], "public_transport": ["station"]},
    "commerce": {"shop": ["supermarket", "mall"], "amenity": ["marketplace"]},
    "work": {"office": None},
    "health": {"amenity": ["hospital", "clinic"]},
    "recreation": {"leisure": ["park", "sports_centre", "pitch"]},
}
POI_COLUMNS = [f"{category}_count" for category in POI_CATEGORIES]

# Fase 2: contrato del dataset integrado
CONTRACT_DTYPES = {
    "Origen_Id": "int64",
    "Fecha": "datetime64[us]",
    "target": "int64",
    "day_of_week": "int32",
    "month": "int32",
    "weather_zone": "int64",
    "temperature_2m_mean": "float64",
    "precipitation_sum": "float64",
    **{column: "int64" for column in POI_COLUMNS},
}

# Fase 2: problema y validación temporal
TARGET_COLUMN = "target"
HORIZON_DAYS = 1  # predecimos el día siguiente
LAGS = [1, 7, 14]  # deben ser >= HORIZON_DAYS
ROLLING_WINDOWS = [7, 28]
FIRST_CUTOFF = TRAIN_END  # primer mes evaluado: enero de 2025
TEST_START = VALIDATION_END  # 2026 se reserva para la prueba final
N_FOLDS = 12
TUNING_FOLD_STEP = 2  # la búsqueda usa una de cada dos ventanas
SEED = 42

# Fase 2: variables
STATION_ID_COLUMNS = ["Origen_Id"]
ONEHOT_COLUMNS = ["day_of_week", "month", "weather_zone"]
CALENDAR_COLUMNS = ["day_of_week", "month", "day_of_month"]
STATION_COLUMNS = ["weather_zone"] + POI_COLUMNS
HISTORY_COLUMNS = [f"lag_{lag}" for lag in LAGS] + [
    f"{prefix}_{window}" for window in ROLLING_WINDOWS for prefix in ("roll_mean", "active_days")
]
FEATURE_SETS = {
    "original_11_variables": ["day_of_week", "month"] + STATION_COLUMNS + WEATHER_COLUMNS,
    "calendario_estacion": STATION_ID_COLUMNS + CALENDAR_COLUMNS + STATION_COLUMNS,
    "calendario_estacion_historia": (
        STATION_ID_COLUMNS + CALENDAR_COLUMNS + STATION_COLUMNS + HISTORY_COLUMNS
    ),
    "historia_y_clima_observado": (
        STATION_ID_COLUMNS + CALENDAR_COLUMNS + STATION_COLUMNS + HISTORY_COLUMNS + WEATHER_COLUMNS
    ),
}
MAIN_FEATURE_SET = "calendario_estacion_historia"
FEATURE_COLUMNS = FEATURE_SETS[MAIN_FEATURE_SET]

# Fase 2: modelos
MODEL_NAMES = ["Random Forest", "HistGradientBoosting"]
RF_INITIAL_PARAMS = {
    "n_estimators": 100,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "max_samples": 0.5,  # cada árbol usa 50 % de las filas
    "random_state": SEED,
    "n_jobs": -1,
}
HGB_INITIAL_PARAMS = {
    "max_iter": 100,
    "max_leaf_nodes": 31,
    "learning_rate": 0.1,
    "random_state": SEED,
}
# Se usan cuando RUN_GRID_SEARCH = False. Random Forest: mejor configuración de la búsqueda
# de la versión anterior. HistGradientBoosting: punto de partida, todavía sin búsqueda propia.
FINAL_MODEL = "Random Forest"
BEST_PARAMS = {
    "Random Forest": {
        "n_estimators": 150,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "max_features": 0.7,
    },
    "HistGradientBoosting": {
        "learning_rate": 0.1,
        "max_iter": 200,
        "max_leaf_nodes": 31,
        "loss": "poisson",
    },
}
PARAMETER_GRIDS = {
    "Random Forest": [
        {
            "model__n_estimators": [150],
            "model__max_depth": [12, 15, 18],
            "model__min_samples_leaf": [5],
            "model__max_features": [0.7, 1.0],
        },
        {
            "model__n_estimators": [150],
            "model__max_depth": [15],
            "model__min_samples_leaf": [3, 8],
            "model__max_features": [0.7],
        },
    ],
    "HistGradientBoosting": {
        "model__learning_rate": [0.05, 0.1],
        "model__max_iter": [200, 400],
        "model__max_leaf_nodes": [31, 63],
        "model__loss": ["squared_error", "poisson"],
    },
}

# Fase 2: interruptores de las partes que tardan
FORCE_REBUILD_DATA = False  # True: reconstruir el Parquet integrado aunque sea válido
ALLOW_UNVERIFIED_CACHE = False  # False: no reutilizar caché sin manifiesto verificable
COMPARE_FEATURE_SETS = False  # comparar los cuatro conjuntos de variables (~15 min)
RUN_WEATHER_REFERENCE = False  # modelo con clima observado (~4 min)
RUN_GRID_SEARCH = False  # búsqueda de hiperparámetros (~1 h)

# Fase 2: versionado y procedencia de los datos
CSV_VERSION_DIR = PROJECT_ROOT / "reports" / "data_versioning"
CSV_SNAPSHOT_MAX_BYTES = 10 * 1024 * 1024  # los CSV pequeños se copian por versión
FILE_VERSION_DIR = CSV_VERSION_DIR / "all_files"
FILE_VERSION_INDEX = FILE_VERSION_DIR / "file_versions.json"
FILE_VERSION_TABLE = FILE_VERSION_DIR / "file_versions.csv"
OUTPUT_VERSION_DIR = CSV_VERSION_DIR / "model_outputs"
PROVENANCE_DIR = PROJECT_ROOT / "reports" / "mlflow_provenance"

# Fase 2: MLflow
MLFLOW_TRACKING_URI = f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
MLFLOW_ARTIFACT_DIR = PROJECT_ROOT / "mlruns"
MODELING_EXPERIMENT = "MiBici_Fase2_Modelado"
REGISTERED_MODEL_NAME = "mibici_salidas_diarias"
MODEL_ALIAS = "champion"

# Fase 2: resultados del notebook 05 (MAE) para comprobar la equivalencia
REFERENCE_RESULTS = {
    "Ingenuo estacional (lag_7)": 7.224,
    "Promedio estación-día": 7.703,
    "Promedio global por día (versión anterior)": 22.986,
    "HistGradientBoosting | calendario_estacion_historia": 5.440,
    "Random Forest | calendario_estacion_historia": 5.456,
    "Modelo final, 12 ventanas de 2025": 5.438,
    "Modelo final, prueba 2026 (todas)": 5.896,
    "Ingenuo estacional, prueba 2026 (todas)": 7.957,
    "Modelo final, prueba 2026 (nuevas)": 3.511,
}

# Entrega final: modelo de referencia, servicio, monitoreo y despliegue
PHASE2_MODEL_VERSION = 7  # versión de la fase 2; la que guarda FINAL_MODEL_PATH
SERVING_DIR = MODELS_DIR / "serving"
SERVING_MODEL_DIR = SERVING_DIR / "model"
REPRODUCIBILITY_DIR = PROJECT_ROOT / "reports" / "reproducibility"
MONITORING_DATA_DIR = DATA_DIR / "monitoring"
DRIFT_REPORTS_DIR = PROJECT_ROOT / "reports" / "drift"
CHAMPION_REPORTS_DIR = PROJECT_ROOT / "reports" / "champion"
# Experimentos de la entrega final, con artefactos en mlruns/ como MiBici_Fase2_Modelado.
# MiBici_Monitoreo_Drift y MiBici_Champion_Challenger quedan con las corridas del notebook 07.
MONITORING_EXPERIMENT = "MiBici_Fase3_Monitoreo_Drift"
CHALLENGER_EXPERIMENT = "MiBici_Fase3_Champion_Challenger"

# Docker: etiqueta vigente de cada champion desplegado. 1.1.0 fue la v10 sin la página /ui;
# 1.2.0 es la v10 con la página. Un champion nuevo recibe la siguiente (1.3.0).
IMAGE_NAME = "ml-service:latest"
DOCKERHUB_REPO = "josecval/mibici-ml-service"
RELEASES = {"7": "1.0.0", "10": "1.2.0"}

# Champion contra challenger: el challenger gana si baja el MAE al menos 2 % y no empeora el RMSE
MIN_IMPROVEMENT = 0.02
# Con False, los scripts solo muestran la decisión; con True, promueven y publican.
AUTO_PROMOTE = False
AUTO_DEPLOY = False

Writing ../build/migracion/mibici_mlops/config.py


## 3. `mibici_mlops/features.py`
Agregamos `add_holidays` para calcular la variable de días festivos que utiliza el challenger B (5.15). Usa la lista de la fase 2 (notebook 05) y se calcula con `month`, `day_of_month` y `day_of_week`, así la API sigue recibiendo las mismas 18 variables. El resto del archivo no cambia.

Al registrar un modelo que la usa, `challenger.py` guarda esta función dentro del modelo para que la imagen de Docker no necesite el paquete `mibici_mlops`.

In [3]:
%%writefile ../build/migracion/mibici_mlops/features.py
"""Variables del modelo: historia de la demanda, calendario y separación temporal."""

import pandas as pd


def add_history_features(data, horizon, lags, windows, target_column="target"):
    # Rezagos y promedios móviles calculados por fecha, no por fila.
    invalid_lags = [lag for lag in lags if lag < horizon]
    if invalid_lags:
        raise ValueError(
            f"Los rezagos {invalid_lags} usarían información futura con horizonte {horizon}."
        )
    if data.duplicated(["Origen_Id", "Fecha"]).any():
        raise ValueError("Hay fechas duplicadas por estación; no se pueden crear rezagos.")
    if not lags or not windows or any(window <= 0 for window in windows):
        raise ValueError("Los rezagos y ventanas deben ser listas no vacías de enteros positivos.")
    demand_grid = data.pivot(index="Fecha", columns="Origen_Id", values=target_column).asfreq("D")
    available = demand_grid.shift(horizon)

    history = {f"lag_{lag}": demand_grid.shift(lag) for lag in lags}
    for window in windows:
        history[f"roll_mean_{window}"] = available.rolling(window, min_periods=1).mean()
        history[f"active_days_{window}"] = available.rolling(window, min_periods=1).count()

    date_positions = demand_grid.index.get_indexer(data["Fecha"])
    station_positions = demand_grid.columns.get_indexer(data["Origen_Id"])
    result = data.copy()
    for name, values in history.items():
        result[name] = values.to_numpy()[date_positions, station_positions]
    return result


def add_calendar_features(data: pd.DataFrame) -> pd.DataFrame:
    # Componentes de la fecha que se conocen antes de predecir.
    result = data.copy()
    result["day_of_month"] = result["Fecha"].dt.day
    result["year"] = result["Fecha"].dt.year
    return result


def build_model_dataset(data, horizon, lags, windows, target_column="target"):
    # Tabla de modelado: orden estable, historia y calendario.
    ordered = data.sort_values(["Origen_Id", "Fecha"], kind="stable").reset_index(drop=True)
    with_history = add_history_features(ordered, horizon, lags, windows, target_column)
    return add_calendar_features(with_history)


def split_by_date(data: pd.DataFrame, test_start: pd.Timestamp):
    # Desarrollo antes de la fecha de corte; prueba desde ella.
    development = data.loc[data["Fecha"] < test_start].reset_index(drop=True)
    test = data.loc[data["Fecha"] >= test_start].reset_index(drop=True)
    return development, test


# Días oficiales de descanso (Ley Federal del Trabajo, art. 74), la lista del notebook 05.
HOLIDAY_DATES = [101, 501, 916, 1225]   # mes * 100 + día
HOLIDAY_MONDAYS = [21, 33, 113]         # mes * 10 + número de lunes: 1.º de febrero, 3.º de marzo y de noviembre


def add_holidays(features: pd.DataFrame) -> pd.DataFrame:
    # Se calcula con month, day_of_month y day_of_week, así la API recibe las mismas 18 variables.
    features = features.copy()
    date_key = features["month"] * 100 + features["day_of_month"]
    monday_key = features["month"] * 10 + (features["day_of_month"] - 1) // 7 + 1
    holiday = date_key.isin(HOLIDAY_DATES) | ((features["day_of_week"] == 0) & monday_key.isin(HOLIDAY_MONDAYS))
    features["is_holiday"] = holiday.astype(int)
    return features

Writing ../build/migracion/mibici_mlops/features.py


## 4. `mibici_mlops/modeling/export_model.py`
Aquí reunimos lo que hicimos en 2.1, 2.3 y 2.6: las versiones del entorno, el hash del dataset integrado y la exportación del champion a `models/serving` con `metadata.json` y la muestra de referencia de 500 filas.

Se agrega `export_ui_days`, que guarda en `models/serving/ui_days.csv` las 18 variables ya calculadas por estación y día, de septiembre y octubre de 2026, con los viajes reales. La página `/ui` utiliza este archivo para consultar las variables ya calculadas, sin volver a generar los rezagos.

`export_serving_model` recibe la carpeta destino como argumento. Durante la verificación usamos una carpeta temporal para no tocar `models/serving`. Desde la terminal se corre con `python -m mibici_mlops.modeling.export_model` o `make export-model`.

In [4]:
%%writefile ../build/migracion/mibici_mlops/modeling/export_model.py
"""Exportar el champion de MLflow a models/serving para el servicio y la imagen de Docker."""

from datetime import datetime, timezone
import hashlib
from importlib import metadata
import json
from pathlib import Path
import platform
import shutil

import mlflow
import mlflow.sklearn
import pandas as pd

from mibici_mlops.config import (
    FEATURE_COLUMNS,
    HORIZON_DAYS,
    INTEGRATED_PATH,
    LAGS,
    MLFLOW_TRACKING_URI,
    MODEL_ALIAS,
    REGISTERED_MODEL_NAME,
    ROLLING_WINDOWS,
    SEED,
    SERVING_DIR,
    TARGET_COLUMN,
    TEST_START,
)
from mibici_mlops import monitoring
from mibici_mlops.features import build_model_dataset, split_by_date

PACKAGES = ["scikit-learn", "pandas", "numpy", "scipy", "joblib", "cloudpickle", "pyarrow", "mlflow"]
REFERENCE_ROWS = 500
UI_DAYS_FILE = "ui_days.csv"
UI_START = pd.Timestamp("2026-09-01")


def environment_versions(packages: list[str] = PACKAGES) -> dict:
    """Versión de Python y de las librerías que afectan al modelo serializado."""
    environment = {"python": platform.python_version()}
    for package in packages:
        try:
            environment[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            environment[package] = None
    return environment


def sha256(path: Path, block: int = 1024 * 1024) -> str:
    """Hash del archivo, para comprobar que los datos son los del entrenamiento."""
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(block), b""):
            digest.update(chunk)
    return digest.hexdigest()


def export_serving_model(version, test: pd.DataFrame, integrated_sha: str, environment: dict,
                         serving_dir: Path = SERVING_DIR) -> pd.DataFrame:
    """Copiar una versión del registro sin volver a serializarla, con su metadata y muestra de referencia."""
    uri = f"models:/{REGISTERED_MODEL_NAME}/{version.version}"
    model_dir = serving_dir / "model"
    if model_dir.exists():
        shutil.rmtree(model_dir)
    model_dir.mkdir(parents=True)
    downloaded = Path(mlflow.artifacts.download_artifacts(artifact_uri=uri, dst_path=str(model_dir)))
    if downloaded.resolve() != model_dir.resolve():
        for item in downloaded.iterdir():
            shutil.move(str(item), model_dir / item.name)
        shutil.rmtree(downloaded)

    metadata_info = {
        "model_name": REGISTERED_MODEL_NAME,
        "model_version": str(version.version),
        "model_alias": MODEL_ALIAS,
        "registry_uri": uri,
        "run_id": version.run_id,
        "features": FEATURE_COLUMNS,
        "seed": SEED,
        "integrated_sha256": integrated_sha,
        "environment": environment,
        "exported_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (serving_dir / "metadata.json").write_text(json.dumps(metadata_info, indent=2, ensure_ascii=False))

    # Muestra fija de la prueba 2026 con la predicción del modelo exportado
    served_model = mlflow.sklearn.load_model(str(model_dir))
    sample = test.sample(n=min(REFERENCE_ROWS, len(test)), random_state=SEED).sort_values(["Fecha", "Origen_Id"])
    reference = sample[["Fecha"] + FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
    reference["reference_prediction"] = served_model.predict(sample[FEATURE_COLUMNS])
    reference.to_csv(serving_dir / "reference_sample.csv", index=False)
    return reference


def export_ui_days(dataset: pd.DataFrame, month: pd.DataFrame, serving_dir: Path = SERVING_DIR,
                   start: pd.Timestamp = UI_START) -> pd.DataFrame:
    """Variables ya calculadas por estación y día para la página /ui, con los viajes reales."""
    columns = ["Fecha"] + FEATURE_COLUMNS + [TARGET_COLUMN]
    days = pd.concat([dataset.loc[dataset["Fecha"] >= start, columns], month[columns]], ignore_index=True)
    days = days.drop_duplicates(["Origen_Id", "Fecha"], keep="last").sort_values(["Fecha", "Origen_Id"])
    days["Fecha"] = days["Fecha"].dt.strftime("%Y-%m-%d")
    serving_dir.mkdir(parents=True, exist_ok=True)
    days.to_csv(serving_dir / UI_DAYS_FILE, index=False)
    return days


def main() -> None:
    """Exportar la versión que tiene el alias champion."""
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    version = mlflow.MlflowClient().get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
    integrated = pd.read_parquet(INTEGRATED_PATH)
    dataset = build_model_dataset(integrated, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
    development, test = split_by_date(dataset, TEST_START)
    export_serving_model(version, test, sha256(INTEGRATED_PATH), environment_versions())
    print(f"Exportado models:/{REGISTERED_MODEL_NAME}/{version.version} a {SERVING_DIR / 'model'}")

    # Días de la página /ui: septiembre del integrado y el mes monitoreado (octubre)
    if monitoring.MONITOR_CSV.is_file():
        hourly, _ = monitoring.load_month_hourly(monitoring.MONITOR_CSV, monitoring.MONITOR_START,
                                                 monitoring.MONITOR_END)
        month_daily = monitoring.build_month_daily(hourly, integrated, set(development["Origen_Id"]))
        month = monitoring.build_month_dataset(integrated, month_daily, monitoring.MONITOR_START)
    else:
        month = dataset.iloc[0:0]
    days = export_ui_days(dataset, month)
    print(f"{UI_DAYS_FILE}: {len(days):,} filas, {days['Fecha'].min()} a {days['Fecha'].max()}")


if __name__ == "__main__":
    main()

Writing ../build/migracion/mibici_mlops/modeling/export_model.py


## 5. `mibici_mlops/serving/`
Recuperamos el servicio que desarrollamos en las secciones 3 y 4.1: esquemas de Pydantic, carga del modelo, `GET /health` y `POST /predict`. No importa nada de `mibici_mlops`, porque la lista de variables la lee de `metadata.json`.

También agregamos una página para consultar predicciones sin tener que escribir una solicitud JSON:

- `GET /ui`: formulario para elegir una estación y un día. Muestra los viajes predichos, los reales y la historia reciente con la que se predijo.
- `GET /predict/estacion?origen_id=35&fecha=2026-10-15`: la ruta que llama la página. Si el día está en `ui_days.csv`, predice con esas variables. Si no está (un día posterior al 31 de octubre o un día sin viajes), usa la última historia conocida de la estación con el calendario del día pedido, y lo marca con `estimated` y `history_end`. Responde 404 solo si la estación no tiene historia antes de ese día.

- `GET /`: la raíz lleva a `/ui`, o a `/docs` si falta `ui_days.csv`. De esta forma, la dirección principal muestra una página del servicio.

Con el servicio en marcha (local o en Docker), las direcciones son:

| Dirección | Para qué |
|---|---|
| http://localhost:8000/ui | página para elegir estación y día |
| http://localhost:8000/docs | documentación interactiva (Swagger), para probar `POST /predict` con *Try it out* |
| http://localhost:8000/redoc | la misma documentación en formato de lectura |
| http://localhost:8000/health | estado del servicio y versión del modelo |

La ruta del modelo viene de la variable de entorno `MODEL_URI`. En la imagen es `/app/models/serving/model`; para correrlo local, `make serve` la apunta a `models/serving/model`.

In [5]:
%%writefile ../build/migracion/mibici_mlops/serving/__init__.py
"""Servicio de predicción con FastAPI."""

Writing ../build/migracion/mibici_mlops/serving/__init__.py


In [6]:
%%writefile ../build/migracion/mibici_mlops/serving/app.py
"""API de predicción de salidas diarias por estación de MiBici (Equipo 8)."""

from contextlib import asynccontextmanager
import json
import os
from pathlib import Path

from fastapi import FastAPI, HTTPException, Query, Request
from fastapi.responses import HTMLResponse, RedirectResponse
import mlflow.sklearn
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field

API_VERSION = "1.1.0"
MAX_INSTANCES = 5_000
MODEL_URI = os.getenv("MODEL_URI", "/app/models/serving/model")
UI_DAYS_FILE = "ui_days.csv"

EXAMPLE_INSTANCE = {
    "Origen_Id": 35, "day_of_week": 2, "month": 10, "day_of_month": 15, "weather_zone": 3,
    "education_count": 4, "transport_count": 1, "commerce_count": 2, "work_count": 6,
    "health_count": 0, "recreation_count": 3,
    "lag_1": 41, "lag_7": 38, "lag_14": 36, "roll_mean_7": 39.4, "active_days_7": 7,
    "roll_mean_28": 37.9, "active_days_28": 28,
}


class StationDayFeatures(BaseModel):
    """Las 18 variables del modelo para una estación y un día."""

    model_config = ConfigDict(extra="forbid", json_schema_extra={"example": EXAMPLE_INSTANCE})

    Origen_Id: int = Field(..., gt=0, description="Id de la estación de origen")
    day_of_week: int = Field(..., ge=0, le=6, description="0 = lunes, 6 = domingo")
    month: int = Field(..., ge=1, le=12)
    day_of_month: int = Field(..., ge=1, le=31)
    weather_zone: int = Field(..., ge=0, description="Zona meteorológica de la estación")
    education_count: int = Field(..., ge=0, description="POI de educación a 500 m")
    transport_count: int = Field(..., ge=0, description="POI de transporte a 500 m")
    commerce_count: int = Field(..., ge=0, description="POI de comercio a 500 m")
    work_count: int = Field(..., ge=0, description="POI de oficinas a 500 m")
    health_count: int = Field(..., ge=0, description="POI de salud a 500 m")
    recreation_count: int = Field(..., ge=0, description="POI de recreación a 500 m")
    lag_1: float | None = Field(None, ge=0, description="Salidas del día anterior")
    lag_7: float | None = Field(None, ge=0, description="Salidas de hace 7 días")
    lag_14: float | None = Field(None, ge=0, description="Salidas de hace 14 días")
    roll_mean_7: float | None = Field(None, ge=0, description="Promedio de los últimos 7 días")
    active_days_7: int = Field(..., ge=0, le=7, description="Días con registro en los últimos 7")
    roll_mean_28: float | None = Field(None, ge=0, description="Promedio de los últimos 28 días")
    active_days_28: int = Field(..., ge=0, le=28, description="Días con registro en los últimos 28")


class PredictRequest(BaseModel):
    model_config = ConfigDict(extra="forbid",
                              json_schema_extra={"example": {"instances": [EXAMPLE_INSTANCE]}})
    instances: list[StationDayFeatures] = Field(..., min_length=1, max_length=MAX_INSTANCES)


class Prediction(BaseModel):
    Origen_Id: int
    predicted_trips: float = Field(..., description="Viajes que inician en la estación ese día")


class PredictResponse(BaseModel):
    model_name: str
    model_version: str
    model_uri: str
    predictions: list[Prediction]


class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_name: str | None = None
    model_version: str | None = None


class StationDayPrediction(BaseModel):
    Origen_Id: int
    Fecha: str
    predicted_trips: float = Field(..., description="Viajes que predice el modelo")
    actual_trips: float | None = Field(None, description="Viajes que hubo en realidad, si ya se conocen")
    model_version: str
    estimated: bool = Field(..., description="True si el día no está en la tabla y se usó la última historia conocida")
    history_end: str = Field(..., description="Último día con viajes reales que se usó para la historia")
    features: dict = Field(..., description="Las 18 variables con las que se predijo")


class LoadedModel:
    def __init__(self, uri: str):
        self.uri = uri
        self.model = mlflow.sklearn.load_model(uri)
        meta_path = Path(uri).parent / "metadata.json"
        meta = json.loads(meta_path.read_text()) if meta_path.is_file() else {}
        self.name = meta.get("model_name", "mibici_salidas_diarias")
        self.version = str(meta.get("model_version", "desconocida"))
        self.features = meta.get("features") or list(self.model.feature_names_in_)
        days_path = Path(uri).parent / UI_DAYS_FILE
        self.days = pd.read_csv(days_path, dtype={"Fecha": str}) if days_path.is_file() else None

    def predict(self, features: pd.DataFrame) -> np.ndarray:
        return np.clip(self.model.predict(features[self.features]), 0, None)


@asynccontextmanager
async def lifespan(app: FastAPI):
    try:
        app.state.model = LoadedModel(MODEL_URI)
    except Exception as error:
        app.state.model, app.state.model_error = None, str(error)
    yield


app = FastAPI(
    title="MiBici · Predicción de salidas diarias por estación",
    description="Equipo 8, MLOps. Viajes que iniciarán en cada estación el día siguiente.",
    version=API_VERSION,
    lifespan=lifespan,
)


@app.get("/health", response_model=HealthResponse, tags=["monitoreo"])
def health(request: Request) -> HealthResponse:
    model = getattr(request.app.state, "model", None)
    return HealthResponse(status="ok" if model else "degradado", model_loaded=model is not None,
                          model_name=getattr(model, "name", None),
                          model_version=getattr(model, "version", None))


@app.post("/predict", response_model=PredictResponse, tags=["predicción"])
def predict(payload: PredictRequest, request: Request) -> PredictResponse:
    model = getattr(request.app.state, "model", None)
    if model is None:
        raise HTTPException(503, f"Modelo no disponible: {request.app.state.model_error}")
    features = pd.DataFrame([row.model_dump() for row in payload.instances])
    history = [c for c in features if c.startswith(("lag_", "roll_"))]
    features[history] = features[history].astype("float64")
    try:
        predictions = model.predict(features)
    except Exception as error:
        raise HTTPException(500, "Error interno al predecir") from error
    return PredictResponse(
        model_name=model.name, model_version=model.version, model_uri=model.uri,
        predictions=[Prediction(Origen_Id=int(s), predicted_trips=round(float(p), 4))
                     for s, p in zip(features["Origen_Id"], predictions)],
    )


def features_from_history(station: pd.DataFrame, date: pd.Timestamp, columns: list[str]) -> pd.DataFrame:
    """Variables de un día que no está en la tabla, con la última historia conocida de la estación."""
    # Se calcula como si el día pedido fuera el siguiente al último con datos; el calendario es el del día pedido
    last = station.iloc[-1]
    last_date = pd.Timestamp(last["Fecha"])
    demand = pd.Series(station["target"].to_numpy(), index=pd.to_datetime(station["Fecha"]))
    row = {column: last[column] for column in columns}
    row.update({"day_of_week": date.day_of_week, "month": date.month, "day_of_month": date.day})
    for column in columns:
        if column.startswith("lag_"):
            row[column] = demand.get(last_date - pd.Timedelta(days=int(column[4:]) - 1), np.nan)
        elif column.startswith(("roll_mean_", "active_days_")):
            window = int(column.rsplit("_", 1)[1])
            recent = demand.loc[last_date - pd.Timedelta(days=window - 1):last_date]
            row[column] = recent.mean() if column.startswith("roll_mean_") else len(recent)
    return pd.DataFrame([row])[columns].astype("float64")


@app.get("/predict/estacion", response_model=StationDayPrediction, tags=["interfaz"])
def predict_station_day(request: Request,
                        origen_id: int = Query(..., gt=0, description="Id de la estación"),
                        fecha: str = Query(..., description="Día a predecir, AAAA-MM-DD")) -> StationDayPrediction:
    """Predicción para una estación y un día."""
    model = getattr(request.app.state, "model", None)
    if model is None:
        raise HTTPException(503, f"Modelo no disponible: {request.app.state.model_error}")
    if model.days is None:
        raise HTTPException(503, f"No se encontró {UI_DAYS_FILE} junto al modelo")
    try:
        date = pd.Timestamp(fecha)
    except ValueError as error:
        raise HTTPException(422, f"Fecha inválida: {fecha}. Usa el formato AAAA-MM-DD") from error
    fecha = date.strftime("%Y-%m-%d")

    station = model.days.loc[model.days["Origen_Id"] == origen_id]
    row = station.loc[station["Fecha"] == fecha]
    if not row.empty:
        features, actual, history_end, estimated = row[model.features], row["target"].iloc[0], None, False
    else:
        history = station.loc[station["Fecha"] < fecha]
        if history.empty:
            raise HTTPException(404, f"La estación {origen_id} no tiene historia antes del {fecha}")
        features = features_from_history(history, date, model.features)
        actual, history_end, estimated = None, history["Fecha"].iloc[-1], True
    if history_end is None:
        previous = station.loc[station["Fecha"] < fecha, "Fecha"]
        history_end = previous.iloc[-1] if not previous.empty else fecha

    values = features.iloc[0]
    return StationDayPrediction(
        Origen_Id=origen_id, Fecha=fecha,
        predicted_trips=round(float(model.predict(features)[0]), 2),
        actual_trips=None if actual is None or pd.isna(actual) else float(actual),
        model_version=model.version, estimated=estimated, history_end=history_end,
        features={k: (None if pd.isna(v) else float(v)) for k, v in values.items()},
    )


UI_PAGE = """<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>MiBici: predicción de salidas</title>
<style>
  body { font-family: system-ui, sans-serif; max-width: 720px; margin: 2rem auto; padding: 0 1rem; color: #1f2933; }
  h1 { font-size: 1.5rem; margin-bottom: .25rem; }
  p.nota { color: #52606d; margin-top: 0; }
  form { display: flex; gap: .75rem; flex-wrap: wrap; align-items: end; margin: 1.5rem 0; }
  label { display: flex; flex-direction: column; font-size: .9rem; gap: .25rem; }
  select, input, button { font-size: 1rem; padding: .45rem .6rem; }
  button { background: #c8102e; color: white; border: 0; border-radius: 4px; cursor: pointer; }
  .resultado { border: 1px solid #d9e2ec; border-radius: 6px; padding: 1rem 1.25rem; }
  .cifra { font-size: 2.25rem; font-weight: 600; }
  table { border-collapse: collapse; margin-top: 1rem; font-size: .9rem; }
  td { padding: .2rem .8rem .2rem 0; }
  .error { color: #c8102e; }
  .aviso { color: #8a5a00; background: #fff7e0; padding: .4rem .6rem; border-radius: 4px; }
</style>
</head>
<body>
<h1>MiBici: viajes que saldrán de una estación</h1>
<p class="nota">Modelo __MODEL__. Hay viajes reales del __START__ al __END__. Para esos días la predicción usa
la demanda real de la estación hasta el día anterior, igual que el pipeline. Para un día posterior se usa
la última historia conocida de la estación y el calendario del día elegido, y se marca como estimación.</p>
<form id="forma">
  <label>Estación <select id="estacion">__OPTIONS__</select></label>
  <label>Día <input type="date" id="fecha" min="__START__" value="__END__"></label>
  <button type="submit">Predecir</button>
</form>
<div id="salida"></div>
<script>
document.getElementById("forma").addEventListener("submit", async (event) => {
  event.preventDefault();
  const estacion = document.getElementById("estacion").value;
  const fecha = document.getElementById("fecha").value;
  const salida = document.getElementById("salida");
  const respuesta = await fetch(`predict/estacion?origen_id=${estacion}&fecha=${fecha}`);
  const datos = await respuesta.json();
  if (!respuesta.ok) { salida.innerHTML = `<p class="error">${datos.detail}</p>`; return; }
  const real = datos.actual_trips === null ? "todavía no se conocen" : datos.actual_trips;
  const aviso = datos.estimated
    ? `<p class="aviso">Estimación: no hay datos de ese día, se usó la historia real hasta el ${datos.history_end}.</p>` : "";
  const v = datos.features;
  salida.innerHTML = `<div class="resultado">
    <div>Estación ${datos.Origen_Id}, ${datos.Fecha}</div>
    <div class="cifra">${datos.predicted_trips} viajes</div>
    ${aviso}
    <div>Viajes reales ese día: ${real}</div>
    <table>
      <tr><td>Viajes el último día con datos (lag_1)</td><td>${v.lag_1 ?? "sin dato"}</td></tr>
      <tr><td>Mismo día de la semana anterior (lag_7)</td><td>${v.lag_7 ?? "sin dato"}</td></tr>
      <tr><td>Promedio de los últimos 7 días</td><td>${v.roll_mean_7 === null ? "sin dato" : v.roll_mean_7.toFixed(1)}</td></tr>
      <tr><td>Promedio de los últimos 28 días</td><td>${v.roll_mean_28 === null ? "sin dato" : v.roll_mean_28.toFixed(1)}</td></tr>
    </table></div>`;
});
</script>
</body>
</html>"""


@app.get("/ui", response_class=HTMLResponse, include_in_schema=False)
def ui(request: Request) -> HTMLResponse:
    """Página para elegir estación y día y ver la predicción."""
    model = getattr(request.app.state, "model", None)
    if model is None or model.days is None:
        return HTMLResponse("<p>El modelo o el archivo de días no están disponibles. Revisa /health.</p>",
                            status_code=503)
    stations = sorted(model.days["Origen_Id"].unique())
    options = "".join(f'<option value="{s}">{s}</option>' for s in stations)
    page = (UI_PAGE.replace("__OPTIONS__", options)
            .replace("__START__", model.days["Fecha"].min())
            .replace("__END__", model.days["Fecha"].max())
            .replace("__MODEL__", f"{model.name} v{model.version}"))
    return HTMLResponse(page)


@app.get("/", include_in_schema=False)
def home(request: Request) -> RedirectResponse:
    """La raíz lleva a la página /ui; si no hay archivo de días, a la documentación /docs."""
    model = getattr(request.app.state, "model", None)
    return RedirectResponse("/ui" if model is not None and model.days is not None else "/docs")

Writing ../build/migracion/mibici_mlops/serving/app.py


## 6. `mibici_mlops/monitoring.py`
Este módulo contiene el monitoreo desarrollado en las secciones 5.1 a 5.14:

- Lectura y limpieza del CSV del mes, demanda diaria, variables con la historia del mes anterior y predicción del modelo en uso (`prepare_month`).
- Desempeño por mes, pruebas de drift (KS, Wasserstein y PSI), diagnóstico de temporada y reportes de Evidently.
- Los tres escenarios sintéticos y sus tipos.
- Las cinco alertas con sus dos niveles y la acción propuesta.
- `run_monitoring`, que hace todo en el mismo orden que el 07 y guarda las tablas y las tres gráficas de 5.13 en la carpeta que se le indique.
- `log_monitoring`, que registra el monitoreo en `MiBici_Fase3_Monitoreo_Drift` con la misma estructura de la Fase 2: contexto de la ejecución (commit, datos, entorno) con `tracking.log_run_context`, descripción, `README.md` con la guía de artefactos el linaje de datos y una carpeta por tema: `monitoring/performance`, `monitoring/drift`, `monitoring/scenarios` y `monitoring/alerts`. El MAE y la mejora por mes quedan como series en Model metrics, y cada escenario es una ejecución hija con sus métricas y el nivel de cada alerta como etiqueta (`alert_A1` a `alert_A5`).

Los valores de referencia (MAE base, demanda media, rezagos vacíos de la prueba) se calculan dentro y entran como argumentos a las funciones de alertas, Esto también permite probar las funciones con conjuntos de datos pequeños. Desde la terminal: `python -m mibici_mlops.monitoring` o `make monitor`.

In [7]:
%%writefile ../build/migracion/mibici_mlops/monitoring.py
"""Monitoreo mensual del modelo: desempeño, data drift, escenarios sintéticos, alertas y acción."""

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance

from mibici_mlops import mlflow_catalog, preprocessing, tracking
from mibici_mlops.config import (
    CHUNK_SIZE,
    DRIFT_REPORTS_DIR,
    FEATURE_COLUMNS,
    HISTORY_COLUMNS,
    HORIZON_DAYS,
    INTEGRATED_PATH,
    LAGS,
    MLFLOW_ARTIFACT_DIR,
    MLFLOW_TRACKING_URI,
    MONITORING_DATA_DIR,
    MONITORING_EXPERIMENT,
    PHASE2_MODEL_VERSION,
    POI_COLUMNS,
    PROJECT_ROOT,
    REGISTERED_MODEL_NAME,
    ROLLING_WINDOWS,
    SEED,
    TARGET_COLUMN,
    TEST_START,
)
from mibici_mlops.dataset import build_daily_demand
from mibici_mlops.features import build_model_dataset, split_by_date
from mibici_mlops.ingestion import read_csv_chunks
from mibici_mlops.modeling.evaluate import predict_seasonal_naive, regression_metrics

# Mes monitoreado. El escenario 3 baja la demanda desde DROP_START.
MONITOR_START = pd.Timestamp("2026-10-01")
MONITOR_END = pd.Timestamp("2026-11-01")
DROP_START = pd.Timestamp("2026-10-16")
MONITOR_CSV = MONITORING_DATA_DIR / "2026" / "datos_abiertos_2026_10.csv"

# Pruebas estadísticas por variable
KS_THRESHOLD = 0.1
WASSERSTEIN_THRESHOLD = 0.1
DATASET_DRIFT_SHARE = 0.3

# Niveles de alerta: (advertencia, crítica)
MISSING_LEVELS = (0.05, 0.15)     # puntos de lag_1 vacío por arriba de la línea base
BIAS_LEVELS = (0.10, 0.20)        # sesgo como fracción de la demanda media de referencia
PSI_LEVELS = (0.10, 0.25)
OPERATION_LEVELS = (0.15, 0.30)   # cambio relativo de estaciones activas o viajes por día
MAE_ALERT_INCREASE = 0.20
SKILL_ALERT_MIN = 0.20
LEVEL_NAMES = {0: "-", 1: "advertencia", 2: "crítica"}

HISTORY_INPUT_COLUMNS = ["Origen_Id", "Fecha", TARGET_COLUMN, "day_of_week", "month",
                         "weather_zone"] + POI_COLUMNS
MONITORED = ["weather_zone"] + POI_COLUMNS + HISTORY_COLUMNS + [TARGET_COLUMN, "prediction"]
BASELINE_NAME = "Línea base (prueba ene-sep 2026)"
SCENARIO_TYPES = {
    BASELINE_NAME: "referencia",
    "Octubre real": "datos reales",
    "1. Rezagos faltantes": "entrada alterada",
    "2. Medias desplazadas": "entrada alterada",
    "3. Cambio estacional": "cambio real en la demanda",
}


# Datos del mes


def load_month_hourly(csv_path: Path, start: pd.Timestamp, end: pd.Timestamp):
    """Leer y limpiar un CSV mensual con la limpieza de la fase 1; devuelve viajes por hora."""
    original_period = (preprocessing.STUDY_START, preprocessing.STUDY_END)
    preprocessing.STUDY_START, preprocessing.STUDY_END = start, end
    try:
        reader, _ = read_csv_chunks(csv_path, chunk_size=CHUNK_SIZE)
        hourly_parts, summaries, seen_ids = [], [], set()
        for chunk in reader:
            cleaned, summary, new_ids = preprocessing.clean_strategy_b(
                preprocessing.normalize_chunk(chunk), seen_ids)
            seen_ids |= new_ids
            summaries.append(summary)
            hourly_parts.append(preprocessing.count_hourly_demand(
                preprocessing.add_trip_features(cleaned)))
    finally:
        preprocessing.STUDY_START, preprocessing.STUDY_END = original_period
    hourly = pd.concat(hourly_parts).groupby(level=[0, 1]).sum().reset_index()
    return hourly, pd.DataFrame(summaries).sum()


def build_month_daily(hourly: pd.DataFrame, integrated: pd.DataFrame,
                      training_stations: set) -> pd.DataFrame:
    """Demanda diaria del mes con los atributos más recientes de cada estación."""
    daily = build_daily_demand(hourly)
    attributes = (integrated.sort_values("Fecha").groupby("Origen_Id")[["weather_zone"] + POI_COLUMNS]
                  .last().reset_index())
    daily = daily.merge(attributes, on="Origen_Id", how="left", validate="many_to_one")
    daily["new_station"] = ~daily["Origen_Id"].isin(training_stations)
    return daily


def build_month_dataset(integrated: pd.DataFrame, month_daily: pd.DataFrame,
                        start: pd.Timestamp) -> pd.DataFrame:
    """Variables del mes usando la historia del integrado para los rezagos de los primeros días."""
    combined = pd.concat([integrated[HISTORY_INPUT_COLUMNS], month_daily[HISTORY_INPUT_COLUMNS]],
                         ignore_index=True)
    dataset = build_model_dataset(combined, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
    month = dataset.loc[dataset["Fecha"] >= start].reset_index(drop=True)
    month["new_station"] = month["Origen_Id"].isin(
        month_daily.loc[month_daily["new_station"], "Origen_Id"])
    return month


def prepare_month(model, integrated: pd.DataFrame, development: pd.DataFrame, csv_path: Path,
                  start: pd.Timestamp, end: pd.Timestamp):
    """Tabla del mes con la predicción del modelo en uso y la del ingenuo estacional."""
    hourly, cleaning = load_month_hourly(csv_path, start, end)
    month_daily = build_month_daily(hourly, integrated, set(development["Origen_Id"]))
    month = build_month_dataset(integrated, month_daily, start)
    month["predicted"] = np.clip(model.predict(month[FEATURE_COLUMNS]), 0, None)
    month["naive"] = predict_seasonal_naive(development, month, TARGET_COLUMN).to_numpy()
    return month, month_daily, cleaning


def monthly_performance(data: pd.DataFrame, predictions, naive) -> pd.DataFrame:
    """MAE, RMSE y mejora frente al ingenuo estacional para cada mes."""
    table = data[["Fecha", TARGET_COLUMN]].assign(pred=predictions, naive=naive)
    rows = []
    for month, part in table.groupby(table["Fecha"].dt.to_period("M")):
        model = regression_metrics(part[TARGET_COLUMN], part["pred"])
        baseline = regression_metrics(part[TARGET_COLUMN], part["naive"])
        rows.append({"mes": str(month), "filas": len(part), "mae": model["mae"],
                     "rmse": model["rmse"], "mae_ingenuo": baseline["mae"],
                     "mejora_vs_ingenuo": 1 - model["mae"] / baseline["mae"]})
    return pd.DataFrame(rows)


# Drift


def psi(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    """Índice de estabilidad con cortes por cuantiles de la referencia."""
    reference, current = reference.dropna(), current.dropna()
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        edges = np.unique(np.concatenate([edges, [reference.min() - 0.5, reference.max() + 0.5]]))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_share = np.histogram(reference, edges)[0] / len(reference)
    cur_share = np.histogram(current, edges)[0] / len(current)
    ref_share, cur_share = np.clip(ref_share, 1e-4, None), np.clip(cur_share, 1e-4, None)
    return float(np.sum((cur_share - ref_share) * np.log(cur_share / ref_share)))


def drift_table(reference: pd.DataFrame, current: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """KS, Wasserstein normalizado y PSI por variable; drift si KS o Wasserstein pasan 0.1."""
    rows = []
    for column in columns:
        ref, cur = reference[column].dropna(), current[column].dropna()
        ks = ks_2samp(ref, cur)
        scale = ref.std() if ref.std() > 0 else 1.0
        rows.append({
            "variable": column,
            "media_ref": ref.mean(), "media_actual": cur.mean(),
            "ks": ks.statistic, "ks_pvalue": ks.pvalue,
            "wasserstein_norm": wasserstein_distance(ref, cur) / scale,
            "psi": psi(ref, cur),
        })
    table = pd.DataFrame(rows)
    table["drift"] = (table["ks"] > KS_THRESHOLD) | (table["wasserstein_norm"] > WASSERSTEIN_THRESHOLD)
    return table


def seasonal_diagnosis(drift_vs_test: pd.DataFrame, drift_vs_season: pd.DataFrame) -> pd.DataFrame:
    """Separar cambios de temporada de cambios que también aparecen contra el mismo mes del año pasado."""
    comparison = drift_vs_test[["variable", "media_ref", "media_actual", "ks", "wasserstein_norm",
                                "psi", "drift"]].merge(
        drift_vs_season[["variable", "media_ref", "ks", "wasserstein_norm", "drift"]].rename(columns={
            "media_ref": "media_oct_2025", "ks": "ks_vs_2025",
            "wasserstein_norm": "wasserstein_vs_2025", "drift": "drift_vs_2025"}),
        on="variable",
    )
    comparison["diagnóstico"] = np.select(
        [comparison["drift"] & comparison["drift_vs_2025"],
         comparison["drift"] & ~comparison["drift_vs_2025"],
         ~comparison["drift"] & comparison["drift_vs_2025"]],
        ["revisar", "estacional", "cambio frente a 2025"],
        default="estable",
    )
    return comparison


def evidently_report(reference: pd.DataFrame, current: pd.DataFrame, path: Path) -> dict:
    """Reporte HTML de Evidently con el mismo umbral de proporción de variables."""
    from evidently import Report
    from evidently.presets import DataDriftPreset

    report = Report([DataDriftPreset(columns=MONITORED, drift_share=DATASET_DRIFT_SHARE)])
    snapshot = report.run(reference_data=reference[MONITORED], current_data=current[MONITORED])
    snapshot.save_html(str(path))
    summary = snapshot.dict()["metrics"][0]["value"]
    return {"variables_con_drift": int(summary["count"]), "proporción": summary["share"],
            "drift_del_conjunto": summary["share"] > DATASET_DRIFT_SHARE, "reporte": path.name}


# Escenarios sintéticos


def make_scenarios(current: pd.DataFrame, month_daily: pd.DataFrame, integrated: pd.DataFrame,
                   start: pd.Timestamp, drop_start: pd.Timestamp, seed: int = SEED) -> dict:
    """Escenarios 1 y 2 alteran solo la entrada; el 3 cambia la demanda y recalcula los rezagos."""
    rng = np.random.default_rng(seed)
    lag_columns = [f"lag_{lag}" for lag in LAGS]
    shifted_columns = lag_columns + [c for c in HISTORY_COLUMNS if c.startswith("roll_mean")]

    missing_lags = current.copy()
    lost = rng.random(len(missing_lags)) < 0.30
    missing_lags.loc[lost, lag_columns] = np.nan

    shifted_means = current.copy()
    shifted_means[shifted_columns] = shifted_means[shifted_columns] * 1.5

    seasonal_daily = month_daily.copy()
    dropped = seasonal_daily["Fecha"] >= drop_start
    seasonal_daily.loc[dropped, TARGET_COLUMN] = np.round(
        seasonal_daily.loc[dropped, TARGET_COLUMN] * 0.6).astype(int)
    combined = pd.concat([integrated[HISTORY_INPUT_COLUMNS], seasonal_daily[HISTORY_INPUT_COLUMNS]],
                         ignore_index=True)
    seasonal = build_model_dataset(combined, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
    seasonal = seasonal.loc[seasonal["Fecha"] >= start].reset_index(drop=True)
    return {"Octubre real": current, "1. Rezagos faltantes": missing_lags,
            "2. Medias desplazadas": shifted_means, "3. Cambio estacional": seasonal}


def evaluate_scenario(model, development: pd.DataFrame, reference: pd.DataFrame,
                      name: str, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Error, sesgo, rezagos vacíos y proporción de variables con drift de un escenario."""
    data = data.copy()
    data["prediction"] = np.clip(model.predict(data[FEATURE_COLUMNS]), 0, None)
    naive = predict_seasonal_naive(development, data, TARGET_COLUMN)
    metrics = regression_metrics(data[TARGET_COLUMN], data["prediction"])
    drift = drift_table(reference, data, MONITORED)
    row = {
        "escenario": name, "mae": metrics["mae"], "rmse": metrics["rmse"],
        "mejora_vs_ingenuo": 1 - metrics["mae"] / regression_metrics(data[TARGET_COLUMN], naive)["mae"],
        "sesgo": float((data["prediction"] - data[TARGET_COLUMN]).mean()),
        "lag_1_vacío": float(data["lag_1"].isna().mean()),
        "variables_con_drift": int(drift["drift"].sum()),
        "proporción_drift": float(drift["drift"].mean()),
    }
    return row, data


# Alertas y acción


def level(value: float, limits: tuple[float, float]) -> int:
    """0 sin alerta, 1 advertencia, 2 crítica."""
    warning, critical = limits
    return 2 if value > critical else 1 if value > warning else 0


def daily_operation(data: pd.DataFrame, target: str = TARGET_COLUMN) -> tuple[float, float]:
    """Estaciones activas por día y viajes totales por día."""
    by_day = data.groupby("Fecha")
    return by_day["Origen_Id"].nunique().mean(), by_day[target].sum().mean()


def performance_level(mae: float, skill: float, baseline_mae: float) -> int:
    """A3: advertencia con una de las dos condiciones, crítica con las dos."""
    return int(mae > (1 + MAE_ALERT_INCREASE) * baseline_mae) + int(skill < SKILL_ALERT_MIN)


def alert_levels(row: pd.Series, baseline_mae: float, baseline_missing: float,
                 reference_demand: float) -> dict:
    """Nivel de las cinco alertas para un periodo o escenario (una fila de la tabla de monitoreo)."""
    drift = level(row["psi_máximo"], PSI_LEVELS)
    if row["proporción_drift"] > DATASET_DRIFT_SHARE:
        drift = max(drift, 1)
    operation = max(abs(row["cambio_estaciones_activas"]), abs(row["cambio_viajes_por_día"]))
    return {
        "A1_datos_faltantes": level(row["lag_1_vacío"] - baseline_missing, MISSING_LEVELS),
        "A2_sesgo": level(abs(row["sesgo"]) / reference_demand, BIAS_LEVELS),
        "A3_desempeño": performance_level(row["mae"], row["mejora_vs_ingenuo"], baseline_mae),
        "A4_drift": drift,
        "A5_operación": level(operation, OPERATION_LEVELS),
    }


def decide(levels, previous_a3: int = 0) -> str:
    """Acción propuesta. Los problemas de datos van primero porque reentrenar no los arregla."""
    data_problem = max(levels["A1_datos_faltantes"], levels["A2_sesgo"])
    if data_problem == 2:
        return "Revisar el pipeline de datos (no reentrenar)"
    if data_problem == 1:
        return "Revisar los datos de entrada"
    if levels["A3_desempeño"] == 2:
        if previous_a3 == 2:
            return "Reentrenar: A3 crítica dos meses seguidos"
        return "Vigilar y confirmar con el siguiente mes"
    if levels["A5_operación"] == 2:
        return "Revisar con operación el cambio de la red"
    if max(levels.values() if isinstance(levels, dict) else levels) >= 1:
        return "Vigilar"
    return "Sin acción"


# Flujo completo


def run_monitoring(model, integrated: pd.DataFrame, dataset: pd.DataFrame,
                   development: pd.DataFrame, test: pd.DataFrame, csv_path: Path,
                   start: pd.Timestamp, end: pd.Timestamp, drop_start: pd.Timestamp,
                   output_dir: Path = DRIFT_REPORTS_DIR) -> dict:
    """Monitoreo de un mes con el modelo que estuvo en uso; guarda tablas y reportes en output_dir."""
    output_dir.mkdir(parents=True, exist_ok=True)
    month, month_daily, cleaning = prepare_month(model, integrated, development, csv_path, start, end)

    # Desempeño mensual contra el ingenuo estacional
    pred_test = np.clip(model.predict(test[FEATURE_COLUMNS]), 0, None)
    test_naive = predict_seasonal_naive(development, test, TARGET_COLUMN).to_numpy()
    performance = pd.concat([monthly_performance(test, pred_test, test_naive),
                             monthly_performance(month, month["predicted"], month["naive"])],
                            ignore_index=True)
    baseline_metrics = regression_metrics(test[TARGET_COLUMN], pred_test)
    baseline_skill = 1 - baseline_metrics["mae"] / regression_metrics(test[TARGET_COLUMN], test_naive)["mae"]

    # Drift contra la prueba y contra el mismo mes de 2025
    reference_test = test.assign(prediction=pred_test)
    season_start, season_end = start - pd.DateOffset(years=1), end - pd.DateOffset(years=1)
    same_month_last_year = dataset.loc[(dataset["Fecha"] >= season_start)
                                       & (dataset["Fecha"] < season_end)].copy()
    same_month_last_year["prediction"] = np.clip(
        model.predict(same_month_last_year[FEATURE_COLUMNS]), 0, None)
    current = month.assign(prediction=month["predicted"])
    drift_vs_test = drift_table(reference_test, current, MONITORED)
    drift_vs_season = drift_table(same_month_last_year, current, MONITORED)
    comparison = seasonal_diagnosis(drift_vs_test, drift_vs_season)
    comparison.to_csv(output_dir / "drift_october_2026.csv", index=False)
    evidently = pd.DataFrame([
        {"referencia": "vs_prueba_2026",
         **evidently_report(reference_test, current, output_dir / "evidently_vs_prueba_2026.html")},
        {"referencia": "vs_octubre_2025",
         **evidently_report(same_month_last_year, current, output_dir / "evidently_vs_octubre_2025.html")},
    ])

    # Escenarios
    scenarios = make_scenarios(current, month_daily, integrated, start, drop_start)
    rows, scenario_data = [], {}
    for name, data in scenarios.items():
        row, scenario_data[name] = evaluate_scenario(model, development, reference_test, name, data)
        rows.append(row)
    baseline_row = {
        "escenario": BASELINE_NAME, "mae": baseline_metrics["mae"], "rmse": baseline_metrics["rmse"],
        "mejora_vs_ingenuo": baseline_skill,
        "sesgo": float((reference_test["prediction"] - reference_test[TARGET_COLUMN]).mean()),
        "lag_1_vacío": float(test["lag_1"].isna().mean()), "variables_con_drift": 0,
        "proporción_drift": 0.0,
    }
    scenario_results = pd.DataFrame([baseline_row] + rows)
    scenario_results["mae_vs_base"] = scenario_results["mae"] / baseline_metrics["mae"] - 1
    scenario_results.insert(1, "tipo", scenario_results["escenario"].map(SCENARIO_TYPES))
    scenario_results.to_csv(output_dir / "scenario_results.csv", index=False)

    # Alertas
    reference_demand = reference_test[TARGET_COLUMN].mean()
    baseline_missing = float(test["lag_1"].isna().mean())
    reference_active, reference_trips = daily_operation(test)
    extra = []
    for name, data in {BASELINE_NAME: reference_test, **scenario_data}.items():
        active, trips = daily_operation(data)
        extra.append({
            "escenario": name,
            "psi_máximo": 0.0 if name == BASELINE_NAME else drift_table(reference_test, data, MONITORED)["psi"].max(),
            "estaciones_activas_día": active, "viajes_por_día": trips,
            "cambio_estaciones_activas": active / reference_active - 1,
            "cambio_viajes_por_día": trips / reference_trips - 1,
        })
    monitor_table = scenario_results.merge(pd.DataFrame(extra), on="escenario").set_index("escenario")
    alerts = pd.DataFrame({
        name: alert_levels(row, baseline_metrics["mae"], baseline_missing, reference_demand)
        for name, row in monitor_table.iterrows()
    }).T

    # Acción, con la regla de los dos meses para A3
    previous_month = (start - pd.DateOffset(months=1)).strftime("%Y-%m")
    previous = performance.loc[performance["mes"] == previous_month].iloc[0]
    previous_a3_real = performance_level(previous["mae"], previous["mejora_vs_ingenuo"],
                                         baseline_metrics["mae"])
    previous_a3 = pd.Series(0, index=alerts.index)
    previous_a3["Octubre real"] = previous_a3_real
    decisions = alerts.replace(LEVEL_NAMES)
    decisions["A3_mes_anterior"] = previous_a3.map(LEVEL_NAMES)
    decisions["acción"] = [decide(alerts.loc[name], previous_a3[name]) for name in alerts.index]
    decisions.to_csv(output_dir / "alerts_and_actions.csv")

    results = {
        "cleaning": cleaning, "month_daily": month_daily, "month": month,
        "performance": performance, "baseline_metrics": baseline_metrics,
        "baseline_skill": baseline_skill, "reference_test": reference_test,
        "same_month_last_year": same_month_last_year, "current": current,
        "comparison": comparison, "evidently": evidently, "scenario_results": scenario_results,
        "scenario_data": scenario_data, "monitor_table": monitor_table, "alerts": alerts,
        "previous_a3": previous_a3, "decisions": decisions,
    }
    plot_monitoring(results, output_dir, drop_start)
    return results


def plot_monitoring(results: dict, output_dir: Path, drop_start: pd.Timestamp) -> None:
    """Las tres gráficas del monitoreo: desempeño por mes, error diario por escenario y demanda."""
    performance, baseline_mae = results["performance"], results["baseline_metrics"]["mae"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(performance["mes"], performance["mae"], marker="o", label="MAE del modelo")
    ax1.plot(performance["mes"], performance["mae_ingenuo"], marker="o", linestyle="--", label="MAE del ingenuo")
    ax1.axhline((1 + MAE_ALERT_INCREASE) * baseline_mae, color="red", linestyle=":", label="Umbral A3")
    ax1.set_title("MAE por mes")
    ax1.set_ylabel("viajes por estación-día")
    ax1.tick_params(axis="x", rotation=45)
    ax1.legend()
    ax2.bar(performance["mes"], performance["mejora_vs_ingenuo"] * 100,
            color=["tab:red" if v < SKILL_ALERT_MIN else "tab:blue" for v in performance["mejora_vs_ingenuo"]])
    ax2.axhline(SKILL_ALERT_MIN * 100, color="red", linestyle=":", label="Umbral A3")
    ax2.set_title("Mejora frente al ingenuo estacional")
    ax2.set_ylabel("%")
    ax2.tick_params(axis="x", rotation=45)
    ax2.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "monthly_performance.png", dpi=150)
    plt.close(fig)

    daily_mae = pd.DataFrame({
        name: data.assign(error=(data["prediction"] - data[TARGET_COLUMN]).abs()).groupby("Fecha")["error"].mean()
        for name, data in results["scenario_data"].items()
    })
    fig, ax = plt.subplots(figsize=(12, 4))
    for name in daily_mae.columns:
        ax.plot(daily_mae.index, daily_mae[name], label=name)
    ax.axhline(baseline_mae, color="gray", linestyle="--", label="MAE de la línea base")
    ax.axvline(drop_start, color="black", linestyle=":", label="Inicio del cambio estacional")
    ax.set_title("Error absoluto medio por día, octubre de 2026")
    ax.set_ylabel("viajes por estación-día")
    ax.legend(ncol=3, fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "daily_error_scenarios.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.arange(0, 201, 5)
    for name, data in [("Prueba ene-sep 2026", results["reference_test"]),
                       ("Octubre 2025", results["same_month_last_year"]), ("Octubre 2026", results["current"])]:
        ax.hist(data[TARGET_COLUMN].clip(upper=200), bins=bins, density=True, histtype="step",
                linewidth=1.5, label=f"{name} (media {data[TARGET_COLUMN].mean():.1f})")
    ax.set_title("Distribución de la demanda diaria por estación")
    ax.set_xlabel("viajes por estación-día (recortado en 200)")
    ax.set_ylabel("densidad")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "demand_distribution.png", dpi=150)
    plt.close(fig)


MONITORING_ARTIFACT_GUIDE = """# Guía de artefactos del monitoreo

- `monitoring/performance`: MAE y mejora frente al ingenuo por mes, de enero a octubre de 2026.
- `monitoring/drift`: drift por variable con su diagnóstico, distribución de la demanda y reportes de Evidently contra la prueba 2026 y contra octubre de 2025.
- `monitoring/scenarios`: resultados de los escenarios sintéticos y su error diario.
- `monitoring/alerts`: nivel de las alertas A1 a A5 por escenario y la acción propuesta.
- `data`: inventario, versiones y procedencia de los datos, igual que en la Fase 2.
- `environment` y `documentation`: entorno, commit y configuración, igual que en la Fase 2.
- `features.json`: variables del modelo.
- `artifact_catalog.json`: lista de archivos con su origen y SHA-256.

En Model metrics, `monthly_mae` y `monthly_skill_vs_naive` son series por mes (paso 0 = enero de 2026).
Cada escenario tiene una ejecución hija con sus métricas y el nivel de cada alerta en las etiquetas.
"""
MONITORING_ARTIFACTS = {
    "monitoring/performance": ["monthly_performance.png"],
    "monitoring/drift": ["drift_october_2026.csv", "demand_distribution.png",
                         "evidently_vs_prueba_2026.html", "evidently_vs_octubre_2025.html"],
    "monitoring/scenarios": ["scenario_results.csv", "daily_error_scenarios.png"],
    "monitoring/alerts": ["alerts_and_actions.csv"],
}
MONITORING_RUN = "Monitoreo | Random Forest v{version} | octubre 2026"


def log_monitoring(results: dict, model_uri: str, csv_name: str, output_dir: Path = DRIFT_REPORTS_DIR,
                   project_root: Path = PROJECT_ROOT) -> str:
    """Registrar el monitoreo en MLflow con la misma estructura de la Fase 2."""
    thresholds = {
        "missing_warning": MISSING_LEVELS[0], "missing_critical": MISSING_LEVELS[1],
        "bias_warning": BIAS_LEVELS[0], "bias_critical": BIAS_LEVELS[1],
        "mae_alert_increase": MAE_ALERT_INCREASE, "skill_alert_min": SKILL_ALERT_MIN,
        "psi_warning": PSI_LEVELS[0], "psi_critical": PSI_LEVELS[1],
        "operation_warning": OPERATION_LEVELS[0], "operation_critical": OPERATION_LEVELS[1],
        "dataset_drift_share": DATASET_DRIFT_SHARE, "ks_threshold": KS_THRESHOLD,
        "wasserstein_threshold": WASSERSTEIN_THRESHOLD,
    }
    metrics = results["baseline_metrics"]
    data_path = project_root / INTEGRATED_PATH.relative_to(PROJECT_ROOT)
    data_sha256 = tracking.calculate_sha256(data_path)
    model_version = model_uri.rsplit("/", 1)[-1]
    run_name = MONITORING_RUN.format(version=model_version)
    model_columns = [TARGET_COLUMN] + FEATURE_COLUMNS
    tracking.setup_mlflow(MLFLOW_TRACKING_URI, MONITORING_EXPERIMENT, project_root / MLFLOW_ARTIFACT_DIR.name)
    mlflow.set_experiment(MONITORING_EXPERIMENT)
    mlflow_catalog.mark_previous_runs(MONITORING_EXPERIMENT, run_name)
    with mlflow.start_run(run_name=run_name, tags=mlflow_catalog.source_tags("mibici_mlops/monitoring.py")) as parent:
        tracking.log_run_context(project_root, data_path, data_sha256)
        mlflow.set_tags({"stage": "monitoring", "phase": "fase_3", "model_uri": model_uri,
                         "model_version": model_version, "monitor_period": "2026-10",
                         "reference_period": "2026-01 a 2026-09", "seasonal_reference": "2025-10"})
        mlflow.set_tag("mlflow.note.content", (
            f"Monitoreo de octubre de 2026 con {model_uri}, el modelo en uso ese mes. Data drift contra la "
            "prueba 2026 y contra octubre de 2025, tres escenarios sintéticos, alertas con dos niveles y "
            "acción propuesta. Cada escenario es una ejecución hija."
        ))
        mlflow_catalog.log_data_lineage(project_root, data_path, data_sha256, "monitoring", {
            "reference": ("mibici_test_2026", results["reference_test"][model_columns], data_path),
            "monitoring": ("mibici_monitoreo_2026_10", results["month"][model_columns], csv_name),
        })
        mlflow.log_text(MONITORING_ARTIFACT_GUIDE, "README.md")
        mlflow.log_dict({"features": FEATURE_COLUMNS}, "features.json")
        mlflow.log_params({**thresholds, "monitor_csv": csv_name, "monitor_rows": len(results["current"]),
                           "new_stations": int(results["month_daily"].loc[
                               results["month_daily"]["new_station"], "Origen_Id"].nunique())})
        mlflow.log_metrics({"baseline_mae": metrics["mae"], "baseline_rmse": metrics["rmse"],
                            "baseline_skill_vs_naive": results["baseline_skill"]})
        for step, row in enumerate(results["performance"].itertuples(index=False)):
            mlflow.log_metric("monthly_mae", float(row.mae), step=step)
            mlflow.log_metric("monthly_skill_vs_naive", float(row.mejora_vs_ingenuo), step=step)
        mlflow_catalog.log_artifact_catalog(
            {folder: [output_dir / name for name in names] for folder, names in MONITORING_ARTIFACTS.items()},
            project_root)
        for name, row in results["monitor_table"].iloc[1:].iterrows():
            with mlflow.start_run(run_name=f"Escenario | Random Forest v{model_version} | {name}", nested=True,
                                  tags=mlflow_catalog.source_tags("mibici_mlops/monitoring.py")):
                mlflow.set_tags({**mlflow_catalog.trace_tags(parent.info.run_id),
                                 "stage": "monitoring_scenario", "phase": "fase_3", "model_uri": model_uri,
                                 "model_version": model_version, "scenario": name,
                                 "scenario_type": SCENARIO_TYPES[name],
                                 "action": results["decisions"].loc[name, "acción"],
                                 **{f"alert_{flag.split('_')[0]}": LEVEL_NAMES[int(level)].replace("-", "sin alerta")
                                    for flag, level in results["alerts"].loc[name].items()}})
                mlflow.log_param("scenario", name)
                mlflow.log_metrics({
                    "mae": row["mae"], "rmse": row["rmse"], "skill_vs_naive": row["mejora_vs_ingenuo"],
                    "bias": row["sesgo"], "lag_1_missing": row["lag_1_vacío"],
                    "drift_share": row["proporción_drift"], "mae_vs_baseline": row["mae_vs_base"],
                    "psi_max": row["psi_máximo"], "active_stations_change": row["cambio_estaciones_activas"],
                    "daily_trips_change": row["cambio_viajes_por_día"],
                    **{flag: float(value) for flag, value in results["alerts"].loc[name].items()},
                })
    return parent.info.run_id


def load_context(model_version: int = PHASE2_MODEL_VERSION) -> dict:
    """Datos integrados, separación temporal y el modelo que estuvo en uso durante el mes."""
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    integrated = pd.read_parquet(INTEGRATED_PATH)
    dataset = build_model_dataset(integrated, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
    development, test = split_by_date(dataset, TEST_START)
    model_uri = f"models:/{REGISTERED_MODEL_NAME}/{model_version}"
    return {"integrated": integrated, "dataset": dataset, "development": development, "test": test,
            "model": mlflow.sklearn.load_model(model_uri), "model_uri": model_uri,
            "model_version": model_version}


def main() -> None:
    """Monitorear octubre de 2026 con la v7 y registrar el resultado en MLflow."""
    context = load_context()
    results = run_monitoring(context["model"], context["integrated"], context["dataset"],
                             context["development"], context["test"], MONITOR_CSV,
                             MONITOR_START, MONITOR_END, DROP_START)
    run_id = log_monitoring(results, context["model_uri"], MONITOR_CSV.name)
    print(results["decisions"][["A3_mes_anterior", "acción"]].to_string())
    print("Ejecución de monitoreo:", run_id)


if __name__ == "__main__":
    main()

Writing ../build/migracion/mibici_mlops/monitoring.py


## 7. `mibici_mlops/modeling/challenger.py`
Corresponde a la sección 5.15: los dos challengers con los hiperparámetros de la v7, la selección en septiembre, la prueba en octubre contra la v7 y la regla de promoción.

- `run_challenger` devuelve las dos tablas, el elegido, la mejora y si cumple la regla.
- `promotion_decision` da las mismas cuatro decisiones del 07: ya promovido, revisar con el equipo, promover o mantener champion.
- `log_challenger` siempre registra la comparación en `MiBici_Fase3_Champion_Challenger`, con el mismo contexto de la Fase 2 el linaje de datos y las tablas en `modeling/comparison` y `modeling/test`, las mismas carpetas del modelo final de la Fase 2. Una nueva versión se registra únicamente cuando la decisión es promover y `auto_promote` es `True`; en ese caso usa `tracking.promote_model`, igual que la Fase 2, para la descripción, las etiquetas y el alias.

Desde la terminal se corre con `python -m mibici_mlops.modeling.challenger` o `make challenger`. Usa `AUTO_PROMOTE` de `config.py`.

In [8]:
%%writefile ../build/migracion/mibici_mlops/modeling/challenger.py
"""Champion contra challenger: selección en un mes, prueba en el siguiente y regla de promoción."""

from pathlib import Path
import time

import cloudpickle
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from mibici_mlops import features as features_module
from mibici_mlops import mlflow_catalog, monitoring, tracking
from mibici_mlops.config import (
    AUTO_PROMOTE,
    CHALLENGER_EXPERIMENT,
    DRIFT_REPORTS_DIR,
    FEATURE_COLUMNS,
    INTEGRATED_PATH,
    MIN_IMPROVEMENT,
    MLFLOW_ARTIFACT_DIR,
    MLFLOW_TRACKING_URI,
    MODEL_ALIAS,
    PROJECT_ROOT,
    REGISTERED_MODEL_NAME,
    TARGET_COLUMN,
)
from mibici_mlops.features import add_holidays
from mibici_mlops.modeling.evaluate import predict_seasonal_naive, regression_metrics
from mibici_mlops.modeling.train import build_model_pipeline

SELECTION_START = pd.Timestamp("2026-09-01")
CANDIDATES = {"Challenger A (reentrenado)": False, "Challenger B (+ festivos)": True}
SOURCE_TAG = "challenger_octubre_2026"
MONTH_NAMES = {1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio", 7: "julio",
               8: "agosto", 9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"}
RF_LOGGED_PARAMS = ("n_estimators", "max_depth", "max_features", "max_samples", "min_samples_leaf")


def make_challenger(base_model, with_holidays: bool) -> Pipeline:
    """Pipeline con los hiperparámetros del modelo de referencia; B calcula is_holiday dentro."""
    if not with_holidays:
        return build_model_pipeline(clone(base_model), FEATURE_COLUMNS)
    base = build_model_pipeline(clone(base_model), FEATURE_COLUMNS + ["is_holiday"])
    return Pipeline([("holidays", FunctionTransformer(add_holidays)), *base.steps])


def fit_challenger(name: str, data: pd.DataFrame, base_model) -> Pipeline:
    start = time.perf_counter()
    model = make_challenger(base_model, CANDIDATES[name]).fit(data[FEATURE_COLUMNS], data[TARGET_COLUMN])
    print(f"{name}: {len(data):,} filas hasta {data['Fecha'].max().date()} | {time.perf_counter() - start:.0f} s")
    return model


def month_metrics(name: str, data: pd.DataFrame, predictions, naive) -> dict:
    """MAE, RMSE, mejora frente al ingenuo y sesgo de un modelo en el mes de selección."""
    predictions = np.clip(predictions, 0, None)
    model = regression_metrics(data[TARGET_COLUMN], predictions)
    return {"modelo": name, "mae": model["mae"], "rmse": model["rmse"],
            "mejora_vs_ingenuo": 1 - model["mae"] / regression_metrics(data[TARGET_COLUMN], naive)["mae"],
            "sesgo": float((predictions - data[TARGET_COLUMN]).mean())}


def evaluate_month(name: str, month: pd.DataFrame, predictions, special_day: str) -> dict:
    """Métricas del mes de prueba, por tipo de estación y en un día especial de referencia."""
    predictions = np.clip(predictions, 0, None)
    y = month[TARGET_COLUMN]
    known, new = ~month["new_station"], month["new_station"]
    special = month["Fecha"] == special_day
    model = regression_metrics(y, predictions)
    return {
        "modelo": name, "mae": model["mae"], "rmse": model["rmse"],
        "mejora_vs_ingenuo": 1 - model["mae"] / regression_metrics(y, month["naive"])["mae"],
        "mae_con_historia": regression_metrics(y[known], predictions[known])["mae"],
        "mae_nuevas": regression_metrics(y[new], predictions[new])["mae"],
        "mae_12_oct": regression_metrics(y[special], predictions[special])["mae"],
        "sesgo": float((predictions - y).mean()),
    }


def run_challenger(reference_model, reference_version, dataset: pd.DataFrame, month: pd.DataFrame,
                   selection_start: pd.Timestamp, train_end: pd.Timestamp,
                   output_dir: Path = DRIFT_REPORTS_DIR, special_day: str = "2026-10-12") -> dict:
    """Elegir entre A y B en el mes de selección y probar al elegido contra la referencia en month."""
    output_dir.mkdir(parents=True, exist_ok=True)
    base_model = reference_model.named_steps["model"]

    # 1. Selección: A y B entrenados hasta el día anterior a selection_start
    train_selection = dataset.loc[dataset["Fecha"] < selection_start]
    selection_month = dataset.loc[(dataset["Fecha"] >= selection_start) & (dataset["Fecha"] < train_end)]
    selection_models = {name: fit_challenger(name, train_selection, base_model) for name in CANDIDATES}
    selection_naive = predict_seasonal_naive(train_selection, selection_month, TARGET_COLUMN)
    selection_table = pd.DataFrame(
        [month_metrics(f"v{reference_version} (referencia)", selection_month,
                       reference_model.predict(selection_month[FEATURE_COLUMNS]), selection_naive)]
        + [month_metrics(name, selection_month, model.predict(selection_month[FEATURE_COLUMNS]), selection_naive)
           for name, model in selection_models.items()]
    ).set_index("modelo")
    best_name = selection_table.loc[list(CANDIDATES), "mae"].idxmin()
    selection_table.to_csv(output_dir / "challenger_selection_september.csv")

    # 2. Prueba: el elegido se reentrena hasta train_end y se compara con la referencia en month
    train_recent = dataset.loc[dataset["Fecha"] < train_end]
    final_model = fit_challenger(best_name, train_recent, base_model)
    month_name = MONTH_NAMES[month["Fecha"].min().month]
    train_month = MONTH_NAMES[train_recent["Fecha"].max().month]
    comparison = pd.DataFrame([
        evaluate_month(f"v{reference_version} (en uso en {month_name})", month,
                       month["predicted"].to_numpy(), special_day),
        evaluate_month(f"{best_name}, entrenado hasta {train_month}", month,
                       final_model.predict(month[FEATURE_COLUMNS]), special_day),
    ]).set_index("modelo")
    comparison.to_csv(output_dir / "challenger_test_october.csv")

    reference_row, challenger_row = comparison.iloc[0], comparison.iloc[1]
    improvement = 1 - challenger_row["mae"] / reference_row["mae"]
    return {
        "selection_table": selection_table, "best_name": best_name, "comparison": comparison,
        "final_model": final_model, "base_model": base_model, "train_rows": len(train_recent),
        "train_end": str(train_recent["Fecha"].max().date()), "improvement": improvement,
        "wins": bool(improvement >= MIN_IMPROVEMENT and challenger_row["rmse"] <= reference_row["rmse"]),
    }


def promotion_decision(results: dict, alias_version, reference_version) -> tuple[str, str]:
    """Aplicar la regla y revisar si el champion actual ya salió de esta comparación."""
    reference_row, challenger_row = results["comparison"].iloc[0], results["comparison"].iloc[1]
    best_name, improvement, wins = results["best_name"], results["improvement"], results["wins"]
    from_this_comparison = alias_version.tags.get("source") == SOURCE_TAG
    same_candidate = alias_version.tags.get("candidate") == best_name

    if wins and from_this_comparison and same_candidate:
        return "ya promovido", f"el champion v{alias_version.version} es {best_name} entrenado hasta septiembre"
    if wins and from_this_comparison:
        return "revisar con el equipo", (f"gana {best_name}, pero el champion v{alias_version.version} es "
                                         f"{alias_version.tags.get('candidate')}")
    if wins:
        return "promover", (f"MAE {improvement:.1%} menor que v{reference_version} y RMSE "
                            f"{reference_row['rmse']:.2f} a {challenger_row['rmse']:.2f}")
    if from_this_comparison:
        return "revisar con el equipo", (f"el challenger ya no gana a la v{reference_version}, pero el "
                                         f"champion es v{alias_version.version}")
    return "mantener champion", (f"mejora de MAE {improvement:.1%} (mínimo {MIN_IMPROVEMENT:.0%}) o RMSE peor "
                                 f"({reference_row['rmse']:.2f} a {challenger_row['rmse']:.2f})")


CHALLENGER_ARTIFACT_GUIDE = """# Guía de artefactos de champion contra challenger

- `modeling/comparison`: MAE, RMSE y mejora frente al ingenuo de la v7 y los dos challengers en septiembre.
- `modeling/test`: el mejor challenger contra la v7 en octubre de 2026.
- `data`: inventario, versiones y procedencia de los datos, igual que en la Fase 2.
- `environment` y `documentation`: entorno, commit y configuración, igual que en la Fase 2.
- `artifact_catalog.json`: lista de archivos con su origen y SHA-256.

La decisión, el candidato y el motivo están en las etiquetas `decision`, `candidate` y `reason`.
"""
CHALLENGER_RUN = "Champion contra challenger | Random Forest | octubre 2026"


def log_challenger(results: dict, decision: str, reason: str, alias_version, reference_uri: str,
                   input_example: pd.DataFrame, auto_promote: bool = AUTO_PROMOTE,
                   output_dir: Path = DRIFT_REPORTS_DIR, project_root: Path = PROJECT_ROOT,
                   month: pd.DataFrame | None = None) -> dict:
    """Registrar la comparación en MLflow y, si se promueve, la versión nueva con el alias champion."""
    promote_now = decision == "promover" and auto_promote
    best_name = results["best_name"]
    data_path = project_root / INTEGRATED_PATH.relative_to(PROJECT_ROOT)
    data_sha256 = tracking.calculate_sha256(data_path)
    tracking.setup_mlflow(MLFLOW_TRACKING_URI, CHALLENGER_EXPERIMENT, project_root / MLFLOW_ARTIFACT_DIR.name)
    mlflow.set_experiment(CHALLENGER_EXPERIMENT)
    mlflow_catalog.mark_previous_runs(CHALLENGER_EXPERIMENT, CHALLENGER_RUN)
    with mlflow.start_run(run_name=CHALLENGER_RUN,
                          tags=mlflow_catalog.source_tags("mibici_mlops/modeling/challenger.py")) as run:
        tracking.log_run_context(project_root, data_path, data_sha256)
        mlflow.set_tag("mlflow.note.content", (
            f"Comparación del champion contra dos challengers. Selección en septiembre, prueba en octubre contra "
            f"{reference_uri}. Decisión: {decision} ({reason})."
        ))
        mlflow.set_tags({"stage": "champion_challenger", "phase": "fase_3", "model_role": "challenger",
                         "model_version": str(alias_version.version), "source": SOURCE_TAG, "reference_uri": reference_uri, "deployed_champion": alias_version.version,
                         "candidate": best_name, "decision": decision, "reason": reason,
                         "auto_promote": str(auto_promote), "selection_period": "2026-09",
                         "test_period": "2026-10", "train_end": results["train_end"]})
        mlflow.log_params({"min_improvement": MIN_IMPROVEMENT, "train_rows": results["train_rows"],
                           "holidays": CANDIDATES[best_name],
                           **{f"rf_{k}": v for k, v in results["base_model"].get_params().items()
                              if k in RF_LOGGED_PARAMS}})
        for name, row in results["selection_table"].iterrows():
            prefix = "v7" if name.startswith("v") else ("challenger_b" if "B" in name else "challenger_a")
            mlflow.log_metrics({f"sep_{prefix}_{k}": float(v) for k, v in row.items()})
        for prefix, (_, row) in zip(["v7", "challenger"], results["comparison"].iterrows()):
            mlflow.log_metrics({f"oct_{prefix}_{k}": float(v) for k, v in row.items()})
        mlflow.log_metrics({"mae_improvement": results["improvement"],
                            "challenger_wins": float(results["wins"])})
        datasets = {}
        if month is not None:
            datasets["testing"] = ("mibici_monitoreo_2026_10", month[[TARGET_COLUMN] + FEATURE_COLUMNS],
                                   monitoring.MONITOR_CSV.name)
        mlflow_catalog.log_data_lineage(project_root, data_path, data_sha256, "challenger_test", datasets)
        mlflow.log_text(CHALLENGER_ARTIFACT_GUIDE, "README.md")
        mlflow.log_dict({"features": FEATURE_COLUMNS}, "features.json")
        mlflow_catalog.log_artifact_catalog({
            "modeling/comparison": [output_dir / "challenger_selection_september.csv"],
            "modeling/test": [output_dir / "challenger_test_october.csv"],
        }, project_root)
        model_info = None
        if promote_now:
            # add_holidays va dentro del modelo porque la imagen no tiene mibici_mlops
            cloudpickle.register_pickle_by_value(features_module)
            try:
                model_info = mlflow.sklearn.log_model(
                    results["final_model"], name="model", registered_model_name=REGISTERED_MODEL_NAME,
                    serialization_format="cloudpickle", input_example=input_example,
                    signature=infer_signature(input_example, results["final_model"].predict(input_example)),
                )
            finally:
                cloudpickle.unregister_pickle_by_value(features_module)

    new_version = None
    if model_info is not None:
        new_version = model_info.registered_model_version
        version_tags = {"source": SOURCE_TAG, "decision": decision, "reason": reason,
                        "replaces_version": alias_version.version, "candidate": best_name,
                        "train_end": results["train_end"]}
        mlflow.MlflowClient().set_registered_model_alias(REGISTERED_MODEL_NAME, "challenger", new_version)
        tracking.promote_model(REGISTERED_MODEL_NAME, new_version, MODEL_ALIAS,
                               f"{best_name} entrenado hasta {results['train_end']}. {reason}.", version_tags)
    return {"run_id": run.info.run_id, "new_version": new_version, "promoted": new_version is not None}


def main(auto_promote: bool = AUTO_PROMOTE) -> None:
    """Comparar challengers contra la v7 y promover solo si gana y auto_promote es True."""
    context = monitoring.load_context()
    month, _, _ = monitoring.prepare_month(context["model"], context["integrated"], context["development"],
                                           monitoring.MONITOR_CSV, monitoring.MONITOR_START,
                                           monitoring.MONITOR_END)
    results = run_challenger(context["model"], context["model_version"], context["dataset"], month,
                             SELECTION_START, monitoring.MONITOR_START)
    alias_version = mlflow.MlflowClient().get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
    decision, reason = promotion_decision(results, alias_version, context["model_version"])
    logged = log_challenger(results, decision, reason, alias_version, context["model_uri"],
                            month[FEATURE_COLUMNS].head(5), auto_promote, month=month)
    print(results["comparison"].round(3).to_string())
    print(f"Decisión: {decision} | {reason}")
    if logged["promoted"]:
        print(f"Alias {MODEL_ALIAS}: versión {alias_version.version} cambia a {logged['new_version']}")


if __name__ == "__main__":
    main()

Writing ../build/migracion/mibici_mlops/modeling/challenger.py


## 8. `mibici_mlops/modeling/champion_report.py`
Agregamos este archivo para registrar en MLflow los resultados del champion, siguiendo la organización que usamos en la fase 2. `build_champion_report` compara el champion con la v7 en octubre y guarda en `reports/champion/` las tablas (selección en septiembre, comparación en octubre, error diario e importancia de variables) y cuatro gráficas.

`log_champion` crea la ejecución "Modelo champion" en `MiBici_Fase3_Champion_Challenger` con el contexto de la Fase 2, descripción, `README.md`, `features.json`, `documentation/model_configuration.json` (hiperparámetros, regla de festivos, periodo de entrenamiento), métricas de septiembre y octubre, el error diario como serie en Model metrics y las tablas y gráficas en `modeling/comparison`, `modeling/test` y `modeling/interpretation`. No registra una versión nueva: agrega a la versión del champion en el Model Registry una descripción y la etiqueta `documentation_run` con el enlace a esta ejecución. Desde la terminal: `make champion`.

In [9]:
%%writefile ../build/migracion/mibici_mlops/modeling/champion_report.py
"""Evidencia del champion en MLflow: métricas, gráficas y configuración, con la estructura de la Fase 2."""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from mibici_mlops import mlflow_catalog, tracking
from mibici_mlops.config import (
    CHALLENGER_EXPERIMENT,
    CHAMPION_REPORTS_DIR,
    FEATURE_COLUMNS,
    HORIZON_DAYS,
    INTEGRATED_PATH,
    MLFLOW_ARTIFACT_DIR,
    MLFLOW_TRACKING_URI,
    MODEL_ALIAS,
    PROJECT_ROOT,
    REGISTERED_MODEL_NAME,
    SEED,
    TARGET_COLUMN,
)
from mibici_mlops.features import HOLIDAY_DATES, HOLIDAY_MONDAYS
from mibici_mlops.modeling.challenger import evaluate_month
from mibici_mlops.modeling.evaluate import regression_metrics

CHAMPION_ARTIFACT_GUIDE = """# Guía de artefactos del champion

- `modeling/comparison`: selección en septiembre de la v7 y los challengers, con su gráfica junto a la prueba de octubre.
- `modeling/test`: champion contra la v7 en octubre de 2026, error diario y predicción contra viajes reales.
- `modeling/interpretation`: importancia de variables del champion.
- `data`: inventario, versiones y procedencia de los datos, igual que en la Fase 2.
- `artifact_catalog.json`: lista de archivos con su origen y SHA-256.
- `features.json` y `documentation/model_configuration.json`: variables, hiperparámetros, regla de días festivos y periodo de entrenamiento.
- `environment` y `documentation/run_configuration.json`: entorno, commit y datos, igual que en la Fase 2.

El modelo está en el Model Registry (`mibici_salidas_diarias`, alias `champion`). Esta ejecución documenta
esa versión; no registra una versión nueva. En Model metrics, `october_daily_mae` es una serie por día
(paso 0 = 1 de octubre de 2026) para el champion y `october_daily_mae_v7` para la v7.
"""


def feature_importance(model) -> pd.DataFrame:
    """Importancia de cada variable del Random Forest después del preprocesamiento."""
    preprocessing, forest = model.named_steps["preprocessing"], model.named_steps["model"]
    try:
        names = [name.split("__", 1)[-1] for name in preprocessing.get_feature_names_out()]
    except (AttributeError, ValueError):
        names = [f"variable_{i}" for i in range(len(forest.feature_importances_))]
    table = pd.DataFrame({"variable": names, "importancia": forest.feature_importances_})
    return table.sort_values("importancia", ascending=False, ignore_index=True)


def build_champion_report(champion_model, champion_version, reference_version, month: pd.DataFrame,
                          selection_table: pd.DataFrame | None = None,
                          output_dir: Path = CHAMPION_REPORTS_DIR) -> dict:
    """Comparar el champion con la v7 en el mes monitoreado y guardar tablas y gráficas.

    month es la tabla del mes con las columnas predicted (v7), naive y new_station.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    champion_pred = np.clip(champion_model.predict(month[FEATURE_COLUMNS]), 0, None)
    reference_label = f"v{reference_version} (en uso en octubre)"
    champion_label = f"v{champion_version} (champion)"
    comparison = pd.DataFrame([
        evaluate_month(reference_label, month, month["predicted"].to_numpy(), "2026-10-12"),
        evaluate_month(champion_label, month, champion_pred, "2026-10-12"),
    ]).set_index("modelo")
    comparison.to_csv(output_dir / "october_comparison.csv")

    errors = month[["Fecha", TARGET_COLUMN]].assign(
        v7=(month["predicted"] - month[TARGET_COLUMN]).abs(), champion=np.abs(champion_pred - month[TARGET_COLUMN]))
    daily = errors.groupby("Fecha")[["v7", "champion"]].mean()
    daily.to_csv(output_dir / "october_daily_mae.csv")
    importance = feature_importance(champion_model)
    importance.to_csv(output_dir / "feature_importance.csv", index=False)
    if selection_table is not None:
        selection_table.to_csv(output_dir / "selection_september.csv")

    # Selección y prueba
    fig, axes = plt.subplots(1, 2 if selection_table is not None else 1, figsize=(12, 4), squeeze=False)
    if selection_table is not None:
        axes[0, 0].bar(selection_table.index, selection_table["mae"], color=["gray", "tab:blue", "tab:green"])
        axes[0, 0].set_title("Selección: MAE en septiembre de 2026")
        axes[0, 0].tick_params(axis="x", rotation=15)
    ax = axes[0, -1]
    ax.bar(comparison.index, comparison["mae"], color=["gray", "tab:green"])
    ax.set_title("Prueba: MAE en octubre de 2026")
    for ax in axes[0]:
        ax.set_ylabel("viajes por estación-día")
    fig.tight_layout()
    fig.savefig(output_dir / "selection_and_test.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(daily.index, daily["v7"], label=reference_label, color="gray")
    ax.plot(daily.index, daily["champion"], label=champion_label, color="tab:green")
    ax.set_title("Error absoluto medio por día, octubre de 2026")
    ax.set_ylabel("viajes por estación-día")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "october_daily_error.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 6))
    limit = float(np.quantile(month[TARGET_COLUMN], 0.995))
    ax.scatter(month[TARGET_COLUMN], champion_pred, s=4, alpha=0.3)
    ax.plot([0, limit], [0, limit], color="red", linestyle=":")
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_title("Champion: predicción contra viajes reales, octubre de 2026")
    ax.set_xlabel("viajes reales")
    ax.set_ylabel("viajes predichos")
    fig.tight_layout()
    fig.savefig(output_dir / "october_pred_vs_real.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6))
    top = importance.head(15).iloc[::-1]
    ax.barh(top["variable"], top["importancia"])
    ax.set_title("Importancia de variables del champion")
    fig.tight_layout()
    fig.savefig(output_dir / "feature_importance.png", dpi=150)
    plt.close(fig)

    return {"comparison": comparison, "daily": daily, "importance": importance,
            "selection_table": selection_table, "output_dir": output_dir, "month": month}


def log_champion(report: dict, champion_version, champion_model, output_dir: Path = CHAMPION_REPORTS_DIR,
                 project_root: Path = PROJECT_ROOT) -> str:
    """Ejecución que documenta al champion y enlace desde su versión en el Model Registry."""
    version = str(champion_version.version)
    model_uri = f"models:/{REGISTERED_MODEL_NAME}/{version}"
    reference_row, champion_row = report["comparison"].iloc[0], report["comparison"].iloc[1]
    improvement = 1 - champion_row["mae"] / reference_row["mae"]
    forest = champion_model.named_steps["model"]
    candidate = champion_version.tags.get("candidate") or type(forest).__name__
    train_end = champion_version.tags.get("train_end", "")
    summary = f"{candidate}, entrenado hasta {train_end}" if train_end else candidate

    data_path = project_root / INTEGRATED_PATH.relative_to(PROJECT_ROOT)
    data_sha256 = tracking.calculate_sha256(data_path)
    run_name = f"Modelo champion | Random Forest | v{version}"
    tracking.setup_mlflow(MLFLOW_TRACKING_URI, CHALLENGER_EXPERIMENT, project_root / MLFLOW_ARTIFACT_DIR.name)
    mlflow.set_experiment(CHALLENGER_EXPERIMENT)
    mlflow_catalog.mark_previous_runs(CHALLENGER_EXPERIMENT, run_name)
    with mlflow.start_run(run_name=run_name,
                          tags=mlflow_catalog.source_tags("mibici_mlops/modeling/champion_report.py")) as run:
        tracking.log_run_context(project_root, data_path, data_sha256)
        mlflow.set_tags({"stage": "champion", "phase": "fase_3", "model_role": "champion",
                         "model_version": version, "model_family": type(forest).__name__,
                         "registered_model_name": REGISTERED_MODEL_NAME,
                         "registered_model_version": version, "registered_model_alias": MODEL_ALIAS,
                         "model_uri": model_uri, "model_run_id": champion_version.run_id,
                         "candidate": candidate, "train_end": train_end})
        mlflow_catalog.log_data_lineage(project_root, data_path, data_sha256, "champion_test", {
            "testing": ("mibici_monitoreo_2026_10", report["month"][[TARGET_COLUMN] + FEATURE_COLUMNS],
                        "data/monitoring/2026/datos_abiertos_2026_10.csv"),
        })
        mlflow.set_tag("mlflow.note.content", (
            f"Champion vigente: {model_uri} ({summary}). En octubre de 2026 "
            f"baja el MAE {improvement:.1%} frente a la v7 ({reference_row['mae']:.3f} a {champion_row['mae']:.3f}). "
            "Es el modelo que sirve la API y la imagen de Docker."
        ))
        mlflow.log_text(CHAMPION_ARTIFACT_GUIDE, "README.md")
        mlflow.log_params({f"model_{k}": v for k, v in forest.get_params().items()
                           if k in ("n_estimators", "max_depth", "max_features", "max_samples", "min_samples_leaf")})
        mlflow.log_params({"registered_model_version": version, "n_features": len(FEATURE_COLUMNS),
                           "holiday_feature": champion_model.steps[0][0] == "holidays"})
        mlflow.log_dict({
            "model_family": type(forest).__name__,
            "model_parameters": {k: str(v) for k, v in forest.get_params().items()},
            "pipeline_steps": [name for name, _ in champion_model.steps],
            "features": FEATURE_COLUMNS,
            "derived_features": ["is_holiday"] if champion_model.steps[0][0] == "holidays" else [],
            "holiday_rule": {"dates_month_day": HOLIDAY_DATES, "mondays_month_n": HOLIDAY_MONDAYS},
            "target": TARGET_COLUMN,
            "forecast_horizon_days": HORIZON_DAYS,
            "train_end": train_end,
            "selection_period": "2026-09",
            "test_period": "2026-10",
            "random_seed": SEED,
        }, "documentation/model_configuration.json")
        mlflow.log_dict({"features": FEATURE_COLUMNS}, "features.json")

        metrics = {f"oct_{k}": float(v) for k, v in champion_row.items()}
        metrics.update({f"oct_v7_{k}": float(v) for k, v in reference_row.items()})
        metrics["oct_mae_improvement_vs_v7"] = float(improvement)
        if report["selection_table"] is not None:
            for name, row in report["selection_table"].iterrows():
                prefix = "v7" if name.startswith("v") else ("challenger_b" if "B" in name else "challenger_a")
                metrics.update({f"sep_{prefix}_{k}": float(v) for k, v in row.items()})
        mlflow.log_metrics({k: v for k, v in metrics.items() if np.isfinite(v)})
        for step, row in enumerate(report["daily"].itertuples(index=False)):
            mlflow.log_metric("october_daily_mae", float(row.champion), step=step)
            mlflow.log_metric("october_daily_mae_v7", float(row.v7), step=step)
        mlflow_catalog.log_artifact_catalog({
            "modeling/comparison": [output_dir / "selection_september.csv", output_dir / "selection_and_test.png"],
            "modeling/test": [output_dir / name for name in (
                "october_comparison.csv", "october_daily_mae.csv", "october_daily_error.png",
                "october_pred_vs_real.png")],
            "modeling/interpretation": [output_dir / "feature_importance.csv", output_dir / "feature_importance.png"],
        }, project_root)

    registry = mlflow.MlflowClient()
    registry.update_model_version(REGISTERED_MODEL_NAME, version, description=(
        f"Champion: {summary}. Octubre 2026: MAE {champion_row['mae']:.3f} "
        f"contra {reference_row['mae']:.3f} de la v7. Evidencia en la ejecución {run.info.run_id} "
        f"del experimento {CHALLENGER_EXPERIMENT}."))
    registry.set_model_version_tag(REGISTERED_MODEL_NAME, version, "documentation_run", run.info.run_id)
    return run.info.run_id


def main() -> None:
    """Documentar en MLflow la versión que tiene el alias champion."""
    import mlflow.sklearn

    from mibici_mlops import monitoring
    from mibici_mlops.config import DRIFT_REPORTS_DIR

    context = monitoring.load_context()
    month, _, _ = monitoring.prepare_month(context["model"], context["integrated"], context["development"],
                                           monitoring.MONITOR_CSV, monitoring.MONITOR_START,
                                           monitoring.MONITOR_END)
    version = mlflow.MlflowClient().get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
    champion_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/{version.version}")
    selection_path = DRIFT_REPORTS_DIR / "challenger_selection_september.csv"
    selection = pd.read_csv(selection_path, index_col=0) if selection_path.is_file() else None
    report = build_champion_report(champion_model, version.version, context["model_version"], month, selection)
    run_id = log_champion(report, version, champion_model)
    print(report["comparison"].round(3).to_string())
    print(f"Champion v{version.version} documentado en la ejecución {run_id}")


if __name__ == "__main__":
    main()

Writing ../build/migracion/mibici_mlops/modeling/champion_report.py


## 8.1 `mibici_mlops/mlflow_catalog.py`
Agregamos este archivo para que MLflow se lea igual en las tres fases. Todas usan la misma base `mlflow.db` y la carpeta `mlruns/` de la raíz.

- `document_experiments` pone a cada experimento una descripción y las etiquetas `phase`, `content` y `status` (vigente o desarrollo). También marca cada versión del Model Registry con la fase y el experimento donde se creó. Solo agrega información: no borra ni cambia ejecuciones, y la descripción del modelo registrado solo se escribe si estaba vacía.
- `log_data_lineage` repite en las ejecuciones de la Fase 3 el linaje de la Fase 2: etiquetas `dataset_role`, `dataset_path` y `dataset_sha256`, las carpetas `data/inventory`, `data/versions` y `data/lineage`, y los conjuntos usados en la columna Dataset.
- `log_artifact_catalog` sube los archivos a sus carpetas y deja `artifact_catalog.json` con el mismo formato de la Fase 2 (origen, ruta en MLflow, tamaño y SHA-256).
- `source_tags` hace que la columna Source muestre el archivo que generó la ejecución, como `main.py` en la Fase 2.
- `mark_previous_runs` no borra nada: antes de registrar una ejecución nueva, marca la anterior con el mismo nombre con `run_status = anterior`. La nueva queda con `vigente`, así el filtro `tags.run_status = "vigente"` muestra una por tipo.
- `trace_tags` repite en las ejecuciones hijas el commit, el entorno y el SHA-256 de los datos de la ejecución padre.
- `log_final_summary` crea la ejecución "Entrega final" en el experimento `MiBici_Entrega_Final`. Junta todo: copia completa de los artefactos de la ejecución de datos de la Fase 1, del modelo final v7, del entrenamiento de la v10, del monitoreo, de la comparación y del champion, cada una en su carpeta; todas sus métricas con el prefijo de su origen; la tabla de métricas, la lista de ejecuciones con enlace, SHA-256 y commit, las alertas por escenario y las revisiones de este notebook en `resumen/`; y un `README.md` con los enlaces.

Las ejecuciones de la Fase 3 usan nombres con el formato de la Fase 2, "Tipo | Modelo | detalle". Las etiquetas para filtrar son `phase`, `stage`, `model_role`, `model_version`, `decision`, `scenario`, `scenario_type`, `action` y `alert_A1` a `alert_A5`. Desde la terminal: `make mlflow-catalog`.

In [10]:
%%writefile ../build/migracion/mibici_mlops/mlflow_catalog.py
"""Orden de MLflow para todas las fases: descripción y etiquetas de cada experimento, linaje de datos
de las ejecuciones de la Fase 3, marca de ejecuciones vigentes y el resumen de la entrega final."""
import json
from pathlib import Path

import mlflow
import pandas as pd

from mibici_mlops.config import (
    CHALLENGER_EXPERIMENT,
    DOCKERHUB_REPO,
    MLFLOW_ARTIFACT_DIR,
    MODEL_ALIAS,
    PHASE2_MODEL_VERSION,
    RELEASES,
    CSV_VERSION_DIR,
    FILE_VERSION_INDEX,
    FILE_VERSION_TABLE,
    MLFLOW_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    MODELING_EXPERIMENT,
    MONITORING_EXPERIMENT,
    PROJECT_ROOT,
    PROVENANCE_DIR,
    REGISTERED_MODEL_NAME,
    TARGET_COLUMN,
)

SUMMARY_EXPERIMENT = "MiBici_Entrega_Final"
SUMMARY_RUN = "Entrega final | Random Forest | resumen de las tres fases"
TRACE_TAGS = ["git_commit", "git_branch", "git_dirty", "implementation_commit", "python_version",
              "sklearn_version", "data_sha256", "data_manifest", "dataset_role", "dataset_path",
              "dataset_sha256"]

# Descripción y etiquetas de cada experimento. Solo se agregan; las ejecuciones no se tocan.
EXPERIMENTS = {
    MLFLOW_EXPERIMENT: (
        "Fase 1. Carga, limpieza e integración de los viajes de MiBici con inventario de archivos, "
        "versiones y procedencia de los datos.",
        {"phase": "fase_1", "content": "datos", "status": "vigente"},
    ),
    MODELING_EXPERIMENT: (
        "Fase 2. Líneas base, validación walk-forward, búsqueda de hiperparámetros y modelo final "
        "registrado como mibici_salidas_diarias v7.",
        {"phase": "fase_2", "content": "modelado", "status": "vigente"},
    ),
    "MiBici_Monitoreo_Drift": (
        "Fase 3, desarrollo en el notebook 07. Corridas de prueba del monitoreo de octubre. "
        f"El resultado final está en {MONITORING_EXPERIMENT}.",
        {"phase": "fase_3", "content": "monitoreo", "status": "desarrollo"},
    ),
    "MiBici_Champion_Challenger": (
        "Fase 3, desarrollo en el notebook 07. Entrenamiento de los challengers y registro de las "
        f"versiones nuevas. La comparación final está en {CHALLENGER_EXPERIMENT}.",
        {"phase": "fase_3", "content": "champion_challenger", "status": "desarrollo"},
    ),
    MONITORING_EXPERIMENT: (
        "Fase 3. Monitoreo de octubre de 2026 con la v7: data drift por variable, tres escenarios "
        "(rezagos faltantes, medias desplazadas y cambio estacional), alertas A1 a A5 y acción "
        "propuesta. Cada escenario es una ejecución hija.",
        {"phase": "fase_3", "content": "monitoreo", "status": "vigente"},
    ),
    CHALLENGER_EXPERIMENT: (
        "Fase 3. Champion contra challenger (selección en septiembre, prueba en octubre) y la "
        "documentación del champion que sirve la API y la imagen de Docker.",
        {"phase": "fase_3", "content": "champion_challenger", "status": "vigente"},
    ),
    SUMMARY_EXPERIMENT: (
        "Entrega final. Una ejecución que junta las tres fases: métricas, gráficas, tablas, decisión del "
        "champion, reproducibilidad, servicio, Docker y pruebas, con el enlace a cada ejecución de origen.",
        {"phase": "fases_1_2_3", "content": "resumen", "status": "vigente"},
    ),
}

REGISTERED_MODEL_DESCRIPTION = (
    "Salidas diarias por estación de MiBici, un día adelante. La v7 es el modelo final de la Fase 2. "
    "El alias champion marca la versión que sirve la API."
)


def versions_by_experiment(client: mlflow.MlflowClient) -> dict:
    """Versiones del Model Registry agrupadas por el experimento de la ejecución que las creó."""
    found = {}
    for version in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'"):
        try:
            experiment_id = client.get_run(version.run_id).info.experiment_id
        except Exception:
            continue
        found.setdefault(experiment_id, []).append(int(version.version))
    return {key: sorted(value) for key, value in found.items()}


def document_experiments(tracking_uri: str = MLFLOW_TRACKING_URI) -> pd.DataFrame:
    """Pone descripción y etiquetas a los experimentos que existan y a las versiones del registro."""
    mlflow.set_tracking_uri(tracking_uri)
    client = mlflow.MlflowClient()
    versions = versions_by_experiment(client)
    rows = []
    for name, (description, tags) in EXPERIMENTS.items():
        experiment = client.get_experiment_by_name(name)
        if experiment is None:
            continue
        registered = versions.get(experiment.experiment_id, [])
        if registered:
            description += " Versiones del registro creadas aquí: " + ", ".join(f"v{v}" for v in registered) + "."
        client.set_experiment_tag(experiment.experiment_id, "mlflow.note.content", description)
        for key, value in tags.items():
            client.set_experiment_tag(experiment.experiment_id, key, value)
        for version in registered:
            client.set_model_version_tag(REGISTERED_MODEL_NAME, str(version), "phase", tags["phase"])
            client.set_model_version_tag(REGISTERED_MODEL_NAME, str(version), "source_experiment", name)
        rows.append({"experimento": name, **tags, "versiones": registered,
                     "ejecuciones": len(client.search_runs([experiment.experiment_id], max_results=5000))})

    model = client.get_registered_model(REGISTERED_MODEL_NAME)
    if not model.description:
        client.update_registered_model(REGISTERED_MODEL_NAME, REGISTERED_MODEL_DESCRIPTION)
    return pd.DataFrame(rows)


def mark_previous_runs(experiment_name: str, run_name: str) -> int:
    """Marca como anteriores las ejecuciones que ya existían con el mismo nombre, con sus hijas.

    No borra nada: solo cambia la etiqueta `run_status` a "anterior". La ejecución nueva se registra
    con "vigente", así el filtro `tags.run_status = "vigente"` deja una sola por tipo.
    """
    client = mlflow.MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        return 0
    escaped = run_name.replace("'", "\\'")
    previous = client.search_runs([experiment.experiment_id], f"attributes.run_name = '{escaped}'")
    for run in previous:
        children = client.search_runs([experiment.experiment_id],
                                      f"tags.mlflow.parentRunId = '{run.info.run_id}'")
        for item in [run, *children]:
            client.set_tag(item.info.run_id, "run_status", "anterior")
    return len(previous)


def log_data_lineage(project_root: Path, data_path: Path, data_sha256: str, role: str,
                     datasets: dict) -> None:
    """Mismo linaje que las ejecuciones de la Fase 2: etiquetas del dataset, inventario, versiones,
    procedencia y los conjuntos usados en la pestaña Datasets.

    `datasets` va de contexto a (nombre, DataFrame, fuente), por ejemplo
    {"monitoring": ("mibici_monitoreo_2026_10", month, csv_path)}.
    """
    project_root = Path(project_root)
    mlflow.set_tags({
        "dataset_role": role,
        "dataset_path": data_path.relative_to(project_root).as_posix(),
        "dataset_sha256": data_sha256,
    })
    files = {
        "data/inventory": [CSV_VERSION_DIR / "csv_versions.csv", CSV_VERSION_DIR / "csv_versions.json"],
        "data/versions": [FILE_VERSION_TABLE, FILE_VERSION_INDEX],
        "data/lineage": [PROVENANCE_DIR / "data_provenance.json"],
    }
    phase1_run = None
    for folder, paths in files.items():
        for path in paths:
            local = project_root / path.relative_to(PROJECT_ROOT)
            if local.is_file():
                mlflow.log_artifact(str(local), folder)
                if local.name == "data_provenance.json":
                    phase1_run = json.loads(local.read_text(encoding="utf-8")).get("phase1_run")
    mlflow.log_dict({
        "integrated_path": data_path.relative_to(project_root).as_posix(),
        "integrated_sha256": data_sha256,
        "phase1_run": phase1_run,
        "datasets": {context: {"name": name, "rows": len(frame), "source": str(source)}
                     for context, (name, frame, source) in datasets.items()},
    }, "data/lineage/dataset_reference.json")
    for context, (name, frame, source) in datasets.items():
        dataset = mlflow.data.from_pandas(
            frame.reset_index(drop=True), source=str(source), name=name,
            targets=TARGET_COLUMN if TARGET_COLUMN in frame.columns else None,
        )
        mlflow.log_input(dataset, context=context)


def log_artifact_catalog(files: dict, project_root: Path) -> list:
    """Sube cada archivo a su carpeta y deja `artifact_catalog.json` con el mismo formato de la Fase 2."""
    from mibici_mlops.tracking import calculate_sha256

    project_root = Path(project_root)
    catalog = []
    for folder, paths in files.items():
        for path in paths:
            path = Path(path)
            if not path.is_file():
                continue
            mlflow.log_artifact(str(path), folder)
            try:
                source = path.resolve().relative_to(project_root.resolve()).as_posix()
            except ValueError:
                source = str(path)
            catalog.append({"file": path.name, "source_path": source, "artifact_path": f"{folder}/{path.name}",
                            "size_bytes": path.stat().st_size, "sha256": calculate_sha256(path)})
    mlflow.log_dict({"files": catalog}, "artifact_catalog.json")
    return catalog


def source_tags(module_path: str) -> dict:
    """La columna Source muestra el archivo que generó la ejecución, como main.py en la Fase 2."""
    return {"mlflow.source.name": module_path, "mlflow.source.type": "LOCAL", "run_status": "vigente"}


def trace_tags(run_id: str) -> dict:
    """Etiquetas de trazabilidad (commit, entorno y SHA-256 de los datos) de una ejecución, para repetirlas."""
    tags = mlflow.MlflowClient().get_run(run_id).data.tags
    return {key: tags[key] for key in TRACE_TAGS if key in tags}


def current_run(client, experiment_name: str, run_name: str | None = None, stage: str | None = None):
    """La ejecución vigente más reciente de un experimento, por nombre o por la etiqueta stage."""
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        return None
    filters = []
    if run_name:
        filters.append(f"attributes.run_name = '{run_name}'")
    if stage:
        filters.append(f"tags.stage = '{stage}'")
    runs = client.search_runs([experiment.experiment_id], " and ".join(filters),
                              order_by=["attributes.start_time DESC"], max_results=20)
    current = [run for run in runs if run.data.tags.get("run_status", "vigente") == "vigente"]
    return (current or runs or [None])[0]


def run_link(run) -> str:
    return f"#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"


def copy_run_artifacts(client, run, destination: str, workdir: Path, max_bytes: int = 50_000_000) -> list:
    """Copia todos los artefactos de una ejecución a una carpeta del resumen. Devuelve los omitidos por tamaño."""
    skipped = []
    pending = [""]
    while pending:
        folder = pending.pop()
        for item in client.list_artifacts(run.info.run_id, folder or None):
            if item.is_dir:
                pending.append(item.path)
            elif item.file_size and item.file_size > max_bytes:
                skipped.append(item.path)
            else:
                local = Path(client.download_artifacts(run.info.run_id, item.path, str(workdir)))
                parent = Path(item.path).parent.as_posix()
                mlflow.log_artifact(str(local), destination if parent == "." else f"{destination}/{parent}")
    return skipped


def log_final_summary(project_root: Path = PROJECT_ROOT, verification: pd.DataFrame | None = None,
                      extra_files: dict | None = None) -> str:
    """Ejecución resumen de la entrega final en su propio experimento.

    Junta las métricas, parámetros y artefactos de las ejecuciones de origen: datos (Fase 1), modelo
    final v7 (Fase 2), monitoreo con sus escenarios, champion contra challenger y el champion.
    `verification` es la tabla de revisiones del notebook 08 y `extra_files` va de carpeta a archivos.
    """
    import tempfile

    from mibici_mlops import tracking
    from mibici_mlops.config import INTEGRATED_PATH

    project_root = Path(project_root)
    tracking.setup_mlflow(MLFLOW_TRACKING_URI, SUMMARY_EXPERIMENT, project_root / MLFLOW_ARTIFACT_DIR.name)
    mlflow.set_experiment(SUMMARY_EXPERIMENT)
    client = mlflow.MlflowClient()

    champion = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
    phase2 = client.get_model_version(REGISTERED_MODEL_NAME, str(PHASE2_MODEL_VERSION))
    sources = {
        "fase_1_datos": current_run(client, MLFLOW_EXPERIMENT),
        "fase_2_modelo_final": client.get_run(phase2.run_id),
        "fase_3_champion_entrenamiento": client.get_run(champion.run_id),
        "fase_3_monitoreo": current_run(client, MONITORING_EXPERIMENT, stage="monitoring"),
        "fase_3_champion_challenger": current_run(client, CHALLENGER_EXPERIMENT, stage="champion_challenger"),
        "fase_3_champion": current_run(client, CHALLENGER_EXPERIMENT, stage="champion"),
    }
    sources = {key: run for key, run in sources.items() if run is not None}
    scenarios = []
    if "fase_3_monitoreo" in sources:
        parent = sources["fase_3_monitoreo"]
        scenarios = client.search_runs([parent.info.experiment_id],
                                       f"tags.mlflow.parentRunId = '{parent.info.run_id}'")

    data_path = project_root / INTEGRATED_PATH.relative_to(PROJECT_ROOT)
    data_sha256 = tracking.calculate_sha256(data_path)
    mark_previous_runs(SUMMARY_EXPERIMENT, SUMMARY_RUN)
    with mlflow.start_run(run_name=SUMMARY_RUN, tags=source_tags("mibici_mlops/mlflow_catalog.py")) as run:
        tracking.log_run_context(project_root, data_path, data_sha256)
        log_data_lineage(project_root, data_path, data_sha256, "summary", {})
        # Los mismos conjuntos de datos de las ejecuciones de origen, para verlos en la columna Dataset
        seen, inputs = set(), []
        for source in sources.values():
            for item in (source.inputs.dataset_inputs if source.inputs else []):
                key = (item.dataset.name, item.dataset.digest)
                if key not in seen:
                    seen.add(key)
                    inputs.append(item)
        if inputs:
            client.log_inputs(run.info.run_id, datasets=inputs)
        mlflow.set_tags({
            "stage": "final_summary", "phase": "fases_1_2_3", "model_role": "champion",
            "model_version": champion.version, "phase2_model_version": phase2.version,
            "registered_model_name": REGISTERED_MODEL_NAME, "docker_image": DOCKERHUB_REPO,
            "docker_tag_champion": RELEASES.get(str(champion.version), "sin publicar"),
            **{f"run_{key}": source.info.run_id for key, source in sources.items()},
        })
        decision = sources.get("fase_3_champion_challenger")
        decision_text = decision.data.tags.get("decision", "") if decision else ""
        if decision_text:
            mlflow.set_tag("decision", decision_text)
        mlflow.set_tag("mlflow.note.content", (
            f"Resumen de la entrega final. Champion: {REGISTERED_MODEL_NAME} v{champion.version}; modelo de la "
            f"Fase 2: v{phase2.version}. Decisión en octubre de 2026: {decision_text or 'sin registrar'}. "
            "Cada carpeta de artefactos copia una ejecución de origen; README.md tiene los enlaces."
        ))
        mlflow.log_params({
            "champion_version": champion.version, "phase2_version": phase2.version,
            "docker_repository": DOCKERHUB_REPO,
            **{f"docker_tag_v{version}": tag for version, tag in RELEASES.items()},
            "api_endpoints": "POST /predict, GET /predict/estacion, GET /ui, GET /health, /docs, /redoc",
        })

        rows = []
        for key, source in sources.items():
            values = {f"{key}.{name}": value for name, value in source.data.metrics.items()}
            mlflow.log_metrics({name.replace(".", "__"): float(v) for name, v in values.items()
                                if pd.notna(v) and abs(v) != float("inf")})
            rows += [{"ejecución": key, "métrica": name, "valor": value}
                     for name, value in source.data.metrics.items()]
        for child in scenarios:
            name = child.data.tags.get("scenario", child.info.run_name)
            rows += [{"ejecución": f"escenario {name}", "métrica": metric, "valor": value}
                     for metric, value in child.data.metrics.items()]
        metrics_table = pd.DataFrame(rows)
        mlflow.log_table(metrics_table, "resumen/metricas_por_ejecucion.json")

        runs_table = pd.DataFrame([{
            "ejecución": key, "nombre": source.info.run_name,
            "experimento": client.get_experiment(source.info.experiment_id).name,
            "run_id": source.info.run_id, "enlace": run_link(source),
            "data_sha256": source.data.tags.get("data_sha256", ""),
            "git_commit": source.data.tags.get("git_commit", ""),
        } for key, source in sources.items()] + [{
            "ejecución": "escenario", "nombre": child.info.run_name,
            "experimento": MONITORING_EXPERIMENT, "run_id": child.info.run_id, "enlace": run_link(child),
            "data_sha256": child.data.tags.get("data_sha256", ""), "git_commit": child.data.tags.get("git_commit", ""),
        } for child in scenarios])
        mlflow.log_table(runs_table, "resumen/ejecuciones.json")

        alerts = pd.DataFrame([{
            "escenario": child.data.tags.get("scenario", child.info.run_name),
            "tipo": child.data.tags.get("scenario_type", ""), "acción": child.data.tags.get("action", ""),
            **{key: value for key, value in child.data.tags.items() if key.startswith("alert_")},
        } for child in scenarios])
        if not alerts.empty:
            mlflow.log_table(alerts, "resumen/alertas_por_escenario.json")
        if verification is not None:
            mlflow.log_table(verification, "resumen/verificacion_notebook_08.json")

        skipped = {}
        with tempfile.TemporaryDirectory() as workdir:
            for key, source in sources.items():
                skipped[key] = copy_run_artifacts(client, source, key, Path(workdir))
        log_artifact_catalog(extra_files or {}, project_root)

        lines = [
            "# Entrega final MiBici", "",
            f"Champion: `models:/{REGISTERED_MODEL_NAME}/{champion.version}` (alias `{MODEL_ALIAS}`). "
            f"Modelo de la Fase 2: v{phase2.version}. Decisión: {decision_text or 'sin registrar'}.",
            f"Imagen: `{DOCKERHUB_REPO}`, etiquetas " + ", ".join(f"v{v}: {t}" for v, t in RELEASES.items()) + ".", "",
            "## Ejecuciones de origen", "",
            "| Parte | Ejecución | Experimento | Enlace | SHA-256 de datos | Commit |", "| --- | --- | --- | --- | --- | --- |",
        ]
        lines += [f"| {r.ejecución} | {r.nombre.replace('|', '/')} | {r.experimento} | [abrir]({r.enlace}) | `{r.data_sha256[:12]}` | `{r.git_commit[:10]}` |"
                  for r in runs_table.itertuples()]
        lines += ["", "## Carpetas", "",
                  "Cada carpeta tiene una copia completa de los artefactos de su ejecución de origen, con su misma estructura.", ""]
        lines += [f"- `{key}/`: {source.info.run_name}" for key, source in sources.items()]
        lines += ["- `resumen/`: métricas de todas las ejecuciones, lista de ejecuciones con enlaces, alertas por escenario y revisiones del notebook 08.",
                  "", "Las métricas llevan el prefijo de su ejecución, por ejemplo `fase_3_champion__oct_mae`."]
        if any(skipped.values()):
            lines += ["", "Archivos no copiados por tamaño (siguen en su ejecución de origen):"]
            lines += [f"- {key}: {path}" for key, paths in skipped.items() for path in paths]
        mlflow.log_text("\n".join(lines), "README.md")
    return run.info.run_id


def main() -> None:
    """Resumen de la entrega final y descripción de todos los experimentos, desde la terminal."""
    print("Resumen:", log_final_summary())
    print(document_experiments().to_string(index=False))


if __name__ == "__main__":
    main()

Writing ../build/migracion/mibici_mlops/mlflow_catalog.py


## 9. `mibici_mlops/modeling/deploy.py`
Separamos en funciones las operaciones de Docker que estaban en el notebook 07: comprobar que Docker está abierto, la etiqueta de cada champion, construir la imagen, levantarla, comparar la muestra de referencia predicha dentro del contenedor (4.7) y publicar en DockerHub sin sobrescribir una etiqueta que ya existe (4.9).

Un champion nuevo recibe la versión menor siguiente a la más alta de `RELEASES` (hoy sería 1.3.0). `main` publica solo si la comparación sale bien y `AUTO_DEPLOY` es `True`. Antes hay que iniciar sesión una vez con `docker login`. Desde la terminal: `python -m mibici_mlops.modeling.deploy` o `make deploy`.

In [11]:
%%writefile ../build/migracion/mibici_mlops/modeling/deploy.py
"""Construir la imagen del servicio, comprobarla con la muestra de referencia y publicarla en DockerHub."""

import json
from pathlib import Path
import subprocess
import time

import numpy as np
import pandas as pd
import requests

from mibici_mlops.config import (
    AUTO_DEPLOY,
    DOCKERHUB_REPO,
    FEATURE_COLUMNS,
    IMAGE_NAME,
    PROJECT_ROOT,
    RELEASES,
    REPRODUCIBILITY_DIR,
    SERVING_DIR,
    TARGET_COLUMN,
)
from mibici_mlops.modeling.evaluate import regression_metrics

CONTAINER = "mibici-api"
PORT = 8000
TOLERANCE = 1e-4


def run(command: list[str], check: bool = True) -> str:
    """Ejecutar un comando y devolver su salida."""
    print("$", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0 and check:
        print(output[-3000:])
        raise RuntimeError(f"Falló: {' '.join(command)}")
    return output


def docker_available() -> bool:
    """Docker Desktop (o el servicio de Docker en WSL) tiene que estar abierto."""
    try:
        output = run(["docker", "version", "--format", "{{.Server.Version}}"], check=False)
    except FileNotFoundError:
        return False
    return not any(text in output for text in ("failed to connect", "Cannot connect", "error during connect"))


def image_tag(model_version: str, releases: dict = RELEASES) -> str:
    """Etiqueta fija de cada champion; uno nuevo recibe la versión menor siguiente a la más alta."""
    if str(model_version) in releases:
        return releases[str(model_version)]
    return f"1.{max(int(tag.split('.')[1]) for tag in releases.values()) + 1}.0"


def to_instance(row: dict) -> dict:
    """Una fila del dataset en el formato JSON que recibe /predict."""
    instance = {}
    for column in FEATURE_COLUMNS:
        value = row[column]
        if pd.isna(value):
            instance[column] = None
        elif column.startswith(("lag_", "roll_")):
            instance[column] = float(value)
        else:
            instance[column] = int(value)
    return instance


def build_image(image: str = IMAGE_NAME, context: Path = PROJECT_ROOT) -> None:
    """Equivale a docker build -t ml-service:latest . desde la raíz del repositorio."""
    start = time.perf_counter()
    run(["docker", "build", "-t", image, str(context)])
    print(f"Imagen {image} construida en {time.perf_counter() - start:.0f} s")


def start_container(image: str = IMAGE_NAME, container: str = CONTAINER, port: int = PORT,
                    timeout: int = 60) -> dict:
    """Levantar el contenedor y esperar a que /health diga que el modelo está cargado."""
    run(["docker", "rm", "-f", container], check=False)
    run(["docker", "run", "-d", "--name", container, "-p", f"{port}:8000", image])
    for _ in range(timeout):
        try:
            health = requests.get(f"http://localhost:{port}/health", timeout=2).json()
            if health.get("model_loaded"):
                return health
        except requests.RequestException:
            pass
        time.sleep(1)
    print(run(["docker", "logs", container], check=False)[-2000:])
    raise RuntimeError("El contenedor no respondió")


def check_container(reference: pd.DataFrame, port: int = PORT) -> tuple[dict, pd.DataFrame]:
    """Predecir la muestra de referencia dentro del contenedor y compararla con la del modelo."""
    payload = {"instances": [to_instance(r) for r in reference.to_dict(orient="records")]}
    start = time.perf_counter()
    response = requests.post(f"http://localhost:{port}/predict", json=payload, timeout=60)
    latency = time.perf_counter() - start
    body = response.json()
    served = np.array([p["predicted_trips"] for p in body["predictions"]])
    expected = reference["reference_prediction"].clip(lower=0).to_numpy()
    difference = np.abs(served - expected)
    local_metrics = regression_metrics(reference[TARGET_COLUMN], expected)
    docker_metrics = regression_metrics(reference[TARGET_COLUMN], served)
    summary = {
        "model_version": body["model_version"], "model_uri": body["model_uri"],
        "rows": len(reference), "max_abs_difference": float(difference.max()),
        "mean_abs_difference": float(difference.mean()), "tolerance": TOLERANCE,
        "reproducible": bool(difference.max() < TOLERANCE),
        "local_mae": local_metrics["mae"], "docker_mae": docker_metrics["mae"],
        "local_rmse": local_metrics["rmse"], "docker_rmse": docker_metrics["rmse"],
        "latency_seconds_500_rows": round(latency, 3),
    }
    comparison = reference[["Fecha", "Origen_Id", TARGET_COLUMN]].assign(
        local_prediction=expected, docker_prediction=served, abs_difference=difference)
    return summary, comparison


def is_published(tag: str, repo: str = DOCKERHUB_REPO) -> bool:
    result = subprocess.run(["docker", "manifest", "inspect", f"{repo}:{tag}"], capture_output=True)
    return result.returncode == 0


def publish(tag: str, image: str = IMAGE_NAME, repo: str = DOCKERHUB_REPO) -> bool:
    """Subir la etiqueta fija y latest. Una etiqueta que ya existe no se sobrescribe."""
    if is_published(tag, repo):
        print(f"{repo}:{tag} ya está publicada; no se sobrescribe.")
        return False
    for name in [tag, "latest"]:
        run(["docker", "tag", image, f"{repo}:{name}"])
        output = run(["docker", "push", f"{repo}:{name}"], check=False)
        if "denied" in output or "unauthorized" in output.lower() or "requested access" in output:
            print("No se pudo publicar: falta iniciar sesión con docker login.")
            return False
    print("Publicada: https://hub.docker.com/r/" + repo)
    return True


def main(auto_deploy: bool = AUTO_DEPLOY) -> None:
    """Construir, comprobar y, si auto_deploy es True, publicar la imagen del champion exportado."""
    if not docker_available():
        raise RuntimeError("Docker no está abierto. Abre Docker Desktop y vuelve a intentarlo.")
    model_version = json.loads((SERVING_DIR / "metadata.json").read_text())["model_version"]
    tag = image_tag(model_version)
    reference = pd.read_csv(SERVING_DIR / "reference_sample.csv", parse_dates=["Fecha"])

    build_image()
    try:
        start_container()
        summary, comparison = check_container(reference)
    finally:
        run(["docker", "rm", "-f", CONTAINER], check=False)
    REPRODUCIBILITY_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(REPRODUCIBILITY_DIR / "docker_comparison.csv", index=False)
    (REPRODUCIBILITY_DIR / "docker_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))

    if not summary["reproducible"]:
        print("La imagen no reproduce la muestra de referencia; no se publica.")
    elif auto_deploy:
        publish(tag)
    else:
        state = "ya publicada" if is_published(tag) else "sin publicar"
        print(f"AUTO_DEPLOY = False: no se publica. {DOCKERHUB_REPO}:{tag} está {state}.")


if __name__ == "__main__":
    main()

Writing ../build/migracion/mibici_mlops/modeling/deploy.py


## 10. `Dockerfile`, `.dockerignore` y `requirements-serving.txt`
Tomamos los archivos de las secciones 4.2 y 4.3 y los dejamos en la raíz para que la imagen se construya con `docker build -t ml-service:latest .`. Las diferencias con el 07 son las rutas: el código del servicio sale de `mibici_mlops/serving/app.py` y las librerías de `requirements-serving.txt`.

`.dockerignore` deja entrar solo los tres elementos que copia el `Dockerfile`.

In [12]:
%%writefile ../build/migracion/Dockerfile
# Servicio de predicción MiBici: FastAPI y el champion de mibici_salidas_diarias
FROM python:3.13.17-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    MODEL_URI=/app/models/serving/model

WORKDIR /app

# Las librerías van antes que el modelo para reutilizar esta capa si solo cambia el modelo
COPY requirements-serving.txt requirements.txt
RUN pip install --upgrade pip && pip install -r requirements.txt

COPY mibici_mlops/serving/app.py app.py
COPY models/serving/ models/serving/

RUN useradd --create-home --uid 1000 appuser && chown -R appuser /app
USER appuser

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s --start-period=20s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing ../build/migracion/Dockerfile


In [13]:
%%writefile ../build/migracion/.dockerignore
# Solo entra a la imagen lo que usa el Dockerfile; datos, MLflow y notebooks se quedan fuera
*
!requirements-serving.txt
!mibici_mlops/serving/app.py
!models/serving/
**/__pycache__

Writing ../build/migracion/.dockerignore


In [14]:
%%writefile ../build/migracion/requirements-serving.txt
numpy==2.5.4
pandas==3.0.6
scikit-learn==1.9.1
scipy==1.18.1
joblib==1.6.0
cloudpickle==3.1.2
mlflow-skinny==3.17.2
fastapi==0.158.1
uvicorn==0.54.0
pydantic==2.14.1

Writing ../build/migracion/requirements-serving.txt


## 11. `requirements.txt`
El archivo original solo instala el paquete (`-e .`). Para que el entorno se pueda reproducir fijamos las versiones de las librerías que usa la entrega. En lugar de escribirlas a mano, las tomamos del entorno donde corrimos el 07, y comprobamos que las del modelo coinciden con las que MLflow guardó junto al champion.

In [15]:
PINNED = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "joblib", "cloudpickle", "mlflow",
          "evidently", "fastapi", "uvicorn", "pydantic", "httpx", "requests", "matplotlib", "pytest"]
lines = ["# Entorno del proyecto con las versiones con las que se entrenó y validó el modelo.",
         "# Generado en el notebook 08 a partir del entorno donde se corrió el 07."]
lines += [f"{package}=={metadata.version(package)}" for package in PINNED]
lines += ["", "# Paquete del proyecto (dependencias en pyproject.toml)", "-e ."]
(STAGE / "requirements.txt").write_text("\n".join(lines) + "\n")
print((STAGE / "requirements.txt").read_text())

# Las librerías del modelo deben ser las que MLflow guardó con el champion
saved = dict(line.split("==") for line in
             (ROOT / "models" / "serving" / "model" / "requirements.txt").read_text().split() if "==" in line)
check_versions = pd.DataFrame([{"paquete": p, "modelo (MLflow)": saved.get(p), "entorno": metadata.version(p)}
                               for p in ["scikit-learn", "pandas", "numpy", "scipy", "cloudpickle"]])
check_versions["coincide"] = check_versions["modelo (MLflow)"] == check_versions["entorno"]
assert check_versions["coincide"].all(), "El entorno no tiene las versiones del modelo"
check_versions

# Entorno del proyecto con las versiones con las que se entrenó y validó el modelo.
# Generado en el notebook 08 a partir del entorno donde se corrió el 07.
numpy==2.5.4
pandas==3.0.6
pyarrow==25.0.1
scipy==1.18.1
scikit-learn==1.9.1
joblib==1.6.0
cloudpickle==3.1.2
mlflow==3.17.2
evidently==0.7.23
fastapi==0.158.1
uvicorn==0.54.0
pydantic==2.14.1
httpx==0.28.1
requests==2.34.2
matplotlib==3.11.2
pytest==9.1.1

# Paquete del proyecto (dependencias en pyproject.toml)
-e .



,paquete,modelo (MLflow),entorno,coincide
0,scikit-learn,1.9.1,1.9.1,True
1,pandas,3.0.6,3.0.6,True
2,numpy,2.5.4,2.5.4,True
3,scipy,1.18.1,1.18.1,True
4,cloudpickle,3.1.2,3.1.2,True


## 12. `pyproject.toml`
Incluimos las bibliotecas necesarias para esta entrega (FastAPI, Uvicorn, Pydantic, Evidently, SciPy, joblib y cloudpickle) y `httpx` a las de desarrollo, porque la usa `TestClient`. Al final va la configuración de pytest: la carpeta de pruebas y la marca `integration`, que antes registraba el `conftest.py` de `build/tests_entrega`.

In [16]:
%%writefile ../build/migracion/pyproject.toml
[build-system]
requires = ["flit_core>=3.2,<4"]
build-backend = "flit_core.buildapi"

[project]
name = "mibici_mlops"
version = "0.0.1"
description = "Proyecto MLOps para predecir las salidas diarias por estación de MiBici con un día de anticipación."
authors = [
    { name = "Equipo 8" },
]
readme = "README.md"
requires-python = ">=3.13,<3.14"
classifiers = [
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3.13",
]
dependencies = [
    "cloudpickle",
    "evidently",
    "fastapi",
    "folium",
    "joblib",
    "loguru",
    "matplotlib",
    "mlflow",
    "numpy",
    "osmium",
    "pandas",
    "pyarrow",
    "pydantic",
    "pyproj",
    "python-dotenv",
    "requests",
    "scikit-learn",
    "scipy",
    "shapely",
    "tqdm",
    "typer",
    "uvicorn",
]

[project.optional-dependencies]
dev = [
    "httpx",
    "ipython",
    "ipywidgets",
    "jupyterlab",
    "mkdocs",
    "notebook",
    "pytest",
    "ruff",
]

[tool.ruff]
line-length = 99
target-version = "py313"
src = ["mibici_mlops"]
include = ["pyproject.toml", "mibici_mlops/**/*.py"]

[tool.ruff.lint]
extend-select = ["I"]

[tool.ruff.lint.isort]
known-first-party = ["mibici_mlops"]
force-sort-within-sections = true

[tool.pytest.ini_options]
testpaths = ["tests"]
markers = ["integration: prueba de extremo a extremo"]

Writing ../build/migracion/pyproject.toml


## 13. `Makefile`
Agregamos comandos para ejecutar las tareas principales: `make test`, `make export-model`, `make serve`, `make docker-build`, `make docker-run`, `make deploy`, `make monitor`, `make challenger`, `make champion` y `make mlflow-catalog`. `make test` ahora corre `pytest -q`, igual que pide la entrega.

In [17]:
%%writefile ../build/migracion/Makefile
#################################################################################
# GLOBALS                                                                       #
#################################################################################

PROJECT_NAME = equipo-8-mibici-mlops
PYTHON_VERSION = 3.13
PYTHON_INTERPRETER = python

#################################################################################
# COMMANDS                                                                      #
#################################################################################


## Install Python dependencies
.PHONY: requirements
requirements:
	uv sync
	



## Delete all compiled Python files
.PHONY: clean
clean:
	find . -type f -name "*.py[co]" -delete
	find . -type d -name "__pycache__" -delete


## Lint using ruff (use `make format` to do formatting)
.PHONY: lint
lint:
	ruff format --check
	ruff check

## Format source code with ruff
.PHONY: format
format:
	ruff check --fix
	ruff format



## Run unit and integration tests
.PHONY: test
test:
	$(PYTHON_INTERPRETER) -m pytest -q


## Set up Python interpreter environment
.PHONY: create_environment
create_environment:
	uv venv --python $(PYTHON_VERSION)
	@echo ">>> New uv virtual environment created. Activate with:"
	@echo ">>> Windows: .\\\\.venv\\\\Scripts\\\\activate"
	@echo ">>> Unix/macOS: source ./.venv/bin/activate"
	



#################################################################################
# PROJECT RULES                                                                 #
#################################################################################


## Make dataset
.PHONY: data
data: requirements
	$(PYTHON_INTERPRETER) mibici_mlops/dataset.py


## Export the champion model from MLflow to models/serving
.PHONY: export-model
export-model:
	$(PYTHON_INTERPRETER) -m mibici_mlops.modeling.export_model


## Serve the API locally on http://localhost:8000/docs
.PHONY: serve
serve:
	MODEL_URI=models/serving/model uvicorn mibici_mlops.serving.app:app --host 0.0.0.0 --port 8000


## Build the Docker image
.PHONY: docker-build
docker-build:
	docker build -t ml-service:latest .


## Run the Docker image on port 8000
.PHONY: docker-run
docker-run:
	docker run -p 8000:8000 ml-service:latest


## Build, check against the reference sample and publish if AUTO_DEPLOY is True
.PHONY: deploy
deploy:
	$(PYTHON_INTERPRETER) -m mibici_mlops.modeling.deploy


## Monitor October 2026: drift, scenarios, alerts and action
.PHONY: monitor
monitor:
	$(PYTHON_INTERPRETER) -m mibici_mlops.monitoring


## Champion vs challenger with the promotion rule
.PHONY: challenger
challenger:
	$(PYTHON_INTERPRETER) -m mibici_mlops.modeling.challenger


## Log the champion evidence (metrics, figures, configuration) to MLflow
.PHONY: champion
champion:
	$(PYTHON_INTERPRETER) -m mibici_mlops.modeling.champion_report


## Final summary run and descriptions and tags for every MLflow experiment
.PHONY: mlflow-catalog
mlflow-catalog:
	$(PYTHON_INTERPRETER) -m mibici_mlops.mlflow_catalog


#################################################################################
# Self Documenting Commands                                                     #
#################################################################################

.DEFAULT_GOAL := help

define PRINT_HELP_PYSCRIPT
import re, sys; \
lines = '\n'.join([line for line in sys.stdin]); \
matches = re.findall(r'\n## (.*)\n[\s\S]+?\n([a-zA-Z_-]+):', lines); \
print('Available rules:\n'); \
print('\n'.join(['{:25}{}'.format(*reversed(match)) for match in matches]))
endef
export PRINT_HELP_PYSCRIPT

help:
	@$(PYTHON_INTERPRETER) -c "${PRINT_HELP_PYSCRIPT}" < $(MAKEFILE_LIST)

Writing ../build/migracion/Makefile


## 14. Pruebas
En el 07 las pruebas nuevas vivían en `build/tests_entrega/` y se corrían aparte de las 17 que ya teníamos. Ahora las integramos en `tests/` para ejecutarlas juntas con `pytest -q`. Los principales ajustes son:

- `conftest.py` junta la fixture `toy_data` que ya existía con las del 07 (datos sintéticos, modelo pequeño y cliente de la API).
- `test_api.py` suma siete pruebas de la página: la predicción por estación y día coincide con el modelo, un día futuro usa la última historia conocida, sin historia da 404, la página lista estaciones, sin modelo responde 503, y la raíz lleva a `/ui` o a `/docs`.
- `helpers.py` tiene los datos sintéticos y `start_client`, que crea el cliente del servicio de `mibici_mlops/serving/app.py`. `to_instance` ahora se importa de `deploy.py` para no repetirla.
- Las pruebas importan `monitoring`, `add_holidays` y `SERVING_DIR` desde el paquete en lugar de los archivos sueltos del 07.

- `test_data.py` solo cambia en `test_csv_versioning_keeps_history`: las dos versiones del CSV tienen el mismo tamaño y, si se escriben en el mismo instante, el sistema de archivos puede dejarles la misma fecha de modificación. Como `versioning.py` usa tamaño y fecha para no recalcular el hash, la prueba podía fallar de manera intermitente. Ahora la prueba adelanta la fecha un segundo; `versioning.py` no cambia.

Las demás pruebas que ya teníamos (`test_features.py`, etc.) no cambian.

In [18]:
%%writefile ../build/migracion/tests/conftest.py
"""Datos y modelos pequeños compartidos por las pruebas."""

import json

import numpy as np
import pandas as pd
import pytest

from helpers import make_station_days, start_client
from mibici_mlops.config import FEATURE_COLUMNS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN
from mibici_mlops.features import build_model_dataset


@pytest.fixture
def toy_data() -> pd.DataFrame:
    """Una estación, tres meses, con el 10 de enero faltante a propósito."""
    dates = pd.date_range("2024-12-01", "2025-02-28")
    data = pd.DataFrame({"Origen_Id": 1, "Fecha": dates, "target": np.arange(1, len(dates) + 1)})
    data = data.loc[data["Fecha"] != "2025-01-10"].reset_index(drop=True)
    data["day_of_week"] = data["Fecha"].dt.day_of_week
    return data


@pytest.fixture(scope="session")
def model_dataset():
    return build_model_dataset(make_station_days(), 1, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)


@pytest.fixture(scope="session")
def trained_pipeline(model_dataset):
    from mibici_mlops.modeling.train import train_final_model

    return train_final_model("Random Forest", {"n_estimators": 10, "max_depth": 5},
                             model_dataset, FEATURE_COLUMNS, TARGET_COLUMN)


@pytest.fixture(scope="session")
def mlflow_model_dir(trained_pipeline, model_dataset, tmp_path_factory):
    """El modelo pequeño guardado como lo deja export_model.py: carpeta MLflow, metadata.json y ui_days.csv."""
    import mlflow.sklearn

    from mibici_mlops.modeling.export_model import export_ui_days

    root = tmp_path_factory.mktemp("serving")
    mlflow.sklearn.save_model(trained_pipeline, str(root / "model"),
                              serialization_format="cloudpickle",
                              pip_requirements=["scikit-learn"])
    (root / "metadata.json").write_text(
        json.dumps({"model_name": "mibici_salidas_diarias", "model_version": "prueba"}))
    export_ui_days(model_dataset, model_dataset.iloc[0:0], root, start=pd.Timestamp("2025-04-01"))
    return root / "model"


@pytest.fixture
def client(mlflow_model_dir, monkeypatch):
    with start_client(mlflow_model_dir, monkeypatch) as test_client:
        yield test_client


@pytest.fixture
def client_without_model(tmp_path, monkeypatch):
    with start_client(tmp_path / "no_existe", monkeypatch) as test_client:
        yield test_client

Writing ../build/migracion/tests/conftest.py


In [19]:
%%writefile ../build/migracion/tests/helpers.py
"""Datos sintéticos y utilidades que comparten las pruebas."""

import numpy as np
import pandas as pd

from mibici_mlops.config import POI_COLUMNS, SEED
from mibici_mlops.modeling.deploy import to_instance


def make_station_days(n_stations: int = 4, start="2025-01-01", end="2025-04-30", seed=SEED):
    """Demanda diaria con patrón semanal y un nivel distinto por estación."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end)
    rows = []
    for station in range(1, n_stations + 1):
        weekly = np.where(dates.day_of_week < 5, 1.0, 0.5)
        rows.append(pd.DataFrame({
            "Origen_Id": station, "Fecha": dates, "target": rng.poisson(10 * station * weekly),
        }))
    data = pd.concat(rows, ignore_index=True)
    data["day_of_week"] = data["Fecha"].dt.day_of_week
    data["month"] = data["Fecha"].dt.month
    data["weather_zone"] = data["Origen_Id"] % 2
    for i, column in enumerate(POI_COLUMNS):
        data[column] = (data["Origen_Id"] * (i + 1)) % 5
    return data


def start_client(model_uri, monkeypatch):
    """Servicio de mibici_mlops/serving/app.py apuntando al modelo indicado."""
    from fastapi.testclient import TestClient

    from mibici_mlops.serving import app as service

    monkeypatch.setattr(service, "MODEL_URI", str(model_uri))
    return TestClient(service.app)

Writing ../build/migracion/tests/helpers.py


In [20]:
%%writefile ../build/migracion/tests/test_metrics.py
"""Pruebas unitarias de las métricas y de la línea base ingenua."""

import numpy as np
import pandas as pd
import pytest

from mibici_mlops.modeling.evaluate import (
    NAIVE_BASELINE, add_skill, predict_seasonal_naive, predict_station_weekday, regression_metrics,
)


def test_metrics_with_known_values():
    metrics = regression_metrics([0, 2, 4], [1, 2, 7])
    assert metrics["mae"] == pytest.approx(4 / 3)
    assert metrics["rmse"] == pytest.approx(np.sqrt(10 / 3))


def test_perfect_prediction_has_zero_error():
    assert regression_metrics([3, 5, 8], [3, 5, 8]) == {"mae": 0.0, "rmse": 0.0}


def test_rmse_is_never_below_mae():
    rng = np.random.default_rng(0)
    metrics = regression_metrics(rng.poisson(10, 200), rng.poisson(10, 200))
    assert metrics["rmse"] >= metrics["mae"]


def test_naive_uses_lag_7_and_falls_back_to_station_weekday():
    train = pd.DataFrame({"Origen_Id": [1, 1, 2], "day_of_week": [0, 0, 0], "target": [10, 20, 40]})
    new = pd.DataFrame({"Origen_Id": [1, 1, 3], "day_of_week": [0, 0, 0], "lag_7": [7.0, None, None]})
    naive = predict_seasonal_naive(train, new)
    assert naive.tolist() == pytest.approx([7.0, 15.0, 70 / 3])  # 70/3: promedio global del lunes


def test_station_weekday_baseline_covers_new_stations():
    train = pd.DataFrame({"Origen_Id": [1, 2], "day_of_week": [3, 3], "target": [10, 30]})
    new = pd.DataFrame({"Origen_Id": [99], "day_of_week": [3]})
    assert predict_station_weekday(train, new).iloc[0] == pytest.approx(20)


def test_skill_is_improvement_over_naive():
    results = pd.DataFrame({"cutoff": ["2025-01", "2025-02"], "mae": [5.0, 9.0]})
    baseline = pd.DataFrame({"cutoff": ["2025-01", "2025-02"],
                             "model": [NAIVE_BASELINE] * 2, "mae": [10.0, 9.0]})
    assert add_skill(results, baseline)["skill_vs_naive"].tolist() == pytest.approx([0.5, 0.0])

Writing ../build/migracion/tests/test_metrics.py


In [21]:
%%writefile ../build/migracion/tests/test_api.py
"""Pruebas del servicio: contrato de entrada y salida, y manejo de errores."""

import math

from helpers import to_instance
import pytest

from mibici_mlops.config import FEATURE_COLUMNS
from mibici_mlops.serving.app import EXAMPLE_INSTANCE, StationDayFeatures


def test_schema_has_the_model_features_in_order():
    assert list(StationDayFeatures.model_fields) == FEATURE_COLUMNS


def test_health_reports_loaded_model(client):
    body = client.get("/health").json()
    assert body == {"status": "ok", "model_loaded": True,
                    "model_name": "mibici_salidas_diarias", "model_version": "prueba"}


def test_example_instance_gets_a_prediction(client):
    response = client.post("/predict", json={"instances": [EXAMPLE_INSTANCE]})
    assert response.status_code == 200
    prediction = response.json()["predictions"][0]
    assert prediction["Origen_Id"] == EXAMPLE_INSTANCE["Origen_Id"]
    assert prediction["predicted_trips"] >= 0


def test_api_matches_the_pipeline(client, model_dataset, trained_pipeline):
    rows = model_dataset.iloc[-20:]
    response = client.post("/predict", json={"instances": [to_instance(r) for r in rows.to_dict("records")]})
    served = [p["predicted_trips"] for p in response.json()["predictions"]]
    expected = trained_pipeline.predict(rows[FEATURE_COLUMNS]).clip(min=0)
    assert served == pytest.approx(expected, abs=1e-4)


def test_new_station_without_history_is_accepted(client):
    instance = {**EXAMPLE_INSTANCE, "Origen_Id": 9999, "lag_1": None, "lag_7": None, "lag_14": None,
                "roll_mean_7": None, "roll_mean_28": None, "active_days_7": 0, "active_days_28": 0}
    response = client.post("/predict", json={"instances": [instance]})
    assert response.status_code == 200
    assert math.isfinite(response.json()["predictions"][0]["predicted_trips"])


@pytest.mark.parametrize("change", [
    {"day_of_week": 7}, {"month": 13}, {"active_days_7": 8}, {"lag_1": -3},
    {"Origen_Id": "estacion"}, {"variable_extra": 1},
])
def test_invalid_values_return_422(client, change):
    response = client.post("/predict", json={"instances": [{**EXAMPLE_INSTANCE, **change}]})
    assert response.status_code == 422


def test_missing_feature_returns_422(client):
    instance = {k: v for k, v in EXAMPLE_INSTANCE.items() if k != "month"}
    assert client.post("/predict", json={"instances": [instance]}).status_code == 422


def test_empty_request_returns_422(client):
    assert client.post("/predict", json={"instances": []}).status_code == 422


def test_without_model_returns_503(client_without_model):
    assert client_without_model.get("/health").json()["model_loaded"] is False
    response = client_without_model.post("/predict", json={"instances": [EXAMPLE_INSTANCE]})
    assert response.status_code == 503


def test_openapi_documents_predict(client):
    schema = client.get("/openapi.json").json()
    assert "post" in schema["paths"]["/predict"]
    assert "StationDayFeatures" in schema["components"]["schemas"]


def test_station_day_prediction_matches_the_pipeline(client, model_dataset, trained_pipeline):
    row = model_dataset.loc[model_dataset["Fecha"] == "2025-04-15"].iloc[[0]]
    response = client.get("/predict/estacion",
                          params={"origen_id": int(row["Origen_Id"].iloc[0]), "fecha": "2025-04-15"})
    assert response.status_code == 200
    body = response.json()
    expected = max(float(trained_pipeline.predict(row[FEATURE_COLUMNS])[0]), 0)
    assert body["predicted_trips"] == pytest.approx(expected, abs=0.01)
    assert body["actual_trips"] == float(row["target"].iloc[0])
    assert list(body["features"]) == FEATURE_COLUMNS
    assert body["estimated"] is False


def test_station_day_without_history_returns_404(client):
    response = client.get("/predict/estacion", params={"origen_id": 1, "fecha": "2019-01-01"})
    assert response.status_code == 404


def test_future_day_uses_the_last_known_history(client, model_dataset):
    station = model_dataset.loc[model_dataset["Origen_Id"] == 1].set_index("Fecha")["target"]
    response = client.get("/predict/estacion", params={"origen_id": 1, "fecha": "2025-05-14"})
    assert response.status_code == 200
    body = response.json()
    assert body["estimated"] is True and body["history_end"] == "2025-04-30"
    assert body["actual_trips"] is None and body["predicted_trips"] >= 0
    features = body["features"]
    assert (features["day_of_week"], features["month"], features["day_of_month"]) == (2, 5, 14)
    assert features["lag_1"] == station["2025-04-30"]
    assert features["lag_7"] == station["2025-04-24"]
    assert features["roll_mean_7"] == pytest.approx(station["2025-04-24":"2025-04-30"].mean())
    assert features["active_days_28"] == 28


def test_ui_page_lists_stations_and_dates(client):
    response = client.get("/ui")
    assert response.status_code == 200
    assert 'option value="1"' in response.text
    assert 'min="2025-04-01"' in response.text and 'value="2025-04-30"' in response.text


def test_ui_without_model_returns_503(client_without_model):
    assert client_without_model.get("/ui").status_code == 503


def test_root_redirects_to_the_page(client):
    assert client.get("/", follow_redirects=False).headers["location"] == "/ui"


def test_root_without_model_redirects_to_docs(client_without_model):
    assert client_without_model.get("/", follow_redirects=False).headers["location"] == "/docs"

Writing ../build/migracion/tests/test_api.py


In [22]:
%%writefile ../build/migracion/tests/test_reproducibility.py
"""Reproducibilidad: misma semilla y mismo modelo guardado dan los mismos resultados."""

import json

import mlflow.sklearn
import numpy as np
import pandas as pd
import pytest

from mibici_mlops.config import FEATURE_COLUMNS, REGISTERED_MODEL_NAME, SERVING_DIR, TARGET_COLUMN
from mibici_mlops.modeling.evaluate import regression_metrics
from mibici_mlops.modeling.train import train_final_model

needs_serving = pytest.mark.skipif(
    not (SERVING_DIR / "reference_sample.csv").is_file(),
    reason="Falta models/serving (se crea con modeling/export_model.py)",
)


def test_same_seed_gives_same_predictions(model_dataset):
    params = {"n_estimators": 10, "max_depth": 5}
    first = train_final_model("Random Forest", params, model_dataset, FEATURE_COLUMNS)
    second = train_final_model("Random Forest", params, model_dataset, FEATURE_COLUMNS)
    np.testing.assert_allclose(first.predict(model_dataset[FEATURE_COLUMNS]),
                               second.predict(model_dataset[FEATURE_COLUMNS]), rtol=0, atol=1e-9)


def test_mlflow_format_keeps_predictions(mlflow_model_dir, trained_pipeline, model_dataset):
    loaded = mlflow.sklearn.load_model(str(mlflow_model_dir))
    np.testing.assert_allclose(loaded.predict(model_dataset[FEATURE_COLUMNS]),
                               trained_pipeline.predict(model_dataset[FEATURE_COLUMNS]),
                               rtol=0, atol=1e-9)


@needs_serving
def test_serving_metadata_describes_the_model():
    meta = json.loads((SERVING_DIR / "metadata.json").read_text())
    assert meta["model_name"] == REGISTERED_MODEL_NAME
    assert meta["registry_uri"] == f"models:/{REGISTERED_MODEL_NAME}/{meta['model_version']}"
    assert meta["features"] == FEATURE_COLUMNS
    assert meta["seed"] == 42


@needs_serving
def test_serving_model_reproduces_reference_sample():
    reference = pd.read_csv(SERVING_DIR / "reference_sample.csv")
    model = mlflow.sklearn.load_model(str(SERVING_DIR / "model"))
    predictions = model.predict(reference[FEATURE_COLUMNS])
    np.testing.assert_allclose(predictions, reference["reference_prediction"], rtol=0, atol=1e-6)
    current = regression_metrics(reference[TARGET_COLUMN], predictions)
    expected = regression_metrics(reference[TARGET_COLUMN], reference["reference_prediction"])
    assert current == pytest.approx(expected, abs=1e-9)

Writing ../build/migracion/tests/test_reproducibility.py


In [23]:
%%writefile ../build/migracion/tests/test_drift.py
"""Pruebas unitarias del monitoreo de drift (mibici_mlops/monitoring.py)."""

import numpy as np
import pandas as pd
import pytest

from mibici_mlops import monitoring

BASE = {"baseline_mae": 6.0, "baseline_missing": 0.10, "reference_demand": 30.0}


def period(**changes) -> pd.Series:
    """Un periodo sin problemas; cada prueba cambia solo lo que quiere revisar."""
    row = {"mae": 6.0, "mejora_vs_ingenuo": 0.25, "sesgo": 0.0, "lag_1_vacío": 0.10,
           "proporción_drift": 0.0, "psi_máximo": 0.02,
           "cambio_estaciones_activas": 0.0, "cambio_viajes_por_día": 0.0}
    return pd.Series({**row, **changes})


@pytest.fixture
def samples():
    rng = np.random.default_rng(42)
    reference = pd.DataFrame({"estable": rng.normal(10, 2, 5_000), "desplazada": rng.normal(10, 2, 5_000)})
    current = pd.DataFrame({"estable": rng.normal(10, 2, 5_000), "desplazada": rng.normal(13, 2, 5_000)})
    return reference, current


def test_psi_is_near_zero_for_the_same_distribution(samples):
    reference, current = samples
    assert monitoring.psi(reference["estable"], current["estable"]) < 0.02


def test_psi_detects_a_shift(samples):
    reference, current = samples
    assert monitoring.psi(reference["desplazada"], current["desplazada"]) > monitoring.PSI_LEVELS[1]


def test_psi_ignores_missing_values():
    reference = pd.Series([1.0, 2.0, 3.0, 4.0] * 50)
    current = pd.concat([reference, pd.Series([np.nan] * 100)])
    assert monitoring.psi(reference, current) == pytest.approx(0.0, abs=1e-9)


def test_drift_table_flags_only_the_shifted_variable(samples):
    reference, current = samples
    table = monitoring.drift_table(reference, current, ["estable", "desplazada"]).set_index("variable")
    assert not table.loc["estable", "drift"]
    assert table.loc["desplazada", "drift"]


def test_constant_variable_does_not_break_the_table():
    reference = pd.DataFrame({"zona": [3] * 100})
    assert not monitoring.drift_table(reference, reference.copy(), ["zona"]).loc[0, "drift"]


def test_level_uses_warning_and_critical_limits():
    assert [monitoring.level(v, (0.1, 0.2)) for v in (0.05, 0.15, 0.25)] == [0, 1, 2]


def test_daily_operation_counts_stations_and_trips():
    data = pd.DataFrame({"Fecha": ["d1", "d1", "d2"], "Origen_Id": [1, 2, 1], "target": [5, 7, 4]})
    assert monitoring.daily_operation(data) == (1.5, 8.0)


def test_normal_period_has_no_alerts():
    levels = monitoring.alert_levels(period(), **BASE)
    assert set(levels.values()) == {0}
    assert monitoring.decide(levels) == "Sin acción"


@pytest.mark.parametrize("changes, alert, expected", [
    ({"lag_1_vacío": 0.20}, "A1_datos_faltantes", 1),
    ({"lag_1_vacío": 0.30}, "A1_datos_faltantes", 2),
    ({"sesgo": -4.0}, "A2_sesgo", 1),
    ({"sesgo": 7.0}, "A2_sesgo", 2),
    ({"mae": 7.5}, "A3_desempeño", 1),
    ({"mae": 7.5, "mejora_vs_ingenuo": 0.10}, "A3_desempeño", 2),
    ({"psi_máximo": 0.15}, "A4_drift", 1),
    ({"proporción_drift": 0.5}, "A4_drift", 1),
    ({"psi_máximo": 0.40}, "A4_drift", 2),
    ({"cambio_estaciones_activas": 0.20}, "A5_operación", 1),
    ({"cambio_viajes_por_día": -0.35}, "A5_operación", 2),
])
def test_each_alert_and_level(changes, alert, expected):
    levels = monitoring.alert_levels(period(**changes), **BASE)
    assert levels[alert] == expected
    assert all(value == 0 for name, value in levels.items() if name != alert)


def test_data_problems_come_before_retraining():
    levels = monitoring.alert_levels(period(lag_1_vacío=0.30, mae=8.0, mejora_vs_ingenuo=0.0), **BASE)
    assert monitoring.decide(levels, previous_a3=2) == "Revisar el pipeline de datos (no reentrenar)"


def test_retrain_only_after_two_critical_months():
    levels = monitoring.alert_levels(period(mae=8.0, mejora_vs_ingenuo=0.10), **BASE)
    assert monitoring.decide(levels, previous_a3=1) == "Vigilar y confirmar con el siguiente mes"
    assert monitoring.decide(levels, previous_a3=2) == "Reentrenar: A3 crítica dos meses seguidos"


def test_operation_change_without_performance_loss():
    levels = monitoring.alert_levels(period(cambio_viajes_por_día=0.60), **BASE)
    assert monitoring.decide(levels) == "Revisar con operación el cambio de la red"


def test_warnings_are_only_watched():
    levels = monitoring.alert_levels(period(psi_máximo=0.15), **BASE)
    assert monitoring.decide(levels) == "Vigilar"

Writing ../build/migracion/tests/test_drift.py


In [24]:
%%writefile ../build/migracion/tests/test_calendar.py
"""Pruebas de la variable de días festivos (add_holidays en mibici_mlops/features.py)."""

import pandas as pd

from mibici_mlops.features import add_holidays

HOLIDAYS_2025 = ["2025-01-01", "2025-02-03", "2025-03-17", "2025-05-01",
                 "2025-09-16", "2025-11-17", "2025-12-25"]


def calendar(dates) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(dates))
    return pd.DataFrame({"Fecha": dates, "month": dates.dt.month,
                         "day_of_month": dates.dt.day, "day_of_week": dates.dt.day_of_week})


def test_marks_the_phase2_holidays_of_2025():
    year = calendar(pd.date_range("2025-01-01", "2025-12-31"))
    marked = year.loc[add_holidays(year)["is_holiday"] == 1, "Fecha"].dt.strftime("%Y-%m-%d")
    assert marked.tolist() == HOLIDAYS_2025


def test_monday_holidays_depend_on_the_weekday():
    # 10 de febrero de 2025 es lunes, pero es el segundo lunes: no es festivo
    assert add_holidays(calendar(["2025-02-10"]))["is_holiday"].iloc[0] == 0


def test_does_not_change_the_input():
    data = calendar(["2025-09-16"])
    add_holidays(data)
    assert "is_holiday" not in data

Writing ../build/migracion/tests/test_calendar.py


In [25]:
%%writefile ../build/migracion/tests/test_integration_pipeline.py
"""Integración: del CSV crudo a las métricas, con limpieza, demanda diaria, variables, modelo y predicción."""

import json

from helpers import make_station_days, start_client, to_instance
import mlflow.sklearn
import numpy as np
import pandas as pd
import pytest

from mibici_mlops.config import (
    FEATURE_COLUMNS, LAGS, POI_COLUMNS, ROLLING_WINDOWS, SEED, SERVING_DIR, TARGET_COLUMN,
)
from mibici_mlops.dataset import build_daily_demand
from mibici_mlops.features import build_model_dataset, split_by_date
from mibici_mlops.ingestion import read_csv_chunks
from mibici_mlops.modeling.evaluate import predict_seasonal_naive, regression_metrics
from mibici_mlops.modeling.train import train_final_model
from mibici_mlops.preprocessing import (
    add_trip_features, clean_strategy_b, count_hourly_demand, normalize_chunk,
)

pytestmark = pytest.mark.integration
TEST_START = pd.Timestamp("2025-04-01")


def write_raw_trips(path, station_days: pd.DataFrame) -> int:
    """Pasar la demanda diaria a viajes individuales con el formato de datos abiertos."""
    rng = np.random.default_rng(SEED)
    rows, trip_id = [], 1
    for record in station_days.itertuples():
        for _ in range(record.target):
            start = record.Fecha + pd.Timedelta(minutes=int(rng.integers(6 * 60, 22 * 60)))
            rows.append({
                "Viaje_Id": trip_id, "Usuario_Id": int(rng.integers(1, 5000)),
                "Genero": rng.choice(["M", "F"]), "Año_de_nacimiento": int(rng.integers(1960, 2005)),
                "Inicio_del_viaje": start.strftime("%Y-%m-%d %H:%M:%S"),
                "Fin_del_viaje": (start + pd.Timedelta(minutes=15)).strftime("%Y-%m-%d %H:%M:%S"),
                "Origen_Id": record.Origen_Id, "Destino_Id": 1 + record.Origen_Id % 4,
            })
            trip_id += 1
    dirty = [
        {**rows[0]},                                                    # Viaje_Id repetido
        {**rows[1], "Viaje_Id": trip_id, "Origen_Id": 407},             # estación excluida
        {**rows[2], "Viaje_Id": trip_id + 1, "Inicio_del_viaje": "no es fecha"},
        {**rows[3], "Viaje_Id": trip_id + 2, "Inicio_del_viaje": "2019-05-01 08:00:00"},
    ]
    pd.DataFrame(rows + dirty).to_csv(path, index=False)
    return len(rows)


@pytest.fixture(scope="module")
def pipeline_run(tmp_path_factory):
    station_days = make_station_days()
    raw_path = tmp_path_factory.mktemp("raw") / "datos_abiertos_2025_01.csv"
    valid_rows = write_raw_trips(raw_path, station_days)

    # 1. Carga y limpieza por bloques, como en el pipeline
    reader, _ = read_csv_chunks(raw_path, chunk_size=5_000)
    hourly_parts, seen_ids, kept = [], set(), 0
    for chunk in reader:
        cleaned, summary, new_ids = clean_strategy_b(normalize_chunk(chunk), seen_ids)
        seen_ids |= new_ids
        kept += summary["kept_rows"]
        hourly_parts.append(count_hourly_demand(add_trip_features(cleaned)))
    hourly = pd.concat(hourly_parts).groupby(level=[0, 1]).sum().reset_index()

    # 2. Demanda diaria y atributos de las estaciones
    daily = build_daily_demand(hourly)
    stations = station_days.drop_duplicates("Origen_Id")[["Origen_Id", "weather_zone"] + POI_COLUMNS]
    integrated = daily.merge(stations, on="Origen_Id", how="left", validate="many_to_one")

    # 3. Variables, separación temporal, entrenamiento y predicción
    dataset = build_model_dataset(integrated, 1, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
    development, test = split_by_date(dataset, TEST_START)
    model = train_final_model("Random Forest", {"n_estimators": 30, "max_depth": 6},
                              development, FEATURE_COLUMNS, TARGET_COLUMN)
    return {"station_days": station_days, "valid_rows": valid_rows, "kept": kept, "daily": daily,
            "development": development, "test": test, "model": model,
            "predictions": model.predict(test[FEATURE_COLUMNS])}


def test_cleaning_removes_only_the_dirty_rows(pipeline_run):
    assert pipeline_run["kept"] == pipeline_run["valid_rows"]


def test_daily_demand_matches_the_source(pipeline_run):
    expected = pipeline_run["station_days"].query("target > 0")[["Origen_Id", "Fecha", "target"]]
    merged = expected.merge(pipeline_run["daily"], on=["Origen_Id", "Fecha"], suffixes=("", "_obtenido"))
    assert len(merged) == len(expected)
    assert (merged["target"] == merged["target_obtenido"]).all()


def test_features_have_the_model_columns(pipeline_run):
    test = pipeline_run["test"]
    assert list(test[FEATURE_COLUMNS].columns) == FEATURE_COLUMNS
    assert test["lag_7"].notna().all()  # en abril ya hay más de 14 días de historia


def test_split_is_temporal(pipeline_run):
    assert pipeline_run["development"]["Fecha"].max() < TEST_START
    assert pipeline_run["test"]["Fecha"].min() >= TEST_START


def test_model_beats_the_seasonal_naive(pipeline_run):
    test, development = pipeline_run["test"], pipeline_run["development"]
    model_metrics = regression_metrics(test[TARGET_COLUMN], pipeline_run["predictions"])
    naive_metrics = regression_metrics(test[TARGET_COLUMN],
                                       predict_seasonal_naive(development, test, TARGET_COLUMN))
    assert np.isfinite(list(model_metrics.values())).all()
    assert model_metrics["mae"] < naive_metrics["mae"]


def test_api_serves_the_trained_model(pipeline_run, tmp_path, monkeypatch):
    model_dir = tmp_path / "model"
    mlflow.sklearn.save_model(pipeline_run["model"], str(model_dir),
                              serialization_format="cloudpickle", pip_requirements=["scikit-learn"])
    test = pipeline_run["test"]
    payload = {"instances": [to_instance(r) for r in test.to_dict("records")]}
    with start_client(model_dir, monkeypatch) as client:
        response = client.post("/predict", json=payload)
    assert response.status_code == 200
    served = np.array([p["predicted_trips"] for p in response.json()["predictions"]])
    assert served == pytest.approx(pipeline_run["predictions"].clip(min=0), abs=1e-4)
    served_mae = regression_metrics(test[TARGET_COLUMN], served)["mae"]
    assert served_mae == pytest.approx(
        regression_metrics(test[TARGET_COLUMN], pipeline_run["predictions"].clip(min=0))["mae"], abs=1e-4)


@pytest.mark.skipif(not (SERVING_DIR / "reference_sample.csv").is_file(),
                    reason="Falta models/serving (se crea con modeling/export_model.py)")
def test_serving_model_through_the_api_matches_reference(monkeypatch):
    meta = json.loads((SERVING_DIR / "metadata.json").read_text())
    reference = pd.read_csv(SERVING_DIR / "reference_sample.csv")
    payload = {"instances": [to_instance(r) for r in reference.to_dict("records")]}
    with start_client(SERVING_DIR / "model", monkeypatch) as client:
        body = client.post("/predict", json=payload).json()
    assert body["model_version"] == meta["model_version"]
    served = np.array([p["predicted_trips"] for p in body["predictions"]])
    assert served == pytest.approx(reference["reference_prediction"].clip(lower=0), abs=1e-4)

Writing ../build/migracion/tests/test_integration_pipeline.py


In [26]:
%%writefile ../build/migracion/tests/test_data.py
import os

import pandas as pd
import pytest

from mibici_mlops.checks import check_integration
from mibici_mlops.dataset import cache_is_current, save_cache_manifest
from mibici_mlops.versioning import version_csv_sources


def test_cache_detects_changed_file(tmp_path):
    data_path = tmp_path / "sample.parquet"
    data_path.write_bytes(b"first version")
    signature = {"period": "2020-2026", "schema": 1}
    save_cache_manifest(data_path, signature)
    assert cache_is_current(data_path, signature)
    assert not cache_is_current(data_path, {"period": "2021-2026", "schema": 1})
    data_path.write_bytes(b"changed version")
    assert not cache_is_current(data_path, signature)


def test_csv_versioning_keeps_history(tmp_path):
    source, versions = tmp_path / "sources", tmp_path / "versions"
    source.mkdir()
    csv_path = source / "sample.csv"
    csv_path.write_text("value\n1\n", encoding="utf-8")
    first = version_csv_sources(source, versions)
    assert len(first) == 1
    assert first.iloc[0]["version"] == 1
    assert version_csv_sources(source, versions).iloc[0]["known_versions"] == 1
    first_mtime = csv_path.stat().st_mtime_ns
    csv_path.write_text("value\n2\n", encoding="utf-8")
    # Mismo tamaño y, en algunos sistemas de archivos, misma marca de tiempo; se adelanta un segundo
    os.utime(csv_path, ns=(first_mtime + 1_000_000_000, first_mtime + 1_000_000_000))
    second = version_csv_sources(source, versions)
    assert second.iloc[0]["version"] == 2
    assert second.iloc[0]["known_versions"] == 2
    assert csv_path.read_text(encoding="utf-8") == "value\n2\n"


def test_integration_rejects_changed_demand():
    daily = pd.DataFrame({
        "Origen_Id": [1, 1],
        "Fecha": pd.to_datetime(["2025-01-01", "2025-01-02"]),
        "target": [10, 20],
    })
    integrated = daily.copy()
    check_integration(daily, integrated)
    integrated.loc[0, "target"] = 11
    with pytest.raises(ValueError):
        check_integration(daily, integrated)

Writing ../build/migracion/tests/test_data.py


## 15. `README.md`
Actualizamos la sección 10 con las instrucciones para ejecutar el proyecto desde la terminal: instalar, exportar el modelo, correr las pruebas, construir y correr la imagen, las direcciones del servicio (http://localhost:8000, `/ui`, `/docs`, `/redoc` y `/health`), cómo descargar la imagen publicada, cómo publicar una versión nueva y cómo correr el monitoreo y el challenger. También se actualizan la estructura del repositorio, la sección de pruebas, la versión del champion y la URL del repositorio.

In [27]:
%%writefile ../build/migracion/README.md
# Equipo 8 — MiBici MLOps

Proyecto de la materia de MLOps para analizar los viajes del sistema MiBici de Guadalajara y predecir **cuántos viajes comenzarán en cada estación durante el día siguiente**. La idea es contar con una estimación de las salidas por estación que ayude a planear mejor la disponibilidad y distribución de bicicletas.

### Problema y variable objetivo

En MiBici no todas las estaciones tienen el mismo movimiento. Algunas concentran muchas salidas y otras registran pocas; además, la actividad cambia según el día de la semana, la época del año y el comportamiento reciente de cada estación. Esto complica anticipar dónde podría hacer falta disponer de más bicicletas.

Por eso definimos la variable objetivo **`target` como el número de viajes que comienzan en una estación durante un día**. Cada fila del conjunto de modelado representa una combinación **estación-fecha**. Para construirla, agrupamos los viajes por estación de origen (`Origen_Id`) y día de inicio del viaje; en el dataset diario, el conteo `Cantidad_viajes` se utiliza como `target`.

Elegimos las **salidas**, en lugar de las llegadas o el total de movimientos, porque queremos estimar cuántas bicicletas podrían necesitar los usuarios para iniciar sus recorridos en cada estación. También elegimos la **escala diaria** porque resulta útil para la planeación operativa y permite comparar estaciones y fechas sin depender de las variaciones de cada hora.

El modelo se plantea para **predecir la demanda del día siguiente** (`horizon_days = 1`), utilizando información histórica, variables de calendario y características de las estaciones. Las variables históricas deben calcularse únicamente con datos anteriores al día que se predice, para evitar utilizar información del futuro.

Esta predicción puede apoyar la identificación de estaciones con mayor demanda esperada y la planeación de la redistribución de bicicletas. **No indica directamente cuántas bicicletas habrá disponibles ni cuántos viajes no pudieron realizarse**: estima los viajes iniciados que aparecen en los registros. Para decidir movimientos concretos de bicicletas también harían falta datos de disponibilidad, capacidad y llegadas.

El repositorio utiliza como base [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/), adaptado para las Fases 1 y 2.

## 1. Alcance y resultados

- **Fase 1:** lectura de datos, validaciones, limpieza, generación de Parquet, particiones temporales, análisis exploratorio y registro de resultados.
- **Fase 2:** refactorización a módulos `.py`, construcción de variables, entrenamiento y comparación de modelos, validación temporal, prueba independiente y registro en MLflow Tracking y Model Registry.
- **Entrega final:** servicio de predicción con FastAPI, imagen de Docker publicada en DockerHub, pruebas unitarias y de integración, monitoreo de data drift con alertas y comparación champion contra challenger (sección 10).
- **Periodo de datos:** enero de 2020 a septiembre de 2026, inclusive. La Fase 1 trabaja con 81 CSV mensuales de viajes y un catálogo de estaciones; el conjunto original supera los 27 millones de registros de viajes.
- **Modelo final registrado:** Random Forest con 18 variables de calendario, estación e historia. La ejecución revisada utiliza 12 ventanas de validación en 2025 y prueba en 2026.

### Resultados de la ejecución revisada

| Métrica | Valor |
| --- | ---: |
| MAE promedio de validación (2025) | 5.4384 viajes/estación-día |
| RMSE promedio de validación (2025) | 8.7375 viajes/estación-día |
| MAE de prueba (2026) | 5.8964 viajes/estación-día |
| RMSE de prueba (2026) | 10.7824 viajes/estación-día |
| MAE de línea base de prueba | 7.9566 viajes/estación-día |
| Reducción relativa del MAE frente a la línea base | 25.89 % |
| Filas de prueba | 108,294 |

Estos valores corresponden a una ejecución concreta, no son resultados garantizados en otros entornos o con otros datos. HistGradientBoosting obtuvo un MAE de validación ligeramente menor que Random Forest en la comparación inicial; la elección del modelo final debe explicarse en el informe.

## 2. Organización del repositorio

La estructura utilizada para ejecutar desde la raíz del repositorio es:

```text
equipo-8-mibici-mlops/
├── data/
│   ├── raw/
│   │   ├── 2020/ ... 2026/       # CSV mensuales, separados por año
│   │   ├── Manual_datos/         # Catálogo de estaciones
│   │   ├── weather/              # Datos de clima y zonas
│   │   └── pois/                 # POI de estaciones y archivo OSM
│   ├── interim/                  # Archivos intermedios
│   └── processed/                # Parquet y conjuntos procesados
├── docs/                         # Documentación y análisis de requerimientos
├── mibici_mlops/
│   ├── __init__.py
│   ├── config.py                 # Rutas, fechas y opciones de ejecución
│   ├── ingestion.py              # Lectura de datos
│   ├── preprocessing.py          # Limpieza
│   ├── pipeline.py               # Procesamiento de Fase 1
│   ├── dataset.py                # Preparación y validación del dataset
│   ├── external.py               # Datos externos: clima y POI
│   ├── features.py               # Variables de calendario, historia y días festivos
│   ├── checks.py                 # Comprobaciones de calidad
│   ├── eda.py                    # Análisis exploratorio
│   ├── plots.py                  # Gráficas
│   ├── reporting.py              # Reportes y tiempos
│   ├── tracking.py               # MLflow Tracking y Registry
│   ├── versioning.py             # Inventarios, SHA-256 y procedencia
│   ├── workflow.py               # Orquestación de Fase 2
│   ├── main.py                   # Punto de entrada
│   ├── monitoring.py             # Data drift, escenarios, alertas y acción
│   ├── mlflow_catalog.py         # Descripción y etiquetas de experimentos, linaje de la Fase 3
│   ├── serving/
│   │   ├── __init__.py
│   │   └── app.py                # API de FastAPI y página /ui
│   └── modeling/
│       ├── __init__.py
│       ├── train.py              # Entrenamiento
│       ├── evaluate.py           # Validación y métricas
│       ├── predict.py            # Predicciones
│       ├── export_model.py       # Exporta el champion a models/serving
│       ├── challenger.py         # Champion contra challenger
│       └── deploy.py             # Construye, comprueba y publica la imagen
├── notebooks/                    # Desarrollo y exploración
├── tests/                        # Pruebas automatizadas
├── models/                       # Modelos serializados locales y models/serving para el servicio
├── references/                   # Referencias del proyecto
├── reports/
│   ├── validation/              # Resultados de Fase 1
│   ├── modeling/                # Comparación, validación y prueba
│   ├── figures/                 # Figuras y gráficas
│   ├── data_versioning/         # Versiones de archivos
│   ├── mlflow_provenance/       # Procedencia del dataset
│   ├── api/                     # Especificación OpenAPI
│   ├── reproducibility/         # Comparación local contra Docker
│   ├── drift/                   # Monitoreo de octubre de 2026
│   └── tests/                   # Salida de pytest
├── .dockerignore
├── Dockerfile                   # Imagen del servicio
├── .gitignore
├── MLproject                    # Configuración de MLflow Projects
├── Makefile
├── pyproject.toml
├── python_env.yaml
├── requirements.txt             # Entorno con versiones fijas
├── requirements-serving.txt     # Librerías de la imagen de Docker
└── README.md
```

Las carpetas de resultados se crean durante la ejecución. La lista muestra los módulos utilizados en la implementación revisada; **comprobar que estén integrados en la rama que se vaya a entregar**.

## 3. Descargar y organizar los datos

Los archivos grandes se mantienen fuera de GitHub. Se obtienen de la [carpeta compartida MiBici en Google Drive](https://drive.google.com/drive/folders/1n83lQ_nEH9u7KvKAYNRcnUhae5tjoAYB?usp=drive_link).

**No es necesario descargar todo el contenido de Drive si ya se tienen los archivos procesados y sus manifiestos válidos**, pero para reconstruir desde cero sí se requieren los archivos de origen.

### 3.1. Archivos necesarios y ubicación

| En Google Drive | Guardar en el repositorio | Uso |
| --- | --- | --- |
| Carpetas `2020`, `2021`, `2022`, `2023`, `2024`, `2025`, `2026` | `data/raw/2020/` ... `data/raw/2026/` | CSV mensuales de viajes (81 en total para el periodo del proyecto) |
| Carpeta `Manual_datos`, incluido `nomenclatura_2026_09.csv` | `data/raw/Manual_datos/` | Catálogo e identificadores de estaciones |
| `weather_daily_zones_2020_2026.csv` | `data/raw/weather/weather_daily_zones_2020_2026.csv` | Información meteorológica por zona |
| `mibici_station_weather_zones.csv` | `data/raw/weather/mibici_station_weather_zones.csv` | Relación entre estaciones y zonas meteorológicas |
| `mibici_station_pois.csv` | `data/raw/pois/mibici_station_pois.csv` | Puntos de interés cercanos a estaciones |
| `mexico-latest.osm.pbf` | `data/raw/pois/mexico-latest.osm.pbf` | Fuente OpenStreetMap para POI; archivo grande, aproximadamente 618 MB |
| `metadato_limites_municipales_jalisco_iieg.xlsx` | `data/raw/metadato_limites_municipales_jalisco_iieg.xlsx` | Material geográfico de referencia; no se ha confirmado como entrada obligatoria del entrenamiento |

**Importante:** no cambiar los nombres de los CSV originales ni colocarlos todos directamente en `data/raw/`. Mantener la estructura por año. La captura de Drive muestra los nombres anteriores, pero no permite comprobar el contenido de cada carpeta anual: verificar que estén los 81 archivos mensuales esperados.

Si se descarga una carpeta como ZIP, descomprimirla antes de ejecutar. Evitar una carpeta adicional como `data/raw/MiBici/2020/` si el código espera `data/raw/2020/`.

### 3.2. Fuentes externas y caché

El código también contempla Open-Meteo y OpenStreetMap. Algunas ejecuciones pueden reutilizar datos externos previamente descargados o generados. Para reproducir **exactamente** una ejecución se deben conservar los mismos archivos, no solo consultar otra vez una API, porque las fuentes externas pueden cambiar.

Los archivos originales de `data/raw/` no deben modificarse. Los resultados de limpieza y transformación se guardan en `data/interim/` y `data/processed/`.

## 4. Preparar el entorno

Las instrucciones siguientes están pensadas para **Linux o WSL Ubuntu**, desde la raíz del repositorio.

### 4.1. Clonar y entrar al proyecto

```bash
git clone https://github.com/MarioSanchezValtierra/equipo-8-mibici-mlops.git
cd equipo-8-mibici-mlops
```

### 4.2. Crear y activar el entorno virtual

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

Si `pyproject.toml` declara correctamente el paquete instalable, también puede utilizarse:

```bash
python -m pip install -e .
```

Este último comando instala el proyecto en modo editable; no sustituye necesariamente todas las dependencias de `requirements.txt` si no están declaradas en `pyproject.toml`.

Comprobar el entorno:

```bash
python --version
python -m pip check
```

Se utilizó Python 3.13 en el entorno de desarrollo revisado. Para reproducir resultados, conviene mantener la misma versión de Python y las mismas versiones de bibliotecas. Si se actualizan dependencias, registrar ese cambio en Git y en MLflow.

## 5. Configuración antes de ejecutar

Revisar `mibici_mlops/config.py`. Las opciones disponibles dependen de la versión del código integrada en la rama:

| Opción | Para qué sirve |
| --- | --- |
| `RUN_ALL_FILES` | `True`: procesamiento completo; `False`: prueba limitada de Fase 1, cuando esta opción está habilitada |
| `FORCE_REBUILD_DATA` | Fuerza la reconstrucción del dataset integrado en lugar de reutilizar el caché |
| `RUN_GRID_SEARCH` | Habilita la búsqueda de hiperparámetros; puede aumentar considerablemente el tiempo de ejecución |
| `RUN_WEATHER_REFERENCE` | Habilita el experimento de referencia con clima, si está implementado en la versión utilizada |
| `HORIZON_DAYS` | Horizonte de predicción: `1` significa estimar las salidas del día siguiente |
| `FIRST_CUTOFF`, `TEST_START`, `N_FOLDS` | Fechas y número de ventanas de validación temporal |
| `MLFLOW_TRACKING_URI` | Ubicación de la base de datos de seguimiento |

Para repetir la ejecución de referencia, **no cambiar las fechas, el horizonte, las variables ni los hiperparámetros**. Las opciones de prueba rápida no equivalen a una evaluación final.

Las rutas deben derivarse de la raíz del repositorio, sin depender de `/home/usuario/` ni del directorio desde el que se abrió VS Code. El proyecto debe poder ejecutarse después de clonarlo en otra computadora.

## 6. Orden de ejecución

### 6.1. Primera ejecución desde cero

1. Clonar el repositorio y seleccionar la rama o el commit que se quiere reproducir.
2. Crear el entorno e instalar las dependencias.
3. Descargar y organizar los archivos de la sección 3.
4. Revisar las opciones de `config.py`.
5. **Ejecutar primero el procesamiento de Fase 1** si todavía no existen los Parquet y una ejecución válida en `reports/validation/`. La preparación de Fase 2 depende de estos resultados.
6. Ejecutar la Fase 2 desde la raíz del repositorio.
7. Revisar los reportes, los resultados de prueba y los registros de MLflow.
8. Ejecutar las pruebas automatizadas.

**Punto a verificar:** el comando exacto para lanzar **solo la Fase 1** depende de cómo quedó integrado el punto de entrada de `MiBiciPipeline` en la rama final. No se documenta un comando no confirmado. Antes de publicar, agregar aquí ese comando verificado; `python -m mibici_mlops.main` corresponde al flujo de Fase 2 que se ejecutó.

### 6.2. Ejecutar la Fase 2

Desde la raíz del repositorio, con el entorno activado:

```bash
python -m mibici_mlops.main
```

Este comando ejecuta la orquestación de Fase 2 que quedó integrada en el paquete `mibici_mlops/`: preparación del dataset, variables, validación, comparación, entrenamiento final, prueba, registro de artefactos y Model Registry.

Si aparece un error indicando que no hay una ejecución completa de Fase 1 en `reports/validation/`, primero deben generarse esos resultados. **No basta con tener los CSV en Drive**.

En una implementación alternativa con paquete bajo `src/` y `main.py` en la raíz, el punto de entrada es `python main.py`; **no mezclar esa estructura con la del repositorio actual**.

### 6.3. Ejecuciones posteriores

Si `data/processed/mibici_integrated.parquet` y su manifiesto siguen siendo válidos, el código puede reutilizar el dataset integrado. Si cambian los CSV, el catálogo, las fuentes externas o las reglas de limpieza, reconstruir los datos y registrar la nueva versión.

No borrar archivos de datos ni reportes anteriores sin revisar sus dependencias. No modificar el conjunto de prueba para mejorar los resultados de validación.

## 7. MLflow Tracking y Model Registry

La implementación registra el experimento `MiBici_Fase2_Modelado`, parámetros, métricas, datasets, artefactos y modelo final. La entrega final agrega dos experimentos con la misma estructura, `MiBici_Fase3_Monitoreo_Drift` y `MiBici_Fase3_Champion_Challenger` (sección 10.7).

### 7.1. Abrir la interfaz

Desde la raíz del repositorio:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Abrir <http://127.0.0.1:5000>. La base `mlflow.db` debe ser la misma configurada en `MLFLOW_TRACKING_URI`; si no aparece el experimento esperado, comprobar primero esa coincidencia.

### 7.2. Qué revisar en cada ejecución

- **Overview:** nombre, descripción, estado, fuente de ejecución, parámetros y métricas.
- **Model metrics:** métricas de validación y prueba; las series exploratorias pueden utilizar `step` como índice de mes o categoría, no necesariamente como época de entrenamiento.
- **Artifacts:** gráficas, tablas, configuración, características, versiones de datos, reportes y tiempos.
- **Datasets:** identificadores de los conjuntos utilizados para entrenamiento, validación y prueba.
- **Models / Registry:** modelo `mibici_salidas_diarias`, versión registrada y alias `champion`.

La versión 7 es el modelo de la Fase 2 (el que guarda `models/mibici_final_model.joblib`). El alias `champion` apunta a la versión 10, que es la que se sirve (sección 10.1). Conviene comprobar directamente en el Registry que el alias apunte a la versión deseada.

### 7.3. Trazabilidad y reproducibilidad

Se registran, cuando están disponibles:

- Commit y rama de Git (`git_commit`, `git_branch`, `implementation_commit`).
- Estado del árbol de trabajo (`git_dirty`).
- Hashes SHA-256, inventarios y procedencia de los archivos.
- Configuración de entrenamiento y lista de características.
- Versiones de Python y bibliotecas.
- Tiempos por etapa y artefactos de evaluación.

**Si `git_dirty=true`, el commit por sí solo no permite reconstruir exactamente el código ejecutado.** Antes de generar la evidencia definitiva, confirmar los cambios que deben formar parte del repositorio. No se debe alterar una ejecución histórica para hacerla parecer limpia.

Los archivos grandes no se copian a MLflow; los manifiestos registran sus rutas relativas y hashes. Esto permite verificar los datos, pero no sustituye su almacenamiento en Drive.

## 8. Resultados generados

| Ruta | Contenido |
| --- | --- |
| `data/processed/` | Parquet procesados e integrados |
| `reports/validation/` | Validaciones y resultados de Fase 1 |
| `reports/modeling/` | Comparación de modelos, validación temporal y resultados de prueba |
| `reports/figures/modeling/` | Gráficas de comparación y predicciones |
| `reports/data_versioning/` | Versiones y hashes de archivos |
| `reports/mlflow_provenance/` | Procedencia de los datos |
| `models/` | Modelo final serializado localmente |
| `mlruns/` | Artefactos y registros locales de MLflow |
| `mlflow.db` | Base de datos local de MLflow |

En los artefactos de MLflow, `README.md` explica las carpetas; `features.json` documenta las 18 variables y `documentation/EDA_METRICS_README.md` describe las métricas exploratorias y sus unidades. También se incluyen archivos como `modeling/comparison/model_comparison.csv`, `modeling/validation/final_walk_forward.csv`, `modeling/test/test_results_2026.csv` y `performance/execution_times.csv`.

## 9. Pruebas automatizadas

Ejecutar desde la raíz:

```bash
pytest -q
```

Son 82 pruebas, todas con datos pequeños generados en el momento, así que no necesitan los CSV originales:

| Archivo | Tipo | Qué revisa |
| --- | --- | --- |
| `test_data.py`, `test_features.py`, `test_evaluate.py`, `test_modeling.py`, `test_structure.py`, `test_project_paths.py` | unitarias | procesamiento, variables, evaluación, estructura y rutas (Fase 2) |
| `test_metrics.py` | unitaria | MAE, RMSE, línea base ingenua y mejora contra ella |
| `test_api.py` | unitaria | esquema de entrada, `/predict`, errores 422 y 503, página `/ui` y documentación |
| `test_reproducibility.py` | unitaria | misma semilla, mismo resultado; el modelo de `models/serving` reproduce su muestra de referencia |
| `test_drift.py` | unitaria | PSI, tabla de drift, niveles de las alertas A1 a A5 y acción propuesta |
| `test_calendar.py` | unitaria | variable de días festivos |
| `test_integration_pipeline.py` | integración | del CSV crudo a las métricas: limpieza, demanda diaria, variables, entrenamiento y predicción directa y por la API |

Las pruebas del modelo desplegado leen `models/serving/`; si no existe (antes de `make export-model`), se saltan.

## 10. Entrega final: servicio de predicción, Docker y monitoreo

El modelo se expone con FastAPI y se empaqueta en una imagen de Docker publicada en DockerHub. Todos los comandos se corren desde la raíz del repositorio, con el entorno activado (sección 4).

### 10.1. Modelo que se sirve

| Uso | Modelo | URI en MLflow |
| --- | --- | --- |
| Reproducir la Fase 2 y monitorear octubre de 2026 | versión 7 (Random Forest de la Fase 2) | `models:/mibici_salidas_diarias/7` |
| Servicio (FastAPI y Docker) | versión 10, alias `champion` (Random Forest reentrenado hasta septiembre de 2026, con la variable de días festivos) | `models:/mibici_salidas_diarias/10` |

La versión 10 salió de la comparación champion contra challenger: se eligió en septiembre y se probó en octubre de 2026, donde bajó el MAE 3.3 % frente a la versión 7 (7.42 contra 7.68) sin empeorar el RMSE.

`make export-model` copia la versión con el alias `champion` a `models/serving/` junto con:

- `metadata.json`: versión, ejecución, variables, semilla, hash de los datos y versiones de las librerías.
- `reference_sample.csv`: 500 filas fijas de la prueba 2026 con la predicción del modelo, para comprobar que el servicio y la imagen responden igual.
- `ui_days.csv`: las variables por estación y día de septiembre y octubre de 2026, que usa la página `/ui`.

Esta carpeta no se sube a Git; se genera con el comando y va dentro de la imagen.

En la interfaz de MLflow el champion está en Models (menú de la izquierda), `mibici_salidas_diarias`: la versión con el alias `champion` tiene una descripción y la etiqueta `documentation_run`, que lleva a la ejecución "Modelo champion" del experimento `MiBici_Fase3_Champion_Challenger`. Ahí están sus métricas de septiembre y octubre, el error diario en Model metrics, las gráficas en `modeling/test`, `modeling/comparison` y `modeling/interpretation`, y la configuración en `documentation/`.

### 10.2. Instalar, exportar el modelo y probar

```bash
pip install -r requirements.txt   # versiones fijas del entorno de entrenamiento
make export-model                 # copia el champion a models/serving
pytest -q                         # pruebas unitarias y de integración (82)
```

### 10.3. Correr el servicio sin Docker

```bash
make serve
```

### 10.4. Construir y correr la imagen

Requiere Docker Desktop abierto. En Windows con WSL, activar una sola vez la integración con Ubuntu en *Settings > Resources > WSL integration*, y opcionalmente *Settings > General > Start Docker Desktop when you sign in*. Si `docker version` no muestra la sección `Server`, Docker no está abierto.

```bash
docker build -t ml-service:latest .
docker run -p 8000:8000 ml-service:latest
```

Para usar la imagen publicada, sin construirla:

```bash
docker pull josecval/mibici-ml-service:1.2.0
docker run -p 8000:8000 josecval/mibici-ml-service:1.2.0
```

### 10.5. Direcciones del servicio

Con el servicio en marcha (local o en Docker):

| Dirección | Para qué |
| --- | --- |
| <http://localhost:8000> | lleva a la página `/ui` |
| <http://localhost:8000/ui> | página para elegir una estación y un día y ver la predicción |
| <http://localhost:8000/docs> | documentación interactiva (Swagger): probar `POST /predict` con *Try it out* |
| <http://localhost:8000/redoc> | la misma documentación en formato de lectura |
| <http://localhost:8000/health> | estado del servicio y versión del modelo cargado |

La página tiene los viajes reales del 1 de septiembre al 31 de octubre de 2026 y para esos días predice con las mismas variables del pipeline. Para un día posterior, o un día en que la estación no tuvo viajes, usa la última historia conocida de la estación con el calendario del día pedido, y la respuesta lo marca como estimación (`estimated` y `history_end`). La ruta que usa la página también se puede llamar directo: `GET /predict/estacion?origen_id=15&fecha=2026-11-19`.

`POST /predict` recibe una lista `instances` con las 18 variables de cada estación y día, y devuelve los viajes predichos. Pydantic valida tipos y rangos (respuesta 422 si algo no cumple) y el servicio responde 503 si no pudo cargar el modelo. Ejemplo:

```bash
curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" -d '{
  "instances": [{"Origen_Id": 35, "day_of_week": 2, "month": 10, "day_of_month": 15, "weather_zone": 3,
    "education_count": 4, "transport_count": 1, "commerce_count": 2, "work_count": 6, "health_count": 0,
    "recreation_count": 3, "lag_1": 41, "lag_7": 38, "lag_14": 36, "roll_mean_7": 39.4,
    "active_days_7": 7, "roll_mean_28": 37.9, "active_days_28": 28}]}'
```

La especificación OpenAPI completa está en `reports/api/openapi.json`.

### 10.6. Imágenes publicadas

Repositorio: <https://hub.docker.com/r/josecval/mibici-ml-service>

| Etiqueta | Contenido |
| --- | --- |
| `1.0.0` | versión 7 (Fase 2) |
| `1.1.0` | versión 10, sin la página `/ui` |
| `1.2.0` y `latest` | versión 10 con la página `/ui` |

Una etiqueta publicada no se sobrescribe. Para publicar una versión nueva (por ejemplo, después de promover otro champion):

```bash
docker login -u josecval            # una sola vez
make export-model
docker build -t ml-service:latest .
docker tag ml-service:latest josecval/mibici-ml-service:<versión>
docker tag ml-service:latest josecval/mibici-ml-service:latest
docker push josecval/mibici-ml-service:<versión>
docker push josecval/mibici-ml-service:latest
```

`make deploy` hace lo mismo de forma automática: construye la imagen, comprueba dentro del contenedor la muestra de referencia y solo publica si coincide y `AUTO_DEPLOY = True` en `config.py`. La etiqueta de cada champion está en `RELEASES`; uno nuevo recibe la siguiente (1.3.0).

### 10.7. Monitoreo de data drift y champion contra challenger

```bash
make monitor      # octubre de 2026 con la versión 7: drift, escenarios, alertas y acción
make challenger   # reentrena los challengers y aplica la regla de promoción
make champion     # documenta al champion en MLflow con métricas y gráficas
make mlflow-catalog  # resumen de la entrega final y descripción de todos los experimentos
```

Todas las fases usan la misma base `mlflow.db` y la carpeta `mlruns/` de la raíz. Cada experimento tiene una descripción y las etiquetas `phase`, `content` y `status`:

| Experimento | Fase | Contenido | Estado |
| --- | --- | --- | --- |
| `MiBici` | 1 | carga, limpieza e integración de los datos | vigente |
| `MiBici_Fase2_Modelado` | 2 | líneas base, validación, búsqueda y modelo final (v7) | vigente |
| `MiBici_Fase3_Monitoreo_Drift` | 3 | monitoreo de octubre, escenarios y alertas | vigente |
| `MiBici_Fase3_Champion_Challenger` | 3 | comparación de modelos y documentación del champion | vigente |
| `MiBici_Monitoreo_Drift`, `MiBici_Champion_Challenger` | 3 | corridas de desarrollo del notebook 07, donde se registró la v10 | desarrollo |
| `MiBici_Entrega_Final` | 1, 2 y 3 | una ejecución resumen que junta todo, con enlaces a cada ejecución de origen | vigente |

Las ejecuciones de la Fase 3 siguen la convención de la Fase 2: nombre "Tipo | Modelo | detalle", la columna Source con el archivo que las genera, el dataset en la columna Dataset, `README.md`, `artifact_catalog.json`, `environment/runtime.json`, `documentation/run_configuration.json` y las carpetas `data/inventory`, `data/versions` y `data/lineage`. Si se vuelve a correr, no se borra nada: la ejecución anterior con el mismo nombre queda con la etiqueta `run_status = anterior` y la nueva con `vigente`, así que el filtro `tags.run_status = "vigente"` muestra una por tipo. Todas guardan en sus etiquetas el commit (`git_commit`), el entorno (`python_version`, `sklearn_version`) y el SHA-256 de los datos (`data_sha256`, `dataset_sha256`).

| Ejecución | Carpetas propias | Etiquetas para filtrar |
| --- | --- | --- |
| Monitoreo \| Random Forest v7 \| octubre 2026 | `monitoring/performance`, `monitoring/drift`, `monitoring/scenarios`, `monitoring/alerts` | `phase`, `stage`, `model_version` |
| Escenario \| Random Forest v7 \| nombre (hijas) | métricas del escenario | `scenario`, `scenario_type`, `action`, `alert_A1` a `alert_A5` |
| Champion contra challenger \| Random Forest \| octubre 2026 | `modeling/comparison`, `modeling/test` | `model_role`, `decision`, `candidate`, `reason` |
| Modelo champion \| Random Forest \| v10 | `modeling/comparison`, `modeling/test`, `modeling/interpretation` | `model_role`, `model_version`, `registered_model_alias` |

La ejecución "Entrega final | Random Forest | resumen de las tres fases" del experimento `MiBici_Entrega_Final` junta todo en un solo lugar: copia completa de los artefactos de la ejecución de datos de la Fase 1, del modelo final v7, del entrenamiento de la v10, del monitoreo, de la comparación y del champion, cada una en su carpeta; todas sus métricas con el prefijo de su origen; `resumen/` con la tabla de métricas, la lista de ejecuciones con enlace, SHA-256 y commit, las alertas por escenario y las revisiones del notebook 08; y un `README.md` con los enlaces. Se genera con `make mlflow-catalog` o desde el notebook 08.

Para ver las etiquetas como columnas: en la vista de ejecuciones, Columns, sección Tags. Para filtrar, por ejemplo: `tags.alert_A3 = "crítica"` o `tags.model_role = "champion"`.

`make monitor` espera el CSV de octubre en `data/monitoring/2026/datos_abiertos_2026_10.csv`. Guarda las tablas y los reportes de Evidently en `reports/drift/` y registra el resultado en el experimento `MiBici_Fase3_Monitoreo_Drift` de MLflow.

Las alertas tienen dos niveles, advertencia y crítica:

| Alerta | Qué mide | Advertencia | Crítica |
| --- | --- | --- | --- |
| A1 | `lag_1` vacío por arriba de la línea base | 5 puntos | 15 puntos |
| A2 | sesgo frente a la demanda media | 10 % | 20 % |
| A3 | MAE y mejora frente al ingenuo | MAE +20 % o mejora < 20 % | las dos |
| A4 | PSI máximo o proporción de variables con drift | PSI > 0.10 o más de 30 % | PSI > 0.25 |
| A5 | cambio en estaciones activas o viajes por día | 15 % | 30 % |

La acción sigue este orden: problemas de datos primero (revisar el pipeline, no reentrenar); reentrenar solo si A3 es crítica dos meses seguidos; revisar con operación si A5 es crítica; vigilar ante cualquier advertencia.

`make challenger` registra la comparación en el experimento `MiBici_Fase3_Champion_Challenger`. El challenger gana si baja el MAE al menos 2 % y no empeora el RMSE. Solo se promueve, es decir, se registra la versión nueva y se le pasa el alias `champion`, si además `AUTO_PROMOTE = True` en `config.py`; con `False` solo muestra la decisión.

### 10.8. Requisitos de la entrega final

| Requisito | Dónde se cumple |
| --- | --- |
| Pruebas unitarias y de integración con `pytest -q` | `tests/` (82 pruebas). `test_integration_pipeline.py` corre el flujo completo: carga, limpieza, variables, entrenamiento, predicción, métricas y API |
| `POST /predict` con Pydantic y OpenAPI | `mibici_mlops/serving/app.py`; esquema en `/docs` y `/redoc`; artefacto `models:/mibici_salidas_diarias/10` (sección 10.1) |
| Reproducibilidad en otro entorno | `requirements.txt` con versiones fijas, `SEED = 42`, muestra de referencia comparada dentro del contenedor (`reports/reproducibility/`), modelo y datos versionados en MLflow con su SHA-256 |
| Docker | `Dockerfile`, `docker build -t ml-service:latest .`, `docker run -p 8000:8000 ml-service:latest`, imagen `josecval/mibici-ml-service` con etiquetas 1.0.0, 1.1.0, 1.2.0 y `latest` |
| Data drift y pérdida de desempeño | `mibici_mlops/monitoring.py`: octubre real y tres escenarios (rezagos faltantes, medias desplazadas, cambio estacional) comparados contra la prueba 2026, alertas A1 a A5 con umbrales y acción propuesta, registrados en `MiBici_Fase3_Monitoreo_Drift` |

## 11. Buenas prácticas de MLOps utilizadas

- **Separación de responsabilidades:** cada módulo atiende una tarea específica; `workflow.py` coordina el orden sin duplicar el procesamiento.
- **Código reutilizable:** los notebooks pueden servir para explorar, pero la implementación se ejecuta desde archivos `.py`.
- **Dependencias explícitas:** entorno virtual y archivos de dependencias.
- **Rutas portables:** archivos localizados desde la raíz del proyecto y no mediante rutas absolutas de un integrante.
- **Datos originales intactos:** las transformaciones generan archivos nuevos.
- **Separación temporal:** entrenamiento, validación y prueba se dividen por fechas; la prueba independiente no se utiliza para seleccionar hiperparámetros.
- **Comparación contra líneas base:** los resultados se interpretan respecto a métodos sencillos, no solo mediante una métrica aislada.
- **Seguimiento de experimentos:** parámetros, métricas, gráficas, versiones y modelos se registran en MLflow.
- **Trazabilidad:** Git, hashes y manifiestos permiten identificar las entradas y el código asociado a una ejecución.
- **Pruebas:** `pytest` permite detectar cambios inesperados en el comportamiento.
- **Servicio reproducible:** la imagen de Docker usa las mismas versiones de librerías que el entrenamiento y se comprueba contra una muestra de referencia antes de publicarla.
- **Monitoreo:** el drift y el desempeño se revisan cada mes con alertas de dos niveles, y un cambio de modelo pasa por una regla fija antes de promoverse.

## 12. Referencias

- [MiBici](https://www.mibici.net/)
- [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/)
- [MLflow](https://mlflow.org/docs/latest/)
- [scikit-learn](https://scikit-learn.org/stable/)
- [pytest](https://docs.pytest.org/)
- [FastAPI](https://fastapi.tiangolo.com/)
- [Docker](https://docs.docker.com/)
- [Evidently](https://docs.evidentlyai.com/)
- [Open-Meteo](https://open-meteo.com/)
- [OpenStreetMap](https://www.openstreetmap.org/)

Writing ../build/migracion/README.md


## 16. Archivos listos para copiar
Comprobamos qué archivos se generaron en `build/migracion/` y si cada uno es nuevo o reemplaza a uno del repositorio.

In [28]:
rows = []
for path in sorted(p for p in STAGE.rglob("*") if p.is_file()):
    relative = path.relative_to(STAGE)
    current = ROOT / relative
    if not current.exists():
        state = "nuevo"
    elif current.read_bytes() == path.read_bytes():
        state = "igual"
    else:
        state = "modificado"
    rows.append({"archivo": str(relative), "estado": state, "líneas": len(path.read_text().splitlines())})
staged_files = pd.DataFrame(rows)
print(staged_files["estado"].value_counts().to_string())
staged_files

estado
igual         13
modificado    11
nuevo          2


,archivo,estado,líneas
0,.dockerignore,igual,6
1,Dockerfile,igual,25
2,Makefile,modificado,141
3,README.md,modificado,497
4,mibici_mlops/config.py,modificado,306
5,mibici_mlops/features.py,igual,67
6,mibici_mlops/mlflow_catalog.py,nuevo,400
7,mibici_mlops/modeling/challenger.py,modificado,254
8,mibici_mlops/modeling/champion_report.py,nuevo,240
9,mibici_mlops/modeling/deploy.py,igual,170


## Verificación contra el notebook 07
Antes de copiar los archivos al repositorio, verificamos que los resultados no cambien. Para hacerlo preparamos en `build/preview/` una copia del repositorio como quedará después de la migración: el código actual más los archivos de `build/migracion/`. Los datos, `mlflow.db` y `mlruns` no se copian; se enlazan a los originales. `models/serving` sí se copia, porque Docker no sigue enlaces fuera de la carpeta que construye.

Las salidas nuevas se guardan en `reports/verificacion_08/` y `build/verificacion_08/`, así no se sobrescribe nada de lo que dejó el 07. Estas comprobaciones no registran ejecuciones en MLflow ni modifican el alias `champion`.

### V.1 Copia del repositorio con los archivos nuevos

In [29]:
PREVIEW = ROOT / "build" / "preview"
VERIFY_REPORTS = ROOT / "reports" / "verificacion_08"
VERIFY_BUILD = ROOT / "build" / "verificacion_08"
NOT_COPIED = {"build", "data", "docs", "mlflow.db", "mlruns", "models", "notebooks", "references", "reports"}

for folder in [PREVIEW, VERIFY_BUILD]:
    if folder.exists():
        shutil.rmtree(folder)
PREVIEW.mkdir(parents=True)
for item in ROOT.iterdir():
    if item.name in NOT_COPIED or item.name.startswith("."):
        continue
    if item.is_dir():
        shutil.copytree(item, PREVIEW / item.name, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    else:
        shutil.copy2(item, PREVIEW / item.name)
shutil.copytree(STAGE, PREVIEW, dirs_exist_ok=True)

# Datos y MLflow se enlazan; models/serving se copia para que Docker lo pueda leer
for name in ["data", "mlflow.db", "mlruns"]:
    if (ROOT / name).exists():
        (PREVIEW / name).symlink_to(ROOT / name, target_is_directory=(ROOT / name).is_dir())
(PREVIEW / "models").mkdir()
shutil.copytree(ROOT / "models" / "serving", PREVIEW / "models" / "serving")
for model_file in (ROOT / "models").glob("*.joblib"):
    (PREVIEW / "models" / model_file.name).symlink_to(model_file)
VERIFY_REPORTS.mkdir(parents=True, exist_ok=True)
print(sorted(p.name for p in PREVIEW.iterdir()))

['.dockerignore', 'Dockerfile', 'MLproject', 'Makefile', 'README.md', 'data', 'mibici_mlops', 'mlflow.db', 'mlflow_backup.db', 'mlruns', 'models', 'pyproject.toml', 'python_env.yaml', 'requirements-serving.txt', 'requirements.txt', 'tests']


Cargamos los módulos desde la copia y revisamos que el paquete y las rutas de `config.py` vienen de `build/preview`.

In [30]:
for name in [m for m in sys.modules if m == "mibici_mlops" or m.startswith("mibici_mlops.")]:
    del sys.modules[name]
sys.path.insert(0, str(PREVIEW))

import mlflow
import mibici_mlops
from mibici_mlops import config, monitoring
from mibici_mlops.modeling import challenger, deploy, export_model
from mibici_mlops.modeling.evaluate import regression_metrics

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
registry = mlflow.MlflowClient()
checks = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append({"revisión": name, "ok": bool(ok), "detalle": detail})
    print("OK   " if ok else "FALLA", name, "|", detail)


check("Paquete importado desde build/preview",
      Path(mibici_mlops.__file__).parent == PREVIEW / "mibici_mlops" and config.PROJECT_ROOT == PREVIEW,
      str(Path(mibici_mlops.__file__).parent))

OK    Paquete importado desde build/preview | /home/josel/mlops/equipo-8-mibici-mlops/build/preview/mibici_mlops


### V.2 Modelo v7 y métricas de la prueba 2026
`load_context` carga los datos, hace la separación temporal y trae la v7 del registro, lo mismo que las secciones 2.4 y 2.5. Después comparamos sus predicciones con el `.joblib` y su MAE y RMSE con los del README.

In [31]:
import joblib

REFERENCE_MAE = config.REFERENCE_RESULTS["Modelo final, prueba 2026 (todas)"]
REFERENCE_RMSE = 10.7824   # el mismo valor de la tabla de 2.5

context = monitoring.load_context()
test = context["test"]
pred_v7 = context["model"].predict(test[config.FEATURE_COLUMNS])
pred_joblib = joblib.load(config.FINAL_MODEL_PATH).predict(test[config.FEATURE_COLUMNS])
v7_metrics = regression_metrics(test[config.TARGET_COLUMN], pred_v7)

check("v7 del registro igual al .joblib", np.abs(pred_v7 - pred_joblib).max() < 1e-9,
      f"diferencia máxima {np.abs(pred_v7 - pred_joblib).max():.2e}")
check("MAE y RMSE de la v7 en la prueba 2026",
      abs(v7_metrics["mae"] - REFERENCE_MAE) < 5e-4 and abs(v7_metrics["rmse"] - REFERENCE_RMSE) < 5e-4,
      f"MAE {v7_metrics['mae']:.6f} | RMSE {v7_metrics['rmse']:.6f}")

OK    v7 del registro igual al .joblib | diferencia máxima 1.14e-13
OK    MAE y RMSE de la v7 en la prueba 2026 | MAE 5.896360 | RMSE 10.782439


### V.3 Exportación del champion
Exportamos a una carpeta temporal la versión con el alias `champion` y la comparamos con lo que dejó 2.6 en `models/serving`: la misma versión, las mismas 500 filas de referencia y las mismas predicciones. En `metadata.json` solo cambia la fecha de exportación.

In [32]:
champion = registry.get_model_version_by_alias(config.REGISTERED_MODEL_NAME, config.MODEL_ALIAS)
integrated_sha = export_model.sha256(config.INTEGRATED_PATH)
exported = export_model.export_serving_model(champion, test, integrated_sha, export_model.environment_versions(),
                                             serving_dir=VERIFY_BUILD / "serving")

saved_reference = pd.read_csv(ROOT / "models" / "serving" / "reference_sample.csv", parse_dates=["Fecha"])
saved_meta = json.loads((ROOT / "models" / "serving" / "metadata.json").read_text())
new_meta = json.loads((VERIFY_BUILD / "serving" / "metadata.json").read_text())
different_keys = [k for k in saved_meta if k != "exported_at" and saved_meta[k] != new_meta.get(k)]

check("Mismas filas de referencia", exported[["Fecha", "Origen_Id"]].reset_index(drop=True)
      .equals(saved_reference[["Fecha", "Origen_Id"]]), f"{len(exported)} filas")
check("Mismas predicciones de referencia",
      np.allclose(exported["reference_prediction"], saved_reference["reference_prediction"], rtol=0, atol=1e-9),
      f"champion v{champion.version}")
check("metadata.json igual salvo la fecha", not different_keys, f"campos distintos: {different_keys}")

OK    Mismas filas de referencia | 500 filas
OK    Mismas predicciones de referencia | champion v10
OK    metadata.json igual salvo la fecha | campos distintos: []


### V.4 Servicio con la muestra de referencia
Probamos el servicio de `mibici_mlops/serving/app.py` con el modelo de `models/serving` con las 500 filas, igual que en 3.5. `to_instance` ahora viene de `deploy.py`.

In [33]:
from fastapi.testclient import TestClient

from mibici_mlops.serving import app as service

service.MODEL_URI = str(ROOT / "models" / "serving" / "model")
payload = {"instances": [deploy.to_instance(r) for r in saved_reference.to_dict(orient="records")]}
with TestClient(service.app) as api:
    health = api.get("/health").json()
    response = api.post("/predict", json=payload)
api_predictions = np.array([p["predicted_trips"] for p in response.json()["predictions"]])
api_difference = np.abs(api_predictions - saved_reference["reference_prediction"].clip(lower=0).to_numpy()).max()

check("API responde con la versión exportada", health["model_version"] == saved_meta["model_version"],
      f"versión {health['model_version']}")
check("API igual a la muestra de referencia", response.status_code == 200 and api_difference < 1e-4,
      f"diferencia máxima {api_difference:.2e}")

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


OK    API responde con la versión exportada | versión 10
OK    API igual a la muestra de referencia | diferencia máxima 4.98e-05


### V.5 Monitoreo de octubre
`run_monitoring` hace todo lo de 5.1 a 5.12 con la v7 y guarda sus tablas en `reports/verificacion_08/`. Después comparamos los resultados con los del notebook 07 en `reports/drift/`: drift por variable con su diagnóstico, resultados por escenario y alertas con la acción.

In [34]:
results = monitoring.run_monitoring(
    context["model"], context["integrated"], context["dataset"], context["development"], test,
    monitoring.MONITOR_CSV, monitoring.MONITOR_START, monitoring.MONITOR_END, monitoring.DROP_START,
    output_dir=VERIFY_REPORTS,
)


def same_table(name: str, index_col=None) -> tuple[bool, str]:
    before = pd.read_csv(ROOT / "reports" / "drift" / name, index_col=index_col)
    after = pd.read_csv(VERIFY_REPORTS / name, index_col=index_col)
    try:
        pd.testing.assert_frame_equal(before, after, check_exact=False, rtol=1e-9, atol=1e-12)
        return True, f"{len(after)} filas"
    except AssertionError as error:
        return False, str(error).splitlines()[0]


for name, index_col in [("drift_october_2026.csv", None), ("scenario_results.csv", None),
                        ("alerts_and_actions.csv", 0)]:
    check(f"Monitoreo: {name}", *same_table(name, index_col))
figures = ["monthly_performance.png", "daily_error_scenarios.png", "demand_distribution.png"]
check("Monitoreo: las tres gráficas de 5.13", all((VERIFY_REPORTS / f).is_file() for f in figures), ", ".join(figures))
results["decisions"][["A3_mes_anterior", "acción"]]

OK    Monitoreo: drift_october_2026.csv | 16 filas
OK    Monitoreo: scenario_results.csv | 5 filas
OK    Monitoreo: alerts_and_actions.csv | 5 filas
OK    Monitoreo: las tres gráficas de 5.13 | monthly_performance.png, daily_error_scenarios.png, demand_distribution.png


,A3_mes_anterior,acción
Línea base (prueba ene-sep 2026),-,Sin acción
Octubre real,crítica,Reentrenar: A3 crítica dos meses seguidos
1. Rezagos faltantes,-,Revisar el pipeline de datos (no reentrenar)
2. Medias desplazadas,-,Revisar el pipeline de datos (no reentrenar)
3. Cambio estacional,-,Vigilar y confirmar con el siguiente mes


### V.6 Días para la página `/ui`
Generamos `ui_days.csv` con septiembre del integrado y el octubre que preparó el monitoreo, en el `models/serving` de la copia (no en el del repositorio). Seleccionamos 200 combinaciones aleatorias de estación y día para comprobar que la ruta de la página da lo mismo que el modelo con esas variables, y que la página carga.

In [35]:
import mlflow.sklearn
from mibici_mlops.modeling.export_model import export_ui_days

ui_days = export_ui_days(context["dataset"], results["month"], PREVIEW / "models" / "serving")
served_model = mlflow.sklearn.load_model(str(PREVIEW / "models" / "serving" / "model"))
sample_days = ui_days.sample(n=min(200, len(ui_days)), random_state=config.SEED)
expected_days = np.clip(served_model.predict(sample_days[config.FEATURE_COLUMNS]), 0, None).round(2)

service.MODEL_URI = str(PREVIEW / "models" / "serving" / "model")
with TestClient(service.app) as api:
    page = api.get("/ui")
    from_page = [api.get("/predict/estacion", params={"origen_id": int(r.Origen_Id), "fecha": r.Fecha})
                 .json()["predicted_trips"] for r in sample_days.itertuples()]
    last_day = (pd.Timestamp(ui_days["Fecha"].max()) + pd.Timedelta(days=19)).strftime("%Y-%m-%d")
    future = api.get("/predict/estacion", params={"origen_id": int(ui_days["Origen_Id"].iloc[-1]), "fecha": last_day})
page_difference = float(np.abs(np.array(from_page) - expected_days).max())

check("Página /ui carga", page.status_code == 200 and "<select" in page.text,
      f"{ui_days['Origen_Id'].nunique()} estaciones, {ui_days['Fecha'].min()} a {ui_days['Fecha'].max()}")
check("Día posterior a los datos usa la última historia", future.status_code == 200 and future.json()["estimated"],
      f"{last_day}: historia hasta {future.json().get('history_end')}")
check("Predicción de la página igual al modelo", page_difference < 0.006,
      f"diferencia máxima {page_difference:.3f} en {len(sample_days)} días (redondeo a 2 decimales)")

OK    Página /ui carga | 469 estaciones, 2026-09-01 a 2026-10-31
OK    Día posterior a los datos usa la última historia | 2026-11-19: historia hasta 2026-10-31
OK    Predicción de la página igual al modelo | diferencia máxima 0.000 en 200 días (redondeo a 2 decimales)


### V.7 Champion contra challenger
Entrenamos nuevamente los modelos A y B con `run_challenger` y comparamos las tablas de septiembre y octubre con las de 5.15. La decisión se compara con la última registrada en MLflow (la del 07 o la de una corrida anterior de este notebook). Aquí no se llama a `log_challenger`, así que no se registra ni se promueve nada.

In [36]:
if RUN_CHALLENGER:
    challenge = challenger.run_challenger(
        context["model"], context["model_version"], context["dataset"], results["month"],
        challenger.SELECTION_START, monitoring.MONITOR_START, output_dir=VERIFY_REPORTS,
    )
    decision, reason = challenger.promotion_decision(challenge, champion, context["model_version"])
    names = [n for n in [config.CHALLENGER_EXPERIMENT, "MiBici_Champion_Challenger"]
             if mlflow.get_experiment_by_name(n) is not None]
    runs = mlflow.search_runs(experiment_names=names, order_by=["start_time DESC"])
    last_run = runs.loc[runs["tags.decision"].notna()].iloc[0]
    for name in ["challenger_selection_september.csv", "challenger_test_october.csv"]:
        check(f"Challenger: {name}", *same_table(name, index_col=0))
    check("Misma decisión que en 5.15", decision == last_run["tags.decision"],
          f"{decision} | registrada: {last_run['tags.decision']}")
    print(f"Elegido: {challenge['best_name']} | mejora de MAE {challenge['improvement']:.1%} | {reason}")
else:
    print("RUN_CHALLENGER = False: se salta esta revisión.")

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning:

`TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.



Challenger A (reentrenado): 785,290 filas hasta 2026-08-31 | 119 s


/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning:

`TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.



Challenger B (+ festivos): 785,290 filas hasta 2026-08-31 | 122 s


/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning:

`TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.



Challenger B (+ festivos): 799,264 filas hasta 2026-09-30 | 115 s
OK    Challenger: challenger_selection_september.csv | 3 filas
OK    Challenger: challenger_test_october.csv | 2 filas
OK    Misma decisión que en 5.15 | ya promovido | registrada: ya promovido
Elegido: Challenger B (+ festivos) | mejora de MAE 3.3% | el champion v10 es Challenger B (+ festivos) entrenado hasta septiembre


### V.8 Imagen de Docker
Construimos la imagen desde `build/preview` con el `Dockerfile` de la raíz, como quedará en el repositorio, con la etiqueta `ml-service:migracion` para no reemplazar `ml-service:latest`. Ejecutamos el contenedor y comparamos la muestra de referencia predicha dentro del contenedor, igual que en 4.7. También revisamos que la página `/ui` responda desde el contenedor. No se publica nada.

In [37]:
import requests

MIGRATION_IMAGE = "ml-service:migracion"

if not RUN_DOCKER:
    print("RUN_DOCKER = False: se salta esta revisión.")
elif not deploy.docker_available():
    print("Docker no está abierto: se salta esta revisión. Para incluirla, abre Docker Desktop y corre esta celda.")
else:
    deploy.build_image(MIGRATION_IMAGE, PREVIEW)
    try:
        deploy.start_container(MIGRATION_IMAGE)
        docker_summary, _ = deploy.check_container(saved_reference)
        ui_status = requests.get(f"http://localhost:{deploy.PORT}/", timeout=10).status_code  # la raíz lleva a /ui
    finally:
        deploy.run(["docker", "rm", "-f", deploy.CONTAINER], check=False)
    check("Imagen construida con el Dockerfile de la raíz",
          docker_summary["model_version"] == saved_meta["model_version"]
          and docker_summary["model_uri"] == "/app/models/serving/model",
          f"versión {docker_summary['model_version']}")
    check("Contenedor igual a la muestra de referencia", docker_summary["reproducible"],
          f"diferencia máxima {docker_summary['max_abs_difference']:.2e}")
    check("http://localhost:8000 abre la página desde el contenedor", ui_status == 200, f"status {ui_status}")
    deploy.run(["docker", "rmi", MIGRATION_IMAGE], check=False)

$ docker version --format {{.Server.Version}}
$ docker build -t ml-service:migracion /home/josel/mlops/equipo-8-mibici-mlops/build/preview
Imagen ml-service:migracion construida en 5 s
$ docker rm -f mibici-api
$ docker run -d --name mibici-api -p 8000:8000 ml-service:migracion
$ docker rm -f mibici-api
OK    Imagen construida con el Dockerfile de la raíz | versión 10
OK    Contenedor igual a la muestra de referencia | diferencia máxima 4.98e-05
OK    http://localhost:8000 abre la página desde el contenedor | status 200
$ docker rmi ml-service:migracion


### V.9 Todas las pruebas con un solo comando
Corremos `pytest -q` dentro de `build/preview`, que es lo mismo que se correrá en la raíz después de mover los archivos. Revisamos que se ejecuten todas las pruebas: las de 6.9, que ahí se corrieron en dos carpetas, más las 7 nuevas de la página. La salida se guarda en `reports/verificacion_08/pytest.txt`.

In [38]:
start = time.perf_counter()
pytest_run = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
                            cwd=PREVIEW, capture_output=True, text=True)
output = (pytest_run.stdout + pytest_run.stderr).strip()
(VERIFY_REPORTS / "pytest.txt").write_text(output, encoding="utf-8")
print(f"$ pytest -q   ({time.perf_counter() - start:.0f} s)")
print(output[-1500:])


def passed(text: str) -> int:
    import re
    return sum(int(n) for n in re.findall(r"(\d+) passed", text))


expected_tests = passed((ROOT / "reports" / "tests" / "pytest_entrega_final.txt").read_text(encoding="utf-8"))
check("pytest -q en una sola corrida", pytest_run.returncode == 0 and passed(output) == expected_tests + 7,
      f"{passed(output)} pasaron | en 6.9: {expected_tests}, más 7 de la página")

$ pytest -q   (8 s)
.. [ 87%]
..........                                                               [100%]
=============================== warnings summary ===============================
tests/test_api.py::test_health_reports_loaded_model
tests/test_integration_pipeline.py::test_cleaning_removes_only_the_dirty_rows
tests/test_modeling.py::test_pipeline_preprocessing_and_prediction
tests/test_reproducibility.py::test_same_seed_gives_same_predictions
tests/test_reproducibility.py::test_same_seed_gives_same_predictions
  /home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
    warnings.warn(

tests/test_api.py::test_health_reports_loaded_model
  /home/josel/mlops/equipo-8-mibici-mlops/.ve

### V.10 Resumen

In [39]:
summary = pd.DataFrame(checks)
summary.to_csv(VERIFY_REPORTS / "resumen_verificacion.csv", index=False)
assert summary["ok"].all(), "Hay revisiones que no coinciden con el notebook 07"
print(f"Las {len(summary)} revisiones coinciden con el notebook 07.")
summary

Las 22 revisiones coinciden con el notebook 07.


,revisión,ok,detalle
0,Paquete importado desde build/preview,True,/home/josel/mlops/equipo-8-mibici-mlops/build/...
1,v7 del registro igual al .joblib,True,diferencia máxima 1.14e-13
2,MAE y RMSE de la v7 en la prueba 2026,True,MAE 5.896360 | RMSE 10.782439
3,Mismas filas de referencia,True,500 filas
4,Mismas predicciones de referencia,True,champion v10
5,metadata.json igual salvo la fecha,True,campos distintos: []
6,API responde con la versión exportada,True,versión 10
7,API igual a la muestra de referencia,True,diferencia máxima 4.98e-05
8,Monitoreo: drift_october_2026.csv,True,16 filas
9,Monitoreo: scenario_results.csv,True,5 filas


## Registro en MLflow
Después de las verificaciones, registramos los resultados en dos experimentos de MLflow. Conservamos la organización utilizada en la fase 2 (contexto de la ejecución, descripción, `README.md` con la guía de artefactos, carpetas por tipo de archivo y series en Model metrics):

| Experimento | Ejecución | Qué contiene |
|---|---|---|
| `MiBici_Fase3_Monitoreo_Drift` | Monitoreo \| Random Forest v7 \| octubre 2026 | desempeño por mes, drift por variable, reportes de Evidently, escenarios, alertas y acción; una ejecución hija por escenario |
| `MiBici_Fase3_Champion_Challenger` | Champion contra challenger \| Random Forest \| octubre 2026 | selección en septiembre, prueba en octubre y decisión |
| `MiBici_Fase3_Champion_Challenger` | Modelo champion \| Random Forest \| v10 | métricas, gráficas, configuración y enlace a la versión del Model Registry |
| `MiBici_Entrega_Final` | Entrega final \| Random Forest \| resumen de las tres fases | todo junto: artefactos completos de cada ejecución de origen, métricas, alertas, revisiones y enlaces |

Al final, `document_experiments` agrega la descripción y las etiquetas de todos los experimentos, incluidos `MiBici`, `MiBici_Fase2_Modelado` y los dos del notebook 07; sus ejecuciones no se modifican. Si este notebook se vuelve a correr, no se borra nada: la ejecución anterior con el mismo nombre queda marcada como `anterior`. Al final, la ejecución "Entrega final" del experimento `MiBici_Entrega_Final` junta todo lo anterior con enlaces a cada ejecución. Los registros usan la raíz del repositorio, no `build/preview`, para que el commit, los datos y `mlruns/` sean los del proyecto. No se promueve ningún modelo: `log_challenger` corre con `auto_promote=False`.

In [40]:
import mlflow.sklearn
from mibici_mlops import mlflow_catalog
from mibici_mlops.modeling import champion_report

if LOG_TO_MLFLOW:
    monitoring_run = monitoring.log_monitoring(results, context["model_uri"], monitoring.MONITOR_CSV.name,
                                               output_dir=VERIFY_REPORTS, project_root=ROOT)
    print("Monitoreo:", monitoring_run)

    selection_table = None
    if RUN_CHALLENGER:
        logged = challenger.log_challenger(challenge, decision, reason, champion, context["model_uri"],
                                           results["month"][config.FEATURE_COLUMNS].head(5), auto_promote=False,
                                           output_dir=VERIFY_REPORTS, project_root=ROOT,
                                           month=results["month"])
        selection_table = challenge["selection_table"]
        print("Champion contra challenger:", logged["run_id"], "| decisión:", decision)

    champion_model = mlflow.sklearn.load_model(f"models:/{config.REGISTERED_MODEL_NAME}/{champion.version}")
    report = champion_report.build_champion_report(champion_model, champion.version, context["model_version"],
                                                   results["month"], selection_table,
                                                   output_dir=ROOT / "reports" / "champion")
    champion_run = champion_report.log_champion(report, champion, champion_model,
                                                output_dir=ROOT / "reports" / "champion", project_root=ROOT)
    print("Modelo champion:", champion_run)
    summary_run = mlflow_catalog.log_final_summary(
        ROOT, summary, extra_files={"verificacion": [VERIFY_REPORTS / "resumen_verificacion.csv",
                                                     VERIFY_REPORTS / "pytest.txt"]})
    print("Entrega final:", summary_run)
    display(report["comparison"].round(3))
else:
    print("LOG_TO_MLFLOW = False: no se registra nada.")

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning:

The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://ww

Monitoreo: e4416a0bf5034587a80718439fe7c2d1


/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning:

The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://ww

Champion contra challenger: b00aad32e29e489b907c1e961583640c | decisión: ya promovido


/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning:

The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://ww

Modelo champion: fd5ed2e6e5a44563ac371b507867d0ce


Entrega final: 726271ea178d4c69ab99cc66b1fe4123


,mae,rmse,mejora_vs_ingenuo,mae_con_historia,mae_nuevas,mae_12_oct,sesgo
modelo,,,,,,,
v7 (en uso en octubre),7.677,15.145,0.182,8.658,3.966,9.711,-1.950
v10 (champion),7.422,13.961,0.209,8.352,3.901,10.455,-1.572


Agregamos la descripción y las etiquetas de cada experimento y comprobamos los registros creados: los experimentos, dónde guardan sus artefactos y la versión del champion en el Model Registry con su descripción.

In [41]:
catalog = mlflow_catalog.document_experiments()
display(catalog)
experiments = pd.DataFrame([{"experimento": e.name, "artefactos": e.artifact_location,
                             "ejecuciones": len(mlflow.search_runs([e.experiment_id]))}
                            for e in mlflow.search_experiments()])
display(experiments)
champion_now = registry.get_model_version_by_alias(config.REGISTERED_MODEL_NAME, config.MODEL_ALIAS)
print(f"Champion: versión {champion_now.version} | alias {champion_now.aliases}")
print("Descripción:", champion_now.description)
print("Ejecución con la evidencia:", champion_now.tags.get("documentation_run"))

,experimento,phase,content,status,versiones,ejecuciones
0,MiBici,fase_1,datos,vigente,[],10
1,MiBici_Fase2_Modelado,fase_2,modelado,vigente,"[1, 2, 3, 4, 5, 6, 7]",553
2,MiBici_Monitoreo_Drift,fase_3,monitoreo,desarrollo,[],50
3,MiBici_Champion_Challenger,fase_3,champion_challenger,desarrollo,"[8, 9, 10]",6
4,MiBici_Fase3_Monitoreo_Drift,fase_3,monitoreo,vigente,[],15
5,MiBici_Fase3_Champion_Challenger,fase_3,champion_challenger,vigente,[],6
6,MiBici_Entrega_Final,fases_1_2_3,resumen,vigente,[],1


,experimento,artefactos,ejecuciones
0,MiBici_Entrega_Final,file:///home/josel/mlops/equipo-8-mibici-mlops...,1
1,MiBici_Fase3_Champion_Challenger,file:///home/josel/mlops/equipo-8-mibici-mlops...,6
2,MiBici_Fase3_Monitoreo_Drift,file:///home/josel/mlops/equipo-8-mibici-mlops...,15
3,MiBici_Champion_Challenger,/home/josel/mlops/equipo-8-mibici-mlops/notebo...,6
4,MiBici_Monitoreo_Drift,/home/josel/mlops/equipo-8-mibici-mlops/notebo...,50
5,MiBici_Fase2_Modelado,file:///home/josel/mlops/equipo-8-mibici-mlops...,553
6,NOMBRE_DEL_EXPERIMENTO,/home/josel/mlops/equipo-8-mibici-mlops/mlruns/2,1
7,MiBici,file:///home/josel/mlops/equipo-8-mibici-mlops...,10
8,Default,/home/josel/mlops/equipo-8-mibici-mlops/notebo...,9


Champion: versión 10 | alias ['challenger', 'champion']
Descripción: Champion: Challenger B (+ festivos), entrenado hasta 2026-09-30. Octubre 2026: MAE 7.422 contra 7.677 de la v7. Evidencia en la ejecución fd5ed2e6e5a44563ac371b507867d0ce del experimento MiBici_Fase3_Champion_Challenger.
Ejecución con la evidencia: fd5ed2e6e5a44563ac371b507867d0ce


Para consultar los resultados en MLflow, ejecutamos: `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000` desde la raíz y abrir http://127.0.0.1:5000.

- Experiments, `MiBici_Fase3_Monitoreo_Drift`: el monitoreo de octubre y sus escenarios.
- Experiments, `MiBici_Fase3_Champion_Challenger`: la comparación y la ejecución "Modelo champion", con sus gráficas en Artifacts y sus series en Model metrics.
- Models (menú de la izquierda), `mibici_salidas_diarias`: la versión con el alias `champion`, su descripción y el enlace a la ejecución con la evidencia. La v7 sigue ahí como el modelo de la Fase 2.

En Experiments, las columnas Description y Tags muestran la fase y el estado de cada experimento. Dentro de cada experimento, en Columns, sección Tags, se pueden mostrar `phase`, `model_role`, `model_version`, `decision`, `scenario`, `action` y `alert_A1` a `alert_A5` como columnas. Para filtrar se escribe en la búsqueda, por ejemplo, `tags.alert_A3 = "crítica"` o `tags.model_role = "champion"`.


### Uso IA:
- **OpenAI. (2026). *ChatGPT*.** Utilizado como apoyo para organizar la estructura del notebook, revisar la redacción, mejorar las convenciones de nombres de variables y funciones, y desarrollar y revisar algunas secciones de código relacionadas con el procesamiento de datos, los pipelines de Scikit-Learn y la validación temporal.
  https://chatgpt.com/